# Can We Trust a Passing Agent?

Auditing and Repairing Coding-Agent Evaluation Through a Gemma 4 Case Study

Section 2d reproduces the evaluator repair shown in Figure 1 of the Writeup. It selects tests from compiled annotations within a frozen scope, then checks that every required invocation completes. Across two source-exposed tasks and seven equivalent layouts each, it restores complete execution in 14/14 task-layouts. Both incorrect candidates are authored, and layouts are dependent perturbations of two tasks.

## Start here to reproduce the evaluator repair

1. Download `EVALUATOR_LAYOUT_REPLAY_KIT_v1.zip` from the [public repository](https://github.com/devinnicholson/passing-agent-evaluator-repair) or the [associated Writeup](https://www.kaggle.com/competitions/gemma-4-developer-agent-paper/writeups/when-a-coding-agent-evaluator-misses-the-test) under Project Files. Preserve the ZIP bytes. Its SHA256 is `718ef0b8da7a19c2d11e2a86e724188b15b0ba2da88337a2d352b18e29338433`.
2. Use a fresh Linux amd64 CPU environment with Python 3.12 and a Java 17 JDK. Extract the ZIP, enter its root directory, and set `JAVA_HOME` to that JDK. The recorded workers used Python 3.12.15 and Debian OpenJDK 17.0.20.1. One CPU and at least one GiB RAM suffice. No model, GPU, cloud account, credentials or private input packet is needed.
3. Follow **Native reproduction in three steps** in Section 2d below. The commands download hash-pinned public dependencies, compile the fixtures and run both native studies. New output directories are required.
4. Inspect each `REPORT.json`, including `completed`, `all_variants_qualified`, `all_complete_controls_match` and `errors`. Check the expected matrix below and the Figure 1 crosswalk in Section 2d. A driver exit of zero alone does not establish success.

| Selector | Complete task-layouts across both cases | Nonempty authored-candidate passes | Empty selections |
|---|---|---|---|
| Released filter | 2/14 | 2/14 | 10/14 |
| Same-line patch | 5/14 | 1/14 | 8/14 |
| Compiled inventory | 14/14 | 0/14 | 0/14 |

These totals cover 84 native executions. The contract labels incomplete execution INCOMPLETE, including the baseline's nonempty candidate passes. The repaired selector accepts all 14 developer controls and rejects all 14 authored-candidate layouts.

## Choose the evidence you want to check

- **Main repair and Figure 1**. Go directly to Section 2d. Figure 1 uses the original-layout authored Jsoup candidate from that native-JUnit study. It is separate from the Maven pair in Sections 4–5.
- **Fixed Gemma 4 31B identification responses**. Section 1 regrades the saved predictions without generating new ones.
- **Secondary Gemma E4B pilot and exploratory coverage example**. Section 7 retains missing/rejected outputs and the non-activated feedback contrast. It provides no evidence that repaired feedback improves agent learning. Its public native replay uses `GEMMA_PAIRED_REPAIR_REPLAY_KIT_v2.zip`, also in Project Files.
- **Secondary Figure 5 reconstruction**. Section 1b calculates scores from the depicted sets without the original agent patches. Leaderboard effects remain unknown.

## Recorded replay and private historical inputs

This public copy preserves the recorded outputs from the submitted Kaggle notebook. Section 1b replaces its embedded TEBench functions with checksum-verified upstream downloads.

The default Python cells recheck embedded saved records. Section 1b fetches checksum-pinned upstream functions on first use and caches them locally. Its first run requires internet access. Fresh native execution uses the public recipe above. The retained outputs come from the prior bounded Azure CPU replay. Saved fresh-worker results document the reproductions already performed.

Earlier Maven and Codec worker recipes require separately retained private input packets. Their embedded saved-record replays remain available, and the later public-source native studies are separate experiments on the same affected tasks.

Select no accelerator for the saved-record replay.


## 1. Reproduce the identification scores

The frozen supplied-file task maps ID `1` to `EntitiesTest.controlCharactersAreEscaped` in `jsoup__001752f1`. The original projected reference is empty. The annotation-layout control restores that one existing method. The model outputs are fixed between grades. A second task, `commons-collections__19119be9`, has empty predicted and projected sets. The release assigns F1 zero when the denominator is zero.

| Task | Seeds | Prediction | Original TP/FP/FN and F1 | Controlled TP/FP/FN and F1 |
|---|---|---|---|---|
| Jsoup | 1729, 2718, 3141, 5772 | `{1}` each | 0/1/0, **0** | 1/0/0, **1** |
| Commons Collections | 1729, 2718, 3141, 5772 | Empty each | 0/0/0, **0** | 0/0/0, **0** |

Each seed repeats the same task. The eight outputs cover two tasks. The metric computes F1 as `2 × TP / (2 × TP + FP + FN)`.

The cell below parses the exact archived final responses and resolves the emitted IDs using the frozen method inventory. It then recomputes each score against both projected references.

The archived responses came from Gemma 4 31B instruction-tuned Q4_0 GGUF, served by llama.cpp b11552 (`23b0202`) on one NVIDIA L4. The context was 8,192 tokens. The run disabled thinking and used a 3,072-token response limit, temperature 0.7, top-k 40 and top-p 0.95. [Pinned Gemma model revision](https://huggingface.co/google/gemma-4-31B-it-qat-q4_0-gguf/tree/59dde24573e7e61570dba08b18a2e1fe246955ed)


In [1]:
import json


def identification_counts(predicted_pairs, reference_pairs):
    prediction = {tuple(pair) for pair in predicted_pairs}
    reference = {tuple(pair) for pair in reference_pairs}
    true_positives = len(prediction & reference)
    false_positives = len(prediction - reference)
    false_negatives = len(reference - prediction)
    denominator = 2 * true_positives + false_positives + false_negatives
    f1 = 2 * true_positives / denominator if denominator else 0.0
    return [true_positives, false_positives, false_negatives, f1]


# The ID map includes every ID emitted in the archived responses.
method_pairs = {'collections': {}, 'jsoup': {1: ['src/test/java/org/jsoup/nodes/EntitiesTest.java', 'controlCharactersAreEscaped']}}
references = {'collections': [[], []], 'jsoup': [[], [['src/test/java/org/jsoup/nodes/EntitiesTest.java', 'controlCharactersAreEscaped']]]}

# Rows contain the run ID, task family, final response, and saved IDs, pairs and scores.
records = [
    ('jsoup-1729', 'jsoup', '```json\n{"modify_ids": [1]}\n```', [1], [['src/test/java/org/jsoup/nodes/EntitiesTest.java', 'controlCharactersAreEscaped']], [0, 1, 0, 0.0], [1, 0, 0, 1.0]),
    ('collections-1729', 'collections', '```json\n{\n  "modify_ids": []\n}\n```', [], [], [0, 0, 0, 0.0], [0, 0, 0, 0.0]),
    ('collections-2718', 'collections', '```json\n{\n  "modify_ids": []\n}\n```', [], [], [0, 0, 0, 0.0], [0, 0, 0, 0.0]),
    ('jsoup-2718', 'jsoup', '```json\n{\n  "modify_ids": [1]\n}\n```', [1], [['src/test/java/org/jsoup/nodes/EntitiesTest.java', 'controlCharactersAreEscaped']], [0, 1, 0, 0.0], [1, 0, 0, 1.0]),
    ('jsoup-3141', 'jsoup', '```json\n{\n  "modify_ids": [\n    1\n  ]\n}\n```', [1], [['src/test/java/org/jsoup/nodes/EntitiesTest.java', 'controlCharactersAreEscaped']], [0, 1, 0, 0.0], [1, 0, 0, 1.0]),
    ('collections-3141', 'collections', '```json\n{\n  "modify_ids": []\n}\n```', [], [], [0, 0, 0, 0.0], [0, 0, 0, 0.0]),
    ('collections-5772', 'collections', '```json\n{\n  "modify_ids": []\n}\n```', [], [], [0, 0, 0, 0.0], [0, 0, 0, 0.0]),
    ('jsoup-5772', 'jsoup', '```json\n{"modify_ids": [1]}\n```', [1], [['src/test/java/org/jsoup/nodes/EntitiesTest.java', 'controlCharactersAreEscaped']], [0, 1, 0, 0.0], [1, 0, 0, 1.0]),
]

opening_fence = '```json\n'
closing_fence = '\n```'
for run_id, family, response, saved_ids, saved_pairs, saved_original, saved_controlled in records:
    assert response.startswith(opening_fence) and response.endswith(closing_fence), run_id
    payload = json.loads(response.removeprefix(opening_fence).removesuffix(closing_fence))
    assert isinstance(payload, dict) and set(payload) == {'modify_ids'}, run_id
    ids = payload['modify_ids']
    assert isinstance(ids, list) and all(type(method_id) is int for method_id in ids), run_id
    assert ids == saved_ids and len(ids) == len(set(ids)), run_id
    prediction = sorted(method_pairs[family][method_id] for method_id in ids)
    assert prediction == saved_pairs, run_id

    original, controlled = references[family]
    before = identification_counts(prediction, original)
    after = identification_counts(prediction, controlled)
    assert before == saved_original and after == saved_controlled, run_id
    print(f'{run_id} | modify_ids={ids} | {before} -> {after}')


jsoup-1729 | modify_ids=[1] | [0, 1, 0, 0.0] -> [1, 0, 0, 1.0]
collections-1729 | modify_ids=[] | [0, 0, 0, 0.0] -> [0, 0, 0, 0.0]
collections-2718 | modify_ids=[] | [0, 0, 0, 0.0] -> [0, 0, 0, 0.0]
jsoup-2718 | modify_ids=[1] | [0, 1, 0, 0.0] -> [1, 0, 0, 1.0]
jsoup-3141 | modify_ids=[1] | [0, 1, 0, 0.0] -> [1, 0, 0, 1.0]
collections-3141 | modify_ids=[] | [0, 0, 0, 0.0] -> [0, 0, 0, 0.0]
collections-5772 | modify_ids=[] | [0, 0, 0, 0.0] -> [0, 0, 0, 0.0]
jsoup-5772 | modify_ids=[1] | [0, 1, 0, 0.0] -> [1, 0, 0, 1.0]


## 1b. Secondary Figure 5 reconstruction and projected scores

For the main experiment and Figure 1, continue to Section 2d.

This post-hoc calculation uses the methods depicted in TEBench Figure 5. The source paper identifies Claude Code, Codex CLI and OpenCode under the Sonnet backbone. Its commit message and five developer method identities match the previously inspected case `jsoup__a3495822`, developer commit `a349582236a7e69ce341f3b4ee913cb8cd19b955` and parent `b6f652cef6dca7904d8ea0b971e4fc3812530015`. [Published case study and Figure 5](https://arxiv.org/html/2605.06125v1#S4.SS4) · [Jsoup developer commit](https://github.com/jhy/jsoup/commit/a349582236a7e69ce341f3b4ee913cb8cd19b955)

We manually transcribed the depicted changes. Each reconstructed set modifies `ElementTest.divAInlineable` and both `HtmlTreeBuilderStateTest.nestedAnchorElements01` and `nestedAnchorElements02`. The Codex set also adds `ElementTest.divAInlineableAfterNonBlankText`. The transcription includes no change to `ElementsTest.unwrap`, consistent with the source paper's case-study text.

The original raw agent patches are unavailable. Unshown changes could alter the complete prediction sets, and the patches' syntactic extraction could alter the labels. The table gives our calculated scores for the depicted method sets. The ordering change applies to this reconstruction. Full-patch and leaderboard scores remain unknown. This check adds a published illustration on a second task in the inspected Jsoup corpus. The original Gemma sample remains four responses on one affected task and four responses on one unaffected task.

The archived source audit records two labels in the original reference and five after the annotation-layout control, with equal full ASTs for the controlled file layouts. For each reconstructed set, we hold the prediction fixed across those references. The released projector gives any added test in `ElementTest.java` one file-level credit. That credit concerns the location of the addition. The source paper describes Codex's added behavioral coverage as partial. [TEBench case study](https://arxiv.org/html/2605.06125v1#S4.SS4)

| Agent depicted in Figure 5 | Original-label TP/FP/FN | Projected F1 (2 labels) | Layout-control TP/FP/FN | Projected F1 (5 labels) |
|---|---|---|---|---|
| Claude Code | 2/1/0 | 4/5 | 3/0/2 | 3/4 |
| Codex CLI | 2/2/0 | 2/3 | 4/0/1 | 8/9 |
| OpenCode (Sonnet) | 2/1/0 | 4/5 | 3/0/2 | 3/4 |

The Azure CPU record confirms the prediction projection and all six metric calculations for this fixed transcription, along with two metric controls. Completeness of the original diffs remains unverified. The cell below fetches the two pure functions directly from the pinned upstream revision, verifies the file and function hashes, and applies them to the retained transcription and saved result. First use requires internet access. Verified source is cached locally for offline replay. It recomputes the projection and metrics and checks them against the recorded values. The count formula in Section 1 supplies a second arithmetic check.

| Source record | SHA256 |
|---|---|
| Published Figure 5 image | `16a7916fcce159e1f5fae83ce19ff549a422624c4fdd15ac90360397a4d23b39` |
| Complete archived reference-layout audit | `50b3ba644576fce18f6b04a928d8d71e56a8c54d8d4819fe2621d7d50a51b760` |
| Released identification evaluator source | `e0d03cb23dc915be300b6f77b22fe4dba88d6d56cea9964e1033664ce3911497` |
| Fixed reconstruction input | `3cb2b43bedd90d618341bc470bd8c0cc338890b131498a2ef84e4e20bbbb98b9` |
| Azure result | `23658d11980e10ecb514c4419e9108cf6f400893f58ef2d690e69dcf37107859` |
| Selected pure function bodies | `062db4e810d999424b306c48098ae9f9cea121aa0a791d0dbd5fbdb9d040c933` |


In [2]:
from fractions import Fraction
import math
from typing import Dict, Set, Tuple


"""Fetch checksum-pinned TEBench source directly from its upstream repository."""
import ast
from hashlib import sha256
from pathlib import Path
from typing import Dict, Set, Tuple
import urllib.request

REVISION = '5f607183bd9f29b6c740eae76b0489dd0c7556d5'
SOURCES = {
    'identification': ('evaluate_user_identification.py', 18723, 'e0d03cb23dc915be300b6f77b22fe4dba88d6d56cea9964e1033664ce3911497'),
    'selection': ('update_evaluation/changed_method_extractor.py', 18793, 'd62db60856a6ede6ecba67170678393c528d8b5f9c6d6cbf61ab58a29a6c42ca'),
}
FUNCTIONS = {
    'extract_user_identified_tests': 'aef826628a2722e13fa6e6dd1c7e850f9c27b4037569d2683ff2e3446c95054a',
    'calculate_metrics': 'ac76d07cc67576bc7bfad8606906830973434f41218e03d079c2ebe6ee0ff2ae',
    '_is_annotated_test_method': 'ab234efdf0ef0a6b735ee588cea352e305256f95df8427474b9a1d0d086c92c2',
}


def source_bytes(label, cache=Path('.tebench-upstream')):
    relative, size, expected = SOURCES[label]
    cache = Path(cache)
    path = cache / relative.replace('/', '__')
    if path.exists():
        raw = path.read_bytes()
    else:
        class NoRedirect(urllib.request.HTTPRedirectHandler):
            def redirect_request(self, *args):
                return None
        url = 'https://raw.githubusercontent.com/iSEngLab/TEBench/' + REVISION + '/' + relative
        with urllib.request.build_opener(NoRedirect()).open(url, timeout=30) as response:
            raw = response.read(size + 1)
    if len(raw) != size or sha256(raw).hexdigest() != expected:
        raise ValueError('Pinned TEBench source differs: ' + relative)
    if not path.exists():
        cache.mkdir(parents=True, exist_ok=True)
        path.write_bytes(raw)
    return raw


def selected_function(label, name, cache=Path('.tebench-upstream')):
    source = source_bytes(label, cache).decode('utf-8')
    matches = [node for node in ast.walk(ast.parse(source))
               if isinstance(node, ast.FunctionDef) and node.name == name]
    if len(matches) != 1:
        raise ValueError('Expected one upstream function: ' + name)
    selected = ast.get_source_segment(source, matches[0]) + '\n'
    if sha256(selected.encode()).hexdigest() != FUNCTIONS[name]:
        raise ValueError('Pinned upstream function differs: ' + name)
    return selected


def load_identification_functions(cache=Path('.tebench-upstream')):
    namespace = {'Dict': Dict, 'Set': Set, 'Tuple': Tuple}
    for name in ('extract_user_identified_tests', 'calculate_metrics'):
        selected = selected_function('identification', name, cache)
        exec(compile(selected, '<checksum-verified-upstream>', 'exec'), namespace)
    return namespace


def prepare_predicate(path, cache=Path('.tebench-upstream')):
    selected = selected_function('selection', '_is_annotated_test_method', cache)
    Path(path).write_text(selected)
    return sha256(selected.encode()).hexdigest()



# Execute only the two checksum-verified upstream functions.
_upstream_functions = load_identification_functions()
extract_user_identified_tests = _upstream_functions['extract_user_identified_tests']
calculate_metrics = _upstream_functions['calculate_metrics']

# Manual transcription of only the changes depicted in Figure 5.
figure_changes = {'Claude Code': {'modified_files': [{'file_path': 'src/test/java/org/jsoup/nodes/ElementTest.java',
                                     'modified_methods': ['divAInlineable'],
                                     'added_methods': []},
                                    {'file_path': 'src/test/java/org/jsoup/parser/HtmlTreeBuilderStateTest.java',
                                     'modified_methods': ['nestedAnchorElements01',
                                                          'nestedAnchorElements02']}]},
 'Codex CLI': {'modified_files': [{'file_path': 'src/test/java/org/jsoup/nodes/ElementTest.java',
                                   'modified_methods': ['divAInlineable'],
                                   'added_methods': ['divAInlineableAfterNonBlankText']},
                                  {'file_path': 'src/test/java/org/jsoup/parser/HtmlTreeBuilderStateTest.java',
                                   'modified_methods': ['nestedAnchorElements01',
                                                        'nestedAnchorElements02']}]},
 'OpenCode (Sonnet)': {'modified_files': [{'file_path': 'src/test/java/org/jsoup/nodes/ElementTest.java',
                                           'modified_methods': ['divAInlineable'],
                                           'added_methods': []},
                                          {'file_path': 'src/test/java/org/jsoup/parser/HtmlTreeBuilderStateTest.java',
                                           'modified_methods': ['nestedAnchorElements01',
                                                                'nestedAnchorElements02']}]}}

# Frozen reference labels and the successful Azure CPU result.
published_case = {'instance_id': 'jsoup__a3495822',
 'references': {'original': [['src/test/java/org/jsoup/parser/HtmlTreeBuilderStateTest.java',
                              'nestedAnchorElements01'],
                             ['src/test/java/org/jsoup/parser/HtmlTreeBuilderStateTest.java',
                              'nestedAnchorElements02']],
                'normalized': [['src/test/java/org/jsoup/nodes/ElementTest.java',
                                '__FILE_LEVEL_ADD__'],
                               ['src/test/java/org/jsoup/nodes/ElementTest.java', 'divAInlineable'],
                               ['src/test/java/org/jsoup/parser/HtmlTreeBuilderStateTest.java',
                                'nestedAnchorElements01'],
                               ['src/test/java/org/jsoup/parser/HtmlTreeBuilderStateTest.java',
                                'nestedAnchorElements02'],
                               ['src/test/java/org/jsoup/select/ElementsTest.java', 'unwrap']]},
 'results': {'Claude Code': {'projected_prediction': [['src/test/java/org/jsoup/nodes/ElementTest.java',
                                                       'divAInlineable'],
                                                      ['src/test/java/org/jsoup/parser/HtmlTreeBuilderStateTest.java',
                                                       'nestedAnchorElements01'],
                                                      ['src/test/java/org/jsoup/parser/HtmlTreeBuilderStateTest.java',
                                                       'nestedAnchorElements02']],
                             'grades': {'original': {'true_positives': 2,
                                                     'false_positives': 1,
                                                     'false_negatives': 0,
                                                     'precision': 0.6666666666666666,
                                                     'recall': 1.0,
                                                     'f1_score': 0.8,
                                                     'gt_total': 2,
                                                     'user_total': 3,
                                                     'f1_fraction': '4/5'},
                                        'normalized': {'true_positives': 3,
                                                       'false_positives': 0,
                                                       'false_negatives': 2,
                                                       'precision': 1.0,
                                                       'recall': 0.6,
                                                       'f1_score': 0.7499999999999999,
                                                       'gt_total': 5,
                                                       'user_total': 3,
                                                       'f1_fraction': '3/4'}}},
             'Codex CLI': {'projected_prediction': [['src/test/java/org/jsoup/nodes/ElementTest.java',
                                                     '__FILE_LEVEL_ADD__'],
                                                    ['src/test/java/org/jsoup/nodes/ElementTest.java',
                                                     'divAInlineable'],
                                                    ['src/test/java/org/jsoup/parser/HtmlTreeBuilderStateTest.java',
                                                     'nestedAnchorElements01'],
                                                    ['src/test/java/org/jsoup/parser/HtmlTreeBuilderStateTest.java',
                                                     'nestedAnchorElements02']],
                           'grades': {'original': {'true_positives': 2,
                                                   'false_positives': 2,
                                                   'false_negatives': 0,
                                                   'precision': 0.5,
                                                   'recall': 1.0,
                                                   'f1_score': 0.6666666666666666,
                                                   'gt_total': 2,
                                                   'user_total': 4,
                                                   'f1_fraction': '2/3'},
                                      'normalized': {'true_positives': 4,
                                                     'false_positives': 0,
                                                     'false_negatives': 1,
                                                     'precision': 1.0,
                                                     'recall': 0.8,
                                                     'f1_score': 0.888888888888889,
                                                     'gt_total': 5,
                                                     'user_total': 4,
                                                     'f1_fraction': '8/9'}}},
             'OpenCode (Sonnet)': {'projected_prediction': [['src/test/java/org/jsoup/nodes/ElementTest.java',
                                                             'divAInlineable'],
                                                            ['src/test/java/org/jsoup/parser/HtmlTreeBuilderStateTest.java',
                                                             'nestedAnchorElements01'],
                                                            ['src/test/java/org/jsoup/parser/HtmlTreeBuilderStateTest.java',
                                                             'nestedAnchorElements02']],
                                   'grades': {'original': {'true_positives': 2,
                                                           'false_positives': 1,
                                                           'false_negatives': 0,
                                                           'precision': 0.6666666666666666,
                                                           'recall': 1.0,
                                                           'f1_score': 0.8,
                                                           'gt_total': 2,
                                                           'user_total': 3,
                                                           'f1_fraction': '4/5'},
                                              'normalized': {'true_positives': 3,
                                                             'false_positives': 0,
                                                             'false_negatives': 2,
                                                             'precision': 1.0,
                                                             'recall': 0.6,
                                                             'f1_score': 0.7499999999999999,
                                                             'gt_total': 5,
                                                             'user_total': 3,
                                                             'f1_fraction': '3/4'}}}}}

for agent, observation in published_case['results'].items():
    prediction = extract_user_identified_tests(figure_changes[agent])
    assert [list(pair) for pair in sorted(prediction)] == observation['projected_prediction'], agent
    replayed = []
    for variant in ('original', 'normalized'):
        reference = {tuple(pair) for pair in published_case['references'][variant]}
        metrics = calculate_metrics(reference, prediction)
        saved = observation['grades'][variant]
        for field, value in metrics.items():
            assert math.isclose(value, saved[field], abs_tol=1e-14), (agent, variant, field)

        counts = identification_counts(prediction, reference)
        expected_counts = [saved['true_positives'], saved['false_positives'], saved['false_negatives']]
        assert counts[:3] == expected_counts
        exact_f1 = Fraction(saved['f1_fraction'])
        assert math.isclose(counts[3], float(exact_f1), abs_tol=1e-14)
        replayed.append(f"{counts[:3]} F1={exact_f1}")
    print(f"{agent} | {' -> '.join(replayed)}")


Claude Code | [2, 1, 0] F1=4/5 -> [3, 0, 2] F1=3/4
Codex CLI | [2, 2, 0] F1=2/3 -> [4, 0, 1] F1=8/9
OpenCode (Sonnet) | [2, 1, 0] F1=4/5 -> [3, 0, 2] F1=3/4


## 2. Verify the required method and layout control

The archived normalization moved 20 eligible inline annotations in each `EntitiesTest` source variant, including the target annotation. Each layout pair produced identical `EntitiesTest` and test-runner classfiles. Against the same changed production classes, the old target test failed in both fresh runs per layout and the updated target passed both. The neighboring `escapeDefaults` test passed across all four variants.

| State | Target results in original layout | Target results in normalized layout | Recorded `EntitiesTest.class` SHA256 in both layouts |
|---|---|---|---|
| Old test | Two failures | Two failures | `eeb7250045776ed9eb215a0a14f1687989ea35ec1d9b9f419be107978c02cf01` |
| Updated test | Two passes | Two passes | `781616326e359f0afcb3fd72521d61165b21c80963b8b7616af736331d3e0bd8` |

The file-wide intervention restores one label in the projected existing-method reference. The direct classfile comparison covers `EntitiesTest` and its runner.


## 2b. Native execution in Commons Codec

This source-exposed development check uses `commons-codec__d594dd25`, developer commit `d594dd25868063be2ea5a7744123393d66cad479` and parent `c7fbb1778e87f3941c33d2ec3e7c59b6550fafbc`. The natural source has two added parameterized methods, `testDecodeEncodeStandard(String)` and `testDecodeEncodeUrl(String)`. Each has nine strings in a multiline `@ValueSource` annotation. The released annotation predicate stops at the closing delimiter before reaching `@ParameterizedTest`. [Pinned developer change](https://github.com/apache/commons-codec/commit/d594dd25868063be2ea5a7744123393d66cad479) · [Pinned selector source](https://github.com/iSEngLab/TEBench/blob/5f607183bd9f29b6c740eae76b0489dd0c7556d5/update_evaluation/changed_method_extractor.py)

The check fixes eight matched changed-method identities from the archived source panel. The released predicate selects six ordinary methods and omits the two parameterized methods. An earlier raw-diff reconstruction additionally names an unchanged context method. That extra signature remains outside this matched changed-method check. The targeted expansion adds the two omitted methods to the same six-method selection.

An Azure CPU worker compiled all 83 developer production sources and the complete `Base64Test` class, with the retained dependency artifacts. It used OpenJDK 17.0.20.1 and JUnit Platform 1.11.0. Production, test and source hash maps stayed fixed through all six JVM runs. Class-loading logs bound `Base64` and `Base64Test` to those fresh compiled directories. JUnit XML records the invocation identities, counts and outcomes. Only the selected-method arguments varied across the three arms.

| Direct JUnit arm | Selected methods | Started per run | Passed | Failed or skipped | Repetitions |
|---|---|---|---|---|---|
| Released selection | Six ordinary methods | 6 | 6 | 0 | 2 |
| Targeted expansion | Those six plus the two omitted methods | 24 | 24 | 0 | 2 |
| Omitted-only control | The two parameterized methods | 18 | 18 | 0 | 2 |

Each omitted method accounts for nine passing invocations in both expanded runs and both direct controls. The order was released, expanded, omitted-only, omitted-only, expanded, released. This establishes actual missing executions in a repository outside Jsoup. All arms passed. Historical Maven gate behavior, composite scores and agent outcomes remain unmeasured by this check. The existing identification audit retains these additions, with file-level credit for added tests. The original Gemma sample remains unchanged.

The first CPU attempt stopped at a class-origin admission check because Java's file URI and Python's URI used different slash forms. The retry compared canonical filesystem paths while preserving the scientific input, assertions, selectors and count requirements. The first failure remains retained. The original two workers and their temporary inputs were removed. The original successful result supplies the default saved records below.

A fresh Azure CPU worker executed the notebook's Section 2b Bash block byte for byte. The shell returned zero using `/usr/local/bin/python3` at Python 3.12.15. All six rows reproduced, including the two 6-versus-24 comparisons and both 18-invocation controls. The fresh source and classfile maps, native commands, Java version receipt, invocation identities and runtime origins match the original run. This verifies the recipe on the same frozen case. The fresh worker and its temporary input were also removed.

The next cell replays the six saved XML projections without external files. Set `CODEC_RESULT_ARCHIVE` to either retained successful `result.tar.gz` to inspect raw evidence. Both modes check the archive hash, admission report, source/classfile identity record, native command structure, class-loading origins and six XML files. The fresh mode additionally checks the returned Bash block and Python receipt. XML timestamps and execution times may differ between workers. The comparison holds invocation identities, counts and outcomes fixed.

The full native rerun uses the separately retained `AZURE_CODEC_INPUT_PRIVATE_v2.tar.gz` with the pinned image and Python 3.12. The byte-for-byte tested Bash block appears below.

| Source record | SHA256 |
|---|---|
| Complete retained source package | `2ee8b42c207811cf03a0e329b7c21702698d37d20fd99ddbeeee6c35ca624452` |
| Developer Base64Test.java | `fc904c0c18fbc7a7dfef314b6f11a3429b3da29811f36244c6c587c1f0f19644` |
| Released selector source | `d62db60856a6ede6ecba67170678393c528d8b5f9c6d6cbf61ab58a29a6c42ca` |
| Exact selected annotation predicate | `ab234efdf0ef0a6b735ee588cea352e305256f95df8427474b9a1d0d086c92c2` |
| Frozen worker input manifest | `8e193c5fbf07143b13a1ab9db4cce75c06aeeed5639c6d3974414c870bbff592` |
| Source and classfile identity record | `c1221f368552b7641bec39c1a2375e336bacb3dee2abd0b62f26043634b15456` |
| Original successful Azure result archive | `d89b1f3ea773c7b5e20cc801065a57e771a6953e79d1d51ec6d5992133cc387b` |
| Fresh notebook reproduction archive | `1002c397af1d2fb327a6f78cffa77f9897d34b96598b40c5a4491c66d2249e77` |
| Executed Codec Bash block | `8ed8a99c86d3cbccb126e74fbdaf7484cad6ef26e8fde2bdc0147aea05a6c2fc` |
| Notebook used by the fresh worker (v8) | `fda85bd1a0c109443cf26f114dd225f641f500d097cefcf0ae28d721dcbacb8a` |

The retained full input has SHA256 `c0b9c11acc2950353d0798eda9f1aba78370636db7c3e649f3cf7db400351d3f`. For a fresh native run, stage it on a fresh Linux amd64 worker using the image pinned in Section 5, with one vCPU and one GiB RAM. Use Python 3.12 and a fresh scratch path. The block calls the same scientific function directly and leaves results on the worker for collection.

```bash
set -eu
input=/inputs/AZURE_CODEC_INPUT_PRIVATE_v2.tar.gz
printf '%s  %s\n' 'c0b9c11acc2950353d0798eda9f1aba78370636db7c3e649f3cf7db400351d3f' "$input" | sha256sum -c -
mkdir -m 755 /tmp/tebench-pair
tar -xzf "$input" -C /tmp/tebench-pair
/usr/bin/timeout -k 8s 275s /usr/local/bin/python3 -I -S -B - <<'PYCODE'
import json
import runpy

api = runpy.run_path("/tmp/tebench-pair/azure_guest.py")
output = api["OUTPUT"]
output.mkdir(mode=0o755)
report = {"completed": False, "admitted": False}
try:
    api["run"](report)
finally:
    (output / "REPORT.json").write_text(json.dumps(report, indent=2) + "\n")
assert report["completed"] and report["admitted"]
PYCODE
```

Collect `/tmp/tebench-pair/results/REPORT.json`, `CLASSFILES.json` and the six `runs/*/xml/TEST-junit-jupiter.xml` reports before deleting the worker. Admission requires the six rows above, nine invocations per omitted method, unchanged fixture hashes and verified runtime origins.


In [3]:
from collections import Counter
from hashlib import sha256
import json
from pathlib import Path, PurePosixPath
import tarfile
from urllib.parse import unquote, urlsplit
import xml.etree.ElementTree as ET


CODEC_RESULT_ARCHIVE = None

# Original saved XML projections and fresh-worker reproduction receipts.
codec_evidence = {'archive_sha256': 'd89b1f3ea773c7b5e20cc801065a57e771a6953e79d1d51ec6d5992133cc387b',
 'instance_id': 'commons-codec__d594dd25',
 'source_archive_sha256': '2ee8b42c207811cf03a0e329b7c21702698d37d20fd99ddbeeee6c35ca624452',
 'source_test_sha256': 'fc904c0c18fbc7a7dfef314b6f11a3429b3da29811f36244c6c587c1f0f19644',
 'selector_source_sha256': 'd62db60856a6ede6ecba67170678393c528d8b5f9c6d6cbf61ab58a29a6c42ca',
 'predicate_sha256': 'ab234efdf0ef0a6b735ee588cea352e305256f95df8427474b9a1d0d086c92c2',
 'input_manifest_sha256': '8e193c5fbf07143b13a1ab9db4cce75c06aeeed5639c6d3974414c870bbff592',
 'classfile_record_sha256': 'c1221f368552b7641bec39c1a2375e336bacb3dee2abd0b62f26043634b15456',
 'released_methods': ['testDecodeBase64DiffChars',
                      'testDecodeBase64StandardDiffChars',
                      'testDecodeBase64UrlDiffChars',
                      'testIsStringBase64',
                      'testIsStringBase64Standard',
                      'testIsStringBase64Url'],
 'omitted_methods': ['testDecodeEncodeStandard', 'testDecodeEncodeUrl'],
 'runs': [{'arm': 'released',
           'started': 6,
           'passed': 6,
           'failed': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testIsStringBase64Url': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()'}],
           'xml_sha256': '69ff1153b67c222da4f4ef0513cc73784fbaf926c6eed04662f715c608ecd5fc',
           'logs': 'runs/00-released',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True},
          {'arm': 'expanded',
           'started': 24,
           'passed': 24,
           'failed': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64Url': 1,
                                      'testDecodeEncodeUrl': 9,
                                      'testDecodeEncodeStandard': 9,
                                      'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[1]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[3]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[2]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[5]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[9]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[4]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[8]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[3]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[2]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[1]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[5]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[4]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[7]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[6]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[9]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[8]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[7]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[6]'}],
           'xml_sha256': 'd0a08265b8dc61faaed122c3034b2871925dbfbbd91953f9eb1886de03d58ac4',
           'logs': 'runs/01-expanded',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True},
          {'arm': 'omitted_only',
           'started': 18,
           'passed': 18,
           'failed': 0,
           'skipped': 0,
           'invocations_per_method': {'testDecodeEncodeUrl': 9, 'testDecodeEncodeStandard': 9},
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[1]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[3]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[2]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[5]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[9]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[4]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[8]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[3]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[2]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[1]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[5]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[4]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[7]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[6]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[9]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[8]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[7]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[6]'}],
           'xml_sha256': '23f811ff52f737e4475fffdac65ad9934feebc62bddf568c0aef8d6496b561cf',
           'logs': 'runs/02-omitted_only',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True},
          {'arm': 'omitted_only',
           'started': 18,
           'passed': 18,
           'failed': 0,
           'skipped': 0,
           'invocations_per_method': {'testDecodeEncodeUrl': 9, 'testDecodeEncodeStandard': 9},
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[1]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[3]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[2]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[5]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[9]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[4]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[8]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[3]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[2]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[1]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[5]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[4]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[7]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[6]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[9]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[8]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[7]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[6]'}],
           'xml_sha256': 'd8c3fbe95b2c8fc84421e5c7fedc35023cccb4e6e55721cca47436715254a6de',
           'logs': 'runs/03-omitted_only',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True},
          {'arm': 'expanded',
           'started': 24,
           'passed': 24,
           'failed': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64Url': 1,
                                      'testDecodeEncodeUrl': 9,
                                      'testDecodeEncodeStandard': 9,
                                      'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[1]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[3]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[2]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[5]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[9]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[4]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[8]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[3]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[2]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[1]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[5]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[4]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[7]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[6]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[9]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[8]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[7]'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[6]'}],
           'xml_sha256': '2978e8f65f2b487eac05ef67440c05d2f379874f236989e050f7bc653fbcf442',
           'logs': 'runs/04-expanded',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True},
          {'arm': 'released',
           'started': 6,
           'passed': 6,
           'failed': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testIsStringBase64Url': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()'},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()'}],
           'xml_sha256': 'b3cc2f9a14678c54750059b1bddc7c444a18a23cea982571d27c0f4ab27622b5',
           'logs': 'runs/05-released',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True}]}

codec_reproduction = {'archive_sha256': '1002c397af1d2fb327a6f78cffa77f9897d34b96598b40c5a4491c66d2249e77',
 'executed_notebook_sha256': 'fda85bd1a0c109443cf26f114dd225f641f500d097cefcf0ae28d721dcbacb8a',
 'script_sha256': '8ed8a99c86d3cbccb126e74fbdaf7484cad6ef26e8fde2bdc0147aea05a6c2fc',
 'python_receipt': {'shell_exit': 0,
                    'image_python': '/usr/local/bin/python3',
                    'image_python_version': '3.12.15',
                    'script_sha256': '8ed8a99c86d3cbccb126e74fbdaf7484cad6ef26e8fde2bdc0147aea05a6c2fc'},
 'xml_sha256': ['a6c5edf535a52aeb5fdb44269b647df7d619aa5d2842c9205946419a86aae110',
                '8d82249eec241c2d6803814356ba0cc3d09ee6b4f7a1f5673230f5c46249fb45',
                '78ff2d85d9c68d7fec1366451dc67cd342d269f2665eda16d408e034a1502590',
                'bf077c06e36953d6bb664b55ebc639359ce2c490fd4e5bdbb215c58ddf00b651',
                '88f2a442e5db3e8ffe99da496874ae0e48ba1ba6d0827f7e5de996431a69caba',
                '02ee0f0172fca2e15c345e91a9aae02ea5450e062dda280b5261ea22a2d04d0f']}

if CODEC_RESULT_ARCHIVE is not None:
    codec_path = Path(CODEC_RESULT_ARCHIVE)
    assert codec_path.stat().st_size <= 4 * 1024 * 1024
    archive_sha = sha256(codec_path.read_bytes()).hexdigest()
    assert archive_sha in {
        codec_evidence['archive_sha256'], codec_reproduction['archive_sha256'],
    }
    is_fresh = archive_sha == codec_reproduction['archive_sha256']
    expected_xml_hashes = (
        codec_reproduction['xml_sha256'] if is_fresh
        else [run['xml_sha256'] for run in codec_evidence['runs']]
    )
    with tarfile.open(codec_path) as archive:
        raw_report = json.load(archive.extractfile('REPORT.json'))
        assert raw_report['completed'] and raw_report['admitted']
        classfiles = archive.extractfile('CLASSFILES.json').read()
        assert sha256(classfiles).hexdigest() == codec_evidence['classfile_record_sha256']
        assert raw_report['fixed_classfiles'] == json.loads(classfiles)
        assert raw_report['input_manifest_sha256'] == codec_evidence['input_manifest_sha256']
        assert raw_report['source_test_sha256'] == codec_evidence['source_test_sha256']
        assert len(raw_report['runs']) == len(codec_evidence['runs']) == len(expected_xml_hashes) == 6
        command_bases = []
        for original in raw_report['runs']:
            command = original['command']
            assert command['returncode'] == 0
            assert not command['timed_out'] and not command['output_limited']
            args = command['argv']
            command_bases.append(args[:args.index('--select-method')])
        assert all(args == command_bases[0] for args in command_bases)
        expected_origins = {
            'org.apache.commons.codec.binary.Base64': '/tmp/tebench-pair/compiled/production',
            'org.apache.commons.codec.binary.Base64Test': '/tmp/tebench-pair/compiled/tests',
        }
        for index, (saved, original) in enumerate(zip(codec_evidence['runs'], raw_report['runs'])):
            comparable = {key: value for key, value in saved.items()
                          if key not in {'xml_sha256', 'testcases'}}
            assert all(original[key] == value for key, value in comparable.items())
            expected_rows = sorted((row['class'], row['name']) for row in saved['testcases'])
            original_rows = sorted((row['class'], row['name']) for row in original['testcases'])
            assert original_rows == expected_rows
            xml_path = saved['logs'] + '/xml/TEST-junit-jupiter.xml'
            xml_bytes = archive.extractfile(xml_path).read()
            assert sha256(xml_bytes).hexdigest() == original['xml_sha256'] == expected_xml_hashes[index]
            suite = ET.fromstring(xml_bytes)
            cases = suite.findall('testcase')
            assert int(suite.attrib['tests']) == len(cases) == saved['started']
            assert all(int(suite.attrib[field]) == 0 for field in ('errors', 'failures', 'skipped'))
            assert not any(case.find(tag) is not None
                           for case in cases for tag in ('failure', 'error', 'skipped'))
            rows = [{'class': case.attrib['classname'], 'name': case.attrib['name']} for case in cases]
            assert rows == original['testcases']
            assert sorted((row['class'], row['name']) for row in rows) == expected_rows

            classlog = archive.extractfile(saved['logs'] + '/classload.log').read().decode()
            for classname, expected_path in expected_origins.items():
                marker = '[class,load] ' + classname + ' source: '
                origins = [line.split(' source: ', 1)[1]
                           for line in classlog.splitlines() if marker in line]
                assert len(origins) == 1
                uri = urlsplit(origins[0])
                assert uri.scheme == 'file' and not uri.netloc and not uri.query and not uri.fragment
                assert PurePosixPath(unquote(uri.path)) == PurePosixPath(expected_path)
        if is_fresh:
            receipt = json.load(archive.extractfile('notebook_status.json'))
            assert receipt == codec_reproduction['python_receipt']
            script = archive.extractfile('notebook_steps.sh').read()
            assert sha256(script).hexdigest() == receipt['script_sha256'] == codec_reproduction['script_sha256']
            assert receipt['shell_exit'] == 0
            assert receipt['image_python'] == '/usr/local/bin/python3'
            assert receipt['image_python_version'] == '3.12.15'
            assert archive.extractfile('notebook_steps.stderr').read() == b''
            stdout = archive.extractfile('notebook_steps.stdout').read().decode().strip()
            assert stdout == '/inputs/AZURE_CODEC_INPUT_PRIVATE_v2.tar.gz: OK'

released_methods = set(codec_evidence['released_methods'])
omitted_methods = set(codec_evidence['omitted_methods'])
assert len(released_methods) == 6 and len(omitted_methods) == 2
assert not released_methods & omitted_methods
expected_methods = {
    'released': released_methods,
    'expanded': released_methods | omitted_methods,
    'omitted_only': omitted_methods,
}
assert [run['arm'] for run in codec_evidence['runs']] == [
    'released', 'expanded', 'omitted_only', 'omitted_only', 'expanded', 'released',
]
for index, run in enumerate(codec_evidence['runs']):
    assert run['fixed_fixture_verified'] and run['runtime_origins_verified']
    assert run['failed'] == run['skipped'] == 0
    rows = run['testcases']
    assert all(row['class'] == 'org.apache.commons.codec.binary.Base64Test' for row in rows)
    assert len({row['name'] for row in rows}) == len(rows)
    counts = Counter(row['name'].split('(', 1)[0] for row in rows)
    expected_counts = {
        method: (9 if method in omitted_methods else 1)
        for method in expected_methods[run['arm']]
    }
    assert dict(counts) == run['invocations_per_method'] == expected_counts
    assert len(rows) == run['started'] == run['passed'] == sum(expected_counts.values())
    print(f"codec {index + 1} {run['arm']} | {len(rows)} started | {run['passed']} passed | 0 failed | 0 skipped")


codec 1 released | 6 started | 6 passed | 0 failed | 0 skipped
codec 2 expanded | 24 started | 24 passed | 0 failed | 0 skipped
codec 3 omitted_only | 18 started | 18 passed | 0 failed | 0 skipped
codec 4 omitted_only | 18 started | 18 passed | 0 failed | 0 skipped
codec 5 expanded | 24 started | 24 passed | 0 failed | 0 skipped
codec 6 released | 6 started | 6 passed | 0 failed | 0 skipped


## 2c. Authored Commons Codec candidate changes pass to fail

Researchers authored the incorrect candidate for this controlled experiment on `commons-codec__d594dd25`, the same task as Section 2b. Section 2b retains the unchanged developer-only control.

The candidate changes exactly one call inside `testDecodeEncodeUrl(String)`.

```diff
- final String encodedText2 = Base64.encodeBase64URLSafeString(StringUtils.getBytesUtf8(decodedText));
+ final String encodedText2 = Base64.encodeBase64String(StringUtils.getBytesUtf8(decodedText));
```

The original nine input strings, assertions, annotations and all production sources are unchanged. The developer's URL-safe call avoids padding and uses the URL alphabet. The authored standard-encoder substitution disagrees with six original expected values. Those exact six pairs were frozen in `PLAN.json` before execution. [Apache API contract](https://commons.apache.org/proper/commons-codec/apidocs/org/apache/commons/codec/binary/Base64.html#encodeBase64URLSafeString(byte%5B%5D))

The pinned released annotation predicate processes the same eight matched changed-method records separately for the developer and candidate sources. The pinned released union helper then produces the same six-method union. Both sources' multiline annotations hide the two parameterized methods. Adding only `testDecodeEncodeUrl` changes selection without changing either compiled test class.

| Compiled test variant | Selection | Started | Passed | Failed | Native JUnit exit | Repetitions per worker |
|---|---|---|---|---|---|---|
| Unmodified developer | Released six methods | 6 | 6 | 0 | 0 | 2 |
| Unmodified developer | Restore URL method | 15 | 15 | 0 | 0 | 2 |
| Unmodified developer | Restore both methods | 24 | 24 | 0 | 0 | 2 |
| Authored incorrect candidate | Released six methods | 6 | 6 | 0 | 0 | 2 |
| Authored incorrect candidate | Restore URL method | 15 | 9 | 6 | 1 | 2 |
| Authored incorrect candidate | Restore both methods | 24 | 18 | 6 | 1 | 2 |

All failures are `org.opentest4j.AssertionFailedError` in the URL method. The expected-versus-actual pairs are `Zg` versus `Zg==`, `Zm8` versus `Zm8=`, `Zm9vYg` versus `Zm9vYg==`, `Zm9vYmE` versus `Zm9vYmE=`, `Zm9vYmF-` versus `Zm9vYmF+`, and `Zm9vYmF_` versus `Zm9vYmF/`. No errors or skipped invocations occurred.

One Azure CPU worker compiled all 83 developer production Java sources once and each complete `Base64Test` variant once. The 129 production classfiles and seven classfiles per test variant remained fixed across selectors. Class-loading origins bound the runtime classes to those directories. Within each variant, only method-selection arguments varied. The order was developer released, URL, both, candidate released, URL, both, both, URL, released, developer both, URL, released. Each row has two observations per worker on this one frozen task. The independent fresh worker described below reproduced the same twelve outcomes. The workers and temporary inputs were removed, with absence confirmed. All repetitions use the same frozen task and authored candidate.

The experiment measures native JUnit outcomes under the released selector and targeted expansions. Complete diff extraction, Maven, coverage and composite benchmark-gate outcomes remain unmeasured. The eight-method source scope and unchanged context-method exclusion from Section 2b still apply. This case adds no Gemma output, agent-error prevalence estimate or leaderboard regrade. All six failures belong to invocations of one restored parameterized method.

The next cell replays the twelve saved XML projections offline using only the standard library. Its default mode checks recorded identities, expected values and counts without running Java or rehashing live classfiles. Set `CODEC_FLIP_RESULT_ARCHIVE` to either retained original or fresh `result.tar.gz` to also check the appropriate frozen archive/report hashes, source and classfile identity maps, native commands, XML and runtime class origins. The fresh mode also checks the returned Bash block, Python receipt and verification record. Recorded fixture checks attest to the worker observations.

A fresh Azure CPU worker executed the Section 2c Bash block below byte for byte under Python 3.12.15. Its Bash exit was zero and stderr was empty. All twelve native outcomes reproduced, including the four expected native JUnit failures. Source and classfile identity maps matched byte for byte. Native command arrays, Java version receipt, invocation identities, six assertion value pairs and runtime class origins matched the original. This reproduces the same authored candidate and fixed task. The fresh worker and temporary input were deleted.

| Frozen record | SHA256 |
|---|---|
| Private full input | `81138cd4a22f8ebbf1765e9c7f153ad5c45e18029711890d2726df10bc7b3dde` |
| Private source packet | `e34c89e0cbcbbf13609ffba024eb93376acfb93afa2ce6007e136d1225536132` |
| Input manifest | `2fb14359746d6e4fb636a229993ef98e895a978e0e5ba92567a7bb8b3df54bb0` |
| Developer test source | `fc904c0c18fbc7a7dfef314b6f11a3429b3da29811f36244c6c587c1f0f19644` |
| Authored candidate test source | `deca710a5ff35b3764f08c53bdbc87f27c062aedc524dfa5e9c2b8219d779ca5` |
| Source and classfile identity record | `f67dc86a5bfd51a9e6384370dd5c7e26cec807f2847f722a3ec959c6ad1a90a3` |
| Original successful Azure result | `30937337dbed976a267059bfff773a33157aeb1e6c4dda5f005857fd3371c23c` |
| Fresh exact-Bash reproduction result | `26351b5fe483f5e3fe2941d87aed9caffb4cd544254b93640a2f49e26e04cc4a` |
| Executed Section 2c Bash block | `2cf91b0b8cd904d6fded3956984da8ff270f4b43d8aa554fe429ef38684fbe30` |
| Notebook executed by the fresh worker (v10) | `b72727db9e60a11a54f946275ca774c5e800b919311c3f38dd549d968f2b524a` |

### Reproduce on a fresh Azure CPU worker

Use the separately retained `AZURE_CODEC_FLIP_INPUT_PRIVATE_v1.tar.gz`. The public notebook does not contain the private input, source packet or raw result. The input includes the complete source, candidate, pinned Java packages, dependency JARs, scientific worker and manifest. Use a fresh root-enabled Linux amd64 Azure CPU worker with one vCPU and one GiB RAM, the pinned image below, **Python 3.12**, and an empty `/tmp/tebench-pair`. The worker needs root for package extraction and then launches compilation and JUnit under an unprivileged identity with blocked network sockets. Training and GPU inference remain paused.

`mcr.microsoft.com/devcontainers/python@sha256:86d5f7e6b9084cbedd80abeefb3e8b14eb319e28d2c7e29015324400ac754748`

The original successful worker ran the payload's scientific function through its custody wrapper. The fresh worker then executed the following exact Bash block from v10. It invokes that same frozen function directly, leaving raw records for collection. The block is unchanged in this revision.

```bash
set -eu
input=/inputs/AZURE_CODEC_FLIP_INPUT_PRIVATE_v1.tar.gz
printf '%s  %s\n' '81138cd4a22f8ebbf1765e9c7f153ad5c45e18029711890d2726df10bc7b3dde' "$input" | sha256sum -c -
mkdir -m 755 /tmp/tebench-pair
tar -xzf "$input" -C /tmp/tebench-pair
/usr/bin/timeout -k 8s 275s /usr/local/bin/python3 -I -S -B - <<'PYCODE'
import json
import runpy
import sys

assert sys.version_info[:2] == (3, 12)
api = runpy.run_path('/tmp/tebench-pair/azure_guest.py')
output = api['OUTPUT']
output.mkdir(mode=0o755)
report = {'completed': False, 'admitted': False}
try:
    api['run'](report)
finally:
    (output / 'REPORT.json').write_text(json.dumps(report, indent=2) + '\n')
assert report['completed'] and report['admitted']
assert len(report['runs']) == 12
PYCODE
```

Collect `results/REPORT.json`, `CLASSFILES.json`, compilation logs, and all twelve `runs/*/xml/TEST-junit-jupiter.xml`, `junit.stdout`, `junit.stderr` and `classload.log` files. Delete the worker after private result custody. Inspect both report admission fields and native reports. The Bash exit is zero when every expected outcome, including the four expected failing native runs, is admitted. Check native exits and XML rather than interpreting the Bash exit as all tests passing. Fresh XML timestamps and report bytes can differ. Compare invocation identities and failure values, selectors, classfile maps and outcomes with the original evidence. The frozen raw-archive replay below accepts the two retained successful archive hashes and is not a generic validator for other reruns.

The private companion packet also includes `azure_host.py`, exact authorization, `PLAN.json`, `CANDIDATE.patch`, `REPRODUCE.txt` and lifecycle records. With an existing authorized Azure CLI session, the controller replay uses the same two archives and a fresh phase ID, as documented in `REPRODUCE.txt`. No account access, cloud launch, training, inference or submission occurs when the notebook's default Python cells run.


In [4]:
from collections import Counter
from hashlib import sha256
import json
from pathlib import Path, PurePosixPath
import tarfile
from urllib.parse import unquote, urlsplit
import xml.etree.ElementTree as ET


CODEC_FLIP_RESULT_ARCHIVE = None

# Allowlisted projections of twelve saved native runs. Not new executions.
codec_flip_evidence = {'candidate_origin': 'authored one-call test-body replacement',
 'archive_sha256': '30937337dbed976a267059bfff773a33157aeb1e6c4dda5f005857fd3371c23c',
 'report_sha256': 'e7aecb05550990128a2d54bba04c5d402abaf585d370023de4f52255ca8cc370',
 'classfile_record_sha256': 'f67dc86a5bfd51a9e6384370dd5c7e26cec807f2847f722a3ec959c6ad1a90a3',
 'input_manifest_sha256': '2fb14359746d6e4fb636a229993ef98e895a978e0e5ba92567a7bb8b3df54bb0',
 'gold_test_sha256': 'fc904c0c18fbc7a7dfef314b6f11a3429b3da29811f36244c6c587c1f0f19644',
 'candidate_test_sha256': 'deca710a5ff35b3764f08c53bdbc87f27c062aedc524dfa5e9c2b8219d779ca5',
 'released_methods': ['testDecodeBase64DiffChars',
                      'testDecodeBase64StandardDiffChars',
                      'testDecodeBase64UrlDiffChars',
                      'testIsStringBase64',
                      'testIsStringBase64Standard',
                      'testIsStringBase64Url'],
 'runs': [{'role': 'gold',
           'arm': 'released',
           'selectors': ['testDecodeBase64DiffChars',
                         'testDecodeBase64StandardDiffChars',
                         'testDecodeBase64UrlDiffChars',
                         'testIsStringBase64',
                         'testIsStringBase64Standard',
                         'testIsStringBase64Url'],
           'started': 6,
           'passed': 6,
           'failed': 0,
           'errors': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testIsStringBase64Url': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'failure_details': [],
           'xml_sha256': 'd295714d2e5fa1ea90711bc88d32f50cc1c2fe897a1097d48dce71f952f4e455',
           'logs': 'runs/00-gold-released',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True,
           'native_exit': 0,
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()',
                          'failure': None}]},
          {'role': 'gold',
           'arm': 'restore_url',
           'selectors': ['testDecodeBase64DiffChars',
                         'testDecodeBase64StandardDiffChars',
                         'testDecodeBase64UrlDiffChars',
                         'testDecodeEncodeUrl',
                         'testIsStringBase64',
                         'testIsStringBase64Standard',
                         'testIsStringBase64Url'],
           'started': 15,
           'passed': 15,
           'failed': 0,
           'errors': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64Url': 1,
                                      'testDecodeEncodeUrl': 9,
                                      'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'failure_details': [],
           'xml_sha256': '6247c5d0b31bff512e0a3b7776dde5f2d92db7c9b83598b26a39a53dbe95d180',
           'logs': 'runs/01-gold-restore_url',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True,
           'native_exit': 0,
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[1]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[3]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[2]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[9]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[8]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[5]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[4]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[7]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[6]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()',
                          'failure': None}]},
          {'role': 'gold',
           'arm': 'restore_both',
           'selectors': ['testDecodeBase64DiffChars',
                         'testDecodeBase64StandardDiffChars',
                         'testDecodeBase64UrlDiffChars',
                         'testDecodeEncodeStandard',
                         'testDecodeEncodeUrl',
                         'testIsStringBase64',
                         'testIsStringBase64Standard',
                         'testIsStringBase64Url'],
           'started': 24,
           'passed': 24,
           'failed': 0,
           'errors': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64Url': 1,
                                      'testDecodeEncodeUrl': 9,
                                      'testDecodeEncodeStandard': 9,
                                      'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'failure_details': [],
           'xml_sha256': '8ee09c479de3ac1941c826dbe3aba235555eb7280105b6f8bb0649684b1b2fe0',
           'logs': 'runs/02-gold-restore_both',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True,
           'native_exit': 0,
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[1]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[3]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[2]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[5]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[9]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[4]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[8]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[3]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[2]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[1]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[5]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[4]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[7]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[6]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[9]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[8]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[7]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[6]',
                          'failure': None}]},
          {'role': 'candidate',
           'arm': 'released',
           'selectors': ['testDecodeBase64DiffChars',
                         'testDecodeBase64StandardDiffChars',
                         'testDecodeBase64UrlDiffChars',
                         'testIsStringBase64',
                         'testIsStringBase64Standard',
                         'testIsStringBase64Url'],
           'started': 6,
           'passed': 6,
           'failed': 0,
           'errors': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testIsStringBase64Url': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'failure_details': [],
           'xml_sha256': '9f3df4175a80b3f15da586cb9bd25cc6b6bf99abedb95abf89fb812c1c856455',
           'logs': 'runs/03-candidate-released',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True,
           'native_exit': 0,
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()',
                          'failure': None}]},
          {'role': 'candidate',
           'arm': 'restore_url',
           'selectors': ['testDecodeBase64DiffChars',
                         'testDecodeBase64StandardDiffChars',
                         'testDecodeBase64UrlDiffChars',
                         'testDecodeEncodeUrl',
                         'testIsStringBase64',
                         'testIsStringBase64Standard',
                         'testIsStringBase64Url'],
           'started': 15,
           'passed': 9,
           'failed': 6,
           'errors': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64Url': 1,
                                      'testDecodeEncodeUrl': 9,
                                      'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'failure_details': [{'name': 'testDecodeEncodeUrl(String)[3]',
                                'message': 'expected: <Zm8> but was: <Zm8=>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[2]',
                                'message': 'expected: <Zg> but was: <Zg==>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[9]',
                                'message': 'expected: <Zm9vYmF_> but was: <Zm9vYmF/>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[8]',
                                'message': 'expected: <Zm9vYmF-> but was: <Zm9vYmF+>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[5]',
                                'message': 'expected: <Zm9vYg> but was: <Zm9vYg==>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[6]',
                                'message': 'expected: <Zm9vYmE> but was: <Zm9vYmE=>',
                                'type': 'org.opentest4j.AssertionFailedError'}],
           'xml_sha256': '9699dba983d2117d306a48aa8c7592c091e6083a012efd288e4ba5a7061f8e33',
           'logs': 'runs/04-candidate-restore_url',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True,
           'native_exit': 1,
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[1]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[3]',
                          'failure': {'message': 'expected: <Zm8> but was: <Zm8=>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[2]',
                          'failure': {'message': 'expected: <Zg> but was: <Zg==>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[9]',
                          'failure': {'message': 'expected: <Zm9vYmF_> but was: <Zm9vYmF/>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[8]',
                          'failure': {'message': 'expected: <Zm9vYmF-> but was: <Zm9vYmF+>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[5]',
                          'failure': {'message': 'expected: <Zm9vYg> but was: <Zm9vYg==>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[4]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[7]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[6]',
                          'failure': {'message': 'expected: <Zm9vYmE> but was: <Zm9vYmE=>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()',
                          'failure': None}]},
          {'role': 'candidate',
           'arm': 'restore_both',
           'selectors': ['testDecodeBase64DiffChars',
                         'testDecodeBase64StandardDiffChars',
                         'testDecodeBase64UrlDiffChars',
                         'testDecodeEncodeStandard',
                         'testDecodeEncodeUrl',
                         'testIsStringBase64',
                         'testIsStringBase64Standard',
                         'testIsStringBase64Url'],
           'started': 24,
           'passed': 18,
           'failed': 6,
           'errors': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64Url': 1,
                                      'testDecodeEncodeUrl': 9,
                                      'testDecodeEncodeStandard': 9,
                                      'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'failure_details': [{'name': 'testDecodeEncodeUrl(String)[3]',
                                'message': 'expected: <Zm8> but was: <Zm8=>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[2]',
                                'message': 'expected: <Zg> but was: <Zg==>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[9]',
                                'message': 'expected: <Zm9vYmF_> but was: <Zm9vYmF/>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[8]',
                                'message': 'expected: <Zm9vYmF-> but was: <Zm9vYmF+>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[5]',
                                'message': 'expected: <Zm9vYg> but was: <Zm9vYg==>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[6]',
                                'message': 'expected: <Zm9vYmE> but was: <Zm9vYmE=>',
                                'type': 'org.opentest4j.AssertionFailedError'}],
           'xml_sha256': '5b1d56341bed86fe7f496062254f89bf1e38d246757006762b8490429973dbcc',
           'logs': 'runs/05-candidate-restore_both',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True,
           'native_exit': 1,
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[1]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[3]',
                          'failure': {'message': 'expected: <Zm8> but was: <Zm8=>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[2]',
                          'failure': {'message': 'expected: <Zg> but was: <Zg==>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[5]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[9]',
                          'failure': {'message': 'expected: <Zm9vYmF_> but was: <Zm9vYmF/>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[4]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[8]',
                          'failure': {'message': 'expected: <Zm9vYmF-> but was: <Zm9vYmF+>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[3]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[2]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[1]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[5]',
                          'failure': {'message': 'expected: <Zm9vYg> but was: <Zm9vYg==>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[4]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[7]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[6]',
                          'failure': {'message': 'expected: <Zm9vYmE> but was: <Zm9vYmE=>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[9]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[8]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[7]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[6]',
                          'failure': None}]},
          {'role': 'candidate',
           'arm': 'restore_both',
           'selectors': ['testDecodeBase64DiffChars',
                         'testDecodeBase64StandardDiffChars',
                         'testDecodeBase64UrlDiffChars',
                         'testDecodeEncodeStandard',
                         'testDecodeEncodeUrl',
                         'testIsStringBase64',
                         'testIsStringBase64Standard',
                         'testIsStringBase64Url'],
           'started': 24,
           'passed': 18,
           'failed': 6,
           'errors': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64Url': 1,
                                      'testDecodeEncodeUrl': 9,
                                      'testDecodeEncodeStandard': 9,
                                      'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'failure_details': [{'name': 'testDecodeEncodeUrl(String)[3]',
                                'message': 'expected: <Zm8> but was: <Zm8=>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[2]',
                                'message': 'expected: <Zg> but was: <Zg==>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[9]',
                                'message': 'expected: <Zm9vYmF_> but was: <Zm9vYmF/>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[8]',
                                'message': 'expected: <Zm9vYmF-> but was: <Zm9vYmF+>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[5]',
                                'message': 'expected: <Zm9vYg> but was: <Zm9vYg==>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[6]',
                                'message': 'expected: <Zm9vYmE> but was: <Zm9vYmE=>',
                                'type': 'org.opentest4j.AssertionFailedError'}],
           'xml_sha256': '6db8c9d41bab19bcb8f0ce33d5a192a8f8266e59f9bc480c9504526f9da822da',
           'logs': 'runs/06-candidate-restore_both',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True,
           'native_exit': 1,
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[1]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[3]',
                          'failure': {'message': 'expected: <Zm8> but was: <Zm8=>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[2]',
                          'failure': {'message': 'expected: <Zg> but was: <Zg==>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[5]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[9]',
                          'failure': {'message': 'expected: <Zm9vYmF_> but was: <Zm9vYmF/>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[4]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[8]',
                          'failure': {'message': 'expected: <Zm9vYmF-> but was: <Zm9vYmF+>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[3]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[2]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[1]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[5]',
                          'failure': {'message': 'expected: <Zm9vYg> but was: <Zm9vYg==>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[4]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[7]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[6]',
                          'failure': {'message': 'expected: <Zm9vYmE> but was: <Zm9vYmE=>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[9]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[8]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[7]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[6]',
                          'failure': None}]},
          {'role': 'candidate',
           'arm': 'restore_url',
           'selectors': ['testDecodeBase64DiffChars',
                         'testDecodeBase64StandardDiffChars',
                         'testDecodeBase64UrlDiffChars',
                         'testDecodeEncodeUrl',
                         'testIsStringBase64',
                         'testIsStringBase64Standard',
                         'testIsStringBase64Url'],
           'started': 15,
           'passed': 9,
           'failed': 6,
           'errors': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64Url': 1,
                                      'testDecodeEncodeUrl': 9,
                                      'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'failure_details': [{'name': 'testDecodeEncodeUrl(String)[3]',
                                'message': 'expected: <Zm8> but was: <Zm8=>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[2]',
                                'message': 'expected: <Zg> but was: <Zg==>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[9]',
                                'message': 'expected: <Zm9vYmF_> but was: <Zm9vYmF/>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[8]',
                                'message': 'expected: <Zm9vYmF-> but was: <Zm9vYmF+>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[5]',
                                'message': 'expected: <Zm9vYg> but was: <Zm9vYg==>',
                                'type': 'org.opentest4j.AssertionFailedError'},
                               {'name': 'testDecodeEncodeUrl(String)[6]',
                                'message': 'expected: <Zm9vYmE> but was: <Zm9vYmE=>',
                                'type': 'org.opentest4j.AssertionFailedError'}],
           'xml_sha256': 'b205fda00f1f862af7a3cb34ecfeef121155b835d585e0f00fe719c3a8daba48',
           'logs': 'runs/07-candidate-restore_url',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True,
           'native_exit': 1,
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[1]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[3]',
                          'failure': {'message': 'expected: <Zm8> but was: <Zm8=>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[2]',
                          'failure': {'message': 'expected: <Zg> but was: <Zg==>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[9]',
                          'failure': {'message': 'expected: <Zm9vYmF_> but was: <Zm9vYmF/>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[8]',
                          'failure': {'message': 'expected: <Zm9vYmF-> but was: <Zm9vYmF+>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[5]',
                          'failure': {'message': 'expected: <Zm9vYg> but was: <Zm9vYg==>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[4]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[7]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[6]',
                          'failure': {'message': 'expected: <Zm9vYmE> but was: <Zm9vYmE=>',
                                      'type': 'org.opentest4j.AssertionFailedError'}},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()',
                          'failure': None}]},
          {'role': 'candidate',
           'arm': 'released',
           'selectors': ['testDecodeBase64DiffChars',
                         'testDecodeBase64StandardDiffChars',
                         'testDecodeBase64UrlDiffChars',
                         'testIsStringBase64',
                         'testIsStringBase64Standard',
                         'testIsStringBase64Url'],
           'started': 6,
           'passed': 6,
           'failed': 0,
           'errors': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testIsStringBase64Url': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'failure_details': [],
           'xml_sha256': 'e759dbc9ab3460e2194298f1945b878435bfd0c64d554cf97388c4e7dfb764de',
           'logs': 'runs/08-candidate-released',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True,
           'native_exit': 0,
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()',
                          'failure': None}]},
          {'role': 'gold',
           'arm': 'restore_both',
           'selectors': ['testDecodeBase64DiffChars',
                         'testDecodeBase64StandardDiffChars',
                         'testDecodeBase64UrlDiffChars',
                         'testDecodeEncodeStandard',
                         'testDecodeEncodeUrl',
                         'testIsStringBase64',
                         'testIsStringBase64Standard',
                         'testIsStringBase64Url'],
           'started': 24,
           'passed': 24,
           'failed': 0,
           'errors': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64Url': 1,
                                      'testDecodeEncodeUrl': 9,
                                      'testDecodeEncodeStandard': 9,
                                      'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'failure_details': [],
           'xml_sha256': '031d5b835b4eda73c7c6f8b348279432e41ee9d9868d95a5b472a92932e867fe',
           'logs': 'runs/09-gold-restore_both',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True,
           'native_exit': 0,
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[1]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[3]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[2]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[5]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[9]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[4]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[8]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[3]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[2]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[1]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[5]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[4]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[7]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[6]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[9]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[8]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[7]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeStandard(String)[6]',
                          'failure': None}]},
          {'role': 'gold',
           'arm': 'restore_url',
           'selectors': ['testDecodeBase64DiffChars',
                         'testDecodeBase64StandardDiffChars',
                         'testDecodeBase64UrlDiffChars',
                         'testDecodeEncodeUrl',
                         'testIsStringBase64',
                         'testIsStringBase64Standard',
                         'testIsStringBase64Url'],
           'started': 15,
           'passed': 15,
           'failed': 0,
           'errors': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64Url': 1,
                                      'testDecodeEncodeUrl': 9,
                                      'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'failure_details': [],
           'xml_sha256': '8ec57df1e08ffe19f92e9c242c3e38476501541ea73e17e52e9f5acb62787e92',
           'logs': 'runs/10-gold-restore_url',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True,
           'native_exit': 0,
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[1]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[3]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[2]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[9]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[8]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[5]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[4]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[7]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeEncodeUrl(String)[6]',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()',
                          'failure': None}]},
          {'role': 'gold',
           'arm': 'released',
           'selectors': ['testDecodeBase64DiffChars',
                         'testDecodeBase64StandardDiffChars',
                         'testDecodeBase64UrlDiffChars',
                         'testIsStringBase64',
                         'testIsStringBase64Standard',
                         'testIsStringBase64Url'],
           'started': 6,
           'passed': 6,
           'failed': 0,
           'errors': 0,
           'skipped': 0,
           'invocations_per_method': {'testIsStringBase64': 1,
                                      'testDecodeBase64DiffChars': 1,
                                      'testIsStringBase64Url': 1,
                                      'testDecodeBase64UrlDiffChars': 1,
                                      'testDecodeBase64StandardDiffChars': 1,
                                      'testIsStringBase64Standard': 1},
           'failure_details': [],
           'xml_sha256': '5541c978539abca3a7b352715f961ed8ada650bfef065ae49c9be79d4b1db53e',
           'logs': 'runs/11-gold-released',
           'fixed_fixture_verified': True,
           'runtime_origins_verified': True,
           'native_exit': 0,
           'testcases': [{'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64DiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Url()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64UrlDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testDecodeBase64StandardDiffChars()',
                          'failure': None},
                         {'class': 'org.apache.commons.codec.binary.Base64Test',
                          'name': 'testIsStringBase64Standard()',
                          'failure': None}]}]}
codec_flip_reproduction = {'archive_sha256': '26351b5fe483f5e3fe2941d87aed9caffb4cd544254b93640a2f49e26e04cc4a',
 'report_sha256': '26800f94740c280ef2e4c91b332b047d4284b73aee5d660187cd36ed2404154b',
 'executed_notebook_sha256': 'b72727db9e60a11a54f946275ca774c5e800b919311c3f38dd549d968f2b524a',
 'script_sha256': '2cf91b0b8cd904d6fded3956984da8ff270f4b43d8aa554fe429ef38684fbe30',
 'verification_sha256': 'da30ec0a2d3c29b00acd77a61a3be9769e61abfd318aaa616a206bda167b7cb6',
 'python_receipt': {'shell_exit': 0,
                    'image_python': '/usr/local/bin/python3',
                    'image_python_version': '3.12.15',
                    'script_sha256': '2cf91b0b8cd904d6fded3956984da8ff270f4b43d8aa554fe429ef38684fbe30'},
 'verification_flags': {'source_and_classfile_maps_equal': True,
                        'native_commands_equal': True,
                        'java_version_receipt_equal': True,
                        'invocation_identities_and_assertion_values_equal': True,
                        'all_twelve_xml_counts_and_outcomes_verified': True,
                        'all_runtime_class_origins_verified': True},
 'xml_sha256': ['58e5c184576169c95e4fe01ad1c98f669babfddb9daa2d7395e0d8737144c794',
                '883297e161a2dd63e4da8f40e384c2b6a8af437e06767c0450a683b228540dd9',
                'dadf7a821e021ecdb5f467bc991111a9f12efbb0467551bdce7ed0b56ccdadda',
                '600723d2812300a9b41c461c758e039171913916fdc9362d4059b3013af59f66',
                'f5ceab66573c9a4092f5f05488d180aa0b8c46bcc5946ace3cd11d055ac16b66',
                '7047512318e692d784945dcaf393cff810f33a95bfbf5bfcc61cb3566c147fc9',
                '402c1602dd9d20753dc5443bb4ee95dff0e98928ae73ba731b020bdd421ada5d',
                '5032dfcfc815153babc7cc8830b9a8864141508bdeb132145214adec7ff5cd8a',
                'e614584330e6f1ed765516d83627a5dbd11e668e037dac8826ba121a235e66b6',
                'a3f0c7690bd26762b3c8970d86e6c792965ffd33116ffc546e4a635f7e7b4e6e',
                '87dd323d066b781a535d6d59db076ec577b58801c16a6deb83a0f0f344fe7bad',
                'daf9e7e713c2a4599bb4f2fd7d548cd5f8bcd0f011db9a7bc563c0be39268121']}

# A second worker reproduced the same twelve outcomes, not twelve new tasks.
assert codec_flip_reproduction['python_receipt']['shell_exit'] == 0
assert codec_flip_reproduction['python_receipt']['image_python'] == '/usr/local/bin/python3'
assert codec_flip_reproduction['python_receipt']['image_python_version'] == '3.12.15'
assert codec_flip_reproduction['python_receipt']['script_sha256'] == codec_flip_reproduction['script_sha256']
assert len(codec_flip_reproduction['xml_sha256']) == 12
assert all(codec_flip_reproduction['verification_flags'].values())

flip_released = set(codec_flip_evidence['released_methods'])
flip_omitted = {'testDecodeEncodeStandard', 'testDecodeEncodeUrl'}
flip_methods = {
    'released': flip_released,
    'restore_url': flip_released | {'testDecodeEncodeUrl'},
    'restore_both': flip_released | flip_omitted,
}
flip_order = [('gold', arm) for arm in ['released', 'restore_url', 'restore_both']]
flip_order += [('candidate', arm) for arm in [
    'released', 'restore_url', 'restore_both', 'restore_both', 'restore_url', 'released',
]]
flip_order += [('gold', arm) for arm in ['restore_both', 'restore_url', 'released']]
assert len(flip_released) == 6 and not flip_released & flip_omitted
assert [(run['role'], run['arm']) for run in codec_flip_evidence['runs']] == flip_order
assert codec_flip_evidence['candidate_origin'] == 'authored one-call test-body replacement'
flip_pairs = [
    ('Zg', 'Zg=='), ('Zm8', 'Zm8='), ('Zm9vYg', 'Zm9vYg=='),
    ('Zm9vYmE', 'Zm9vYmE='), ('Zm9vYmF-', 'Zm9vYmF+'), ('Zm9vYmF_', 'Zm9vYmF/'),
]
flip_expected_messages = sorted('expected: <' + a + '> but was: <' + b + '>'
                                for a, b in flip_pairs)

if CODEC_FLIP_RESULT_ARCHIVE is not None:
    flip_path = Path(CODEC_FLIP_RESULT_ARCHIVE)
    assert flip_path.stat().st_size <= 4 * 1024 * 1024
    flip_archive_sha = sha256(flip_path.read_bytes()).hexdigest()
    assert flip_archive_sha in {codec_flip_evidence['archive_sha256'], codec_flip_reproduction['archive_sha256']}
    flip_is_fresh = flip_archive_sha == codec_flip_reproduction['archive_sha256']
    expected_report_sha = (codec_flip_reproduction['report_sha256'] if flip_is_fresh
                           else codec_flip_evidence['report_sha256'])
    expected_xml_hashes = (codec_flip_reproduction['xml_sha256'] if flip_is_fresh
                           else [run['xml_sha256'] for run in codec_flip_evidence['runs']])
    with tarfile.open(flip_path) as archive:
        report_bytes = archive.extractfile('REPORT.json').read()
        assert sha256(report_bytes).hexdigest() == expected_report_sha
        raw_flip = json.loads(report_bytes)
        assert raw_flip['completed'] and raw_flip['admitted']
        if not flip_is_fresh:
            assert not raw_flip['training_performed'] and not raw_flip['model_inference_performed']
            assert not raw_flip['maven_executed'] and not raw_flip['benchmark_gate_regraded']
        assert raw_flip['input_manifest_sha256'] == codec_flip_evidence['input_manifest_sha256']
        assert raw_flip['classfile_counts'] == {
            'project': 166, 'candidate': 1, 'production': 129,
            'gold_classes': 7, 'candidate_classes': 7,
        }
        class_bytes = archive.extractfile('CLASSFILES.json').read()
        assert sha256(class_bytes).hexdigest() == codec_flip_evidence['classfile_record_sha256']
        flip_maps = json.loads(class_bytes)
        test_source = 'src/test/java/org/apache/commons/codec/binary/Base64Test.java'
        assert flip_maps['project'][test_source] == codec_flip_evidence['gold_test_sha256']
        assert flip_maps['candidate']['Base64Test.java'] == codec_flip_evidence['candidate_test_sha256']
        assert raw_flip['selection']['released_union'] == sorted(flip_released)
        assert sorted(raw_flip['selection']['gold']) == sorted(flip_released)
        assert sorted(raw_flip['selection']['candidate']) == sorted(flip_released)
        assert raw_flip['plan']['expected_url_failure_pairs'] == [list(pair) for pair in flip_pairs]
        assert len(raw_flip['runs']) == len(codec_flip_evidence['runs']) == 12
        flip_command_bases = {}
        for index, (saved, original) in enumerate(zip(codec_flip_evidence['runs'], raw_flip['runs'])):
            for field in ('role', 'arm', 'selectors', 'started', 'passed', 'failed', 'errors',
                          'skipped', 'invocations_per_method',
                          'logs', 'fixed_fixture_verified', 'runtime_origins_verified'):
                assert original[field] == saved[field]
            failure_key = lambda item: (item['name'], item['type'], item['message'])
            assert sorted(original['failure_details'], key=failure_key) == sorted(saved['failure_details'], key=failure_key)
            command = original['command']
            assert command['returncode'] == saved['native_exit']
            assert not command['timed_out'] and not command['output_limited']
            argv = command['argv']
            cut = argv.index('--select-method')
            role = saved['role']
            base = argv[:cut]
            assert base == flip_command_bases.setdefault(role, base)
            assert argv[0] == '/usr/lib/jvm/java-17-openjdk-amd64/bin/java'
            assert '--include-engine' in base and base[base.index('--include-engine') + 1] == 'junit-jupiter'
            assert '--fail-if-no-tests' in base
            assert argv[cut::2] == ['--select-method'] * len(saved['selectors'])
            expected_selectors = ['org.apache.commons.codec.binary.Base64Test#' + method +
                                  ('(java.lang.String)' if method in flip_omitted else '()')
                                  for method in saved['selectors']]
            assert argv[cut + 1::2] == expected_selectors
            xml_bytes = archive.extractfile(saved['logs'] + '/xml/TEST-junit-jupiter.xml').read()
            assert sha256(xml_bytes).hexdigest() == original['xml_sha256'] == expected_xml_hashes[index]
            suite = ET.fromstring(xml_bytes)
            assert int(suite.attrib['tests']) == saved['started']
            assert int(suite.attrib['failures']) == saved['failed']
            assert int(suite.attrib['errors']) == int(suite.attrib['skipped']) == 0
            actual_rows = []
            for case in suite.findall('testcase'):
                assert case.find('error') is None and case.find('skipped') is None
                failure = case.find('failure')
                actual_rows.append({
                    'class': case.attrib['classname'], 'name': case.attrib['name'],
                    'failure': dict(failure.attrib) if failure is not None else None,
                })
            row_key = lambda row: (row['class'], row['name'], json.dumps(row['failure'], sort_keys=True))
            assert sorted(actual_rows, key=row_key) == sorted(saved['testcases'], key=row_key)
            classlog = archive.extractfile(saved['logs'] + '/classload.log').read().decode()
            expected_origins = {
                'org.apache.commons.codec.binary.Base64': '/tmp/tebench-pair/compiled/production',
                'org.apache.commons.codec.binary.Base64Test': '/tmp/tebench-pair/compiled/' + role,
            }
            for classname, expected_path in expected_origins.items():
                marker = '[class,load] ' + classname + ' source: '
                origins = [line.split(' source: ', 1)[1]
                           for line in classlog.splitlines() if marker in line]
                assert len(origins) == 1
                uri = urlsplit(origins[0])
                assert uri.scheme == 'file' and not uri.netloc and not uri.query and not uri.fragment
                assert PurePosixPath(unquote(uri.path)) == PurePosixPath(expected_path)
        if flip_is_fresh:
            receipt = json.load(archive.extractfile('notebook_status.json'))
            assert receipt == codec_flip_reproduction['python_receipt']
            script = archive.extractfile('notebook_steps.sh').read()
            assert sha256(script).hexdigest() == codec_flip_reproduction['script_sha256']
            assert archive.extractfile('notebook_steps.stderr').read() == b''
            assert archive.extractfile('notebook_steps.stdout').read().decode().strip() == (
                '/inputs/AZURE_CODEC_FLIP_INPUT_PRIVATE_v1.tar.gz: OK')
            verification_bytes = archive.extractfile('VERIFICATION.json').read()
            assert sha256(verification_bytes).hexdigest() == codec_flip_reproduction['verification_sha256']
            verification = json.loads(verification_bytes)
            assert verification['verified']
            assert verification['notebook_sha256'] == codec_flip_reproduction['executed_notebook_sha256']
            assert verification['notebook_script_sha256'] == codec_flip_reproduction['script_sha256']
            assert verification['original_result_sha256'] == codec_flip_evidence['archive_sha256']
            assert verification['python_receipt'] == receipt
            assert verification['training'] == verification['gpu_inference'] == 'paused'
            assert verification['submitted'] is False
            assert verification['full_benchmark_gate_regraded'] is False
            for key, value in codec_flip_reproduction['verification_flags'].items():
                assert verification[key] == value is True

for index, run in enumerate(codec_flip_evidence['runs']):
    role, arm = run['role'], run['arm']
    expected_methods = flip_methods[arm]
    expected_failures = 6 if role == 'candidate' and arm != 'released' else 0
    expected_count = {'released': 6, 'restore_url': 15, 'restore_both': 24}[arm]
    assert run['selectors'] == sorted(expected_methods)
    assert run['fixed_fixture_verified'] and run['runtime_origins_verified']
    assert run['errors'] == run['skipped'] == 0
    assert run['started'] == expected_count
    assert run['failed'] == expected_failures
    assert run['passed'] == expected_count - expected_failures
    assert run['native_exit'] == (1 if expected_failures else 0)
    rows = run['testcases']
    assert len(rows) == expected_count == len({row['name'] for row in rows})
    assert all(row['class'] == 'org.apache.commons.codec.binary.Base64Test' for row in rows)
    counts = Counter(row['name'].split('(', 1)[0] for row in rows)
    assert dict(counts) == run['invocations_per_method'] == {
        method: (9 if method in flip_omitted else 1) for method in expected_methods
    }
    failed = [row for row in rows if row['failure'] is not None]
    assert len(failed) == expected_failures
    assert all(row['name'].startswith('testDecodeEncodeUrl(String)') for row in failed)
    assert all(row['failure']['type'] == 'org.opentest4j.AssertionFailedError' for row in failed)
    assert sorted(row['failure']['message'] for row in failed) == (
        flip_expected_messages if expected_failures else []
    )
    assert run['failure_details'] == [dict(name=row['name'], **row['failure']) for row in failed]
    print(f"codec authored {index + 1} {role} {arm} | {expected_count} started | "
          f"{run['passed']} passed | {expected_failures} failed | exit {run['native_exit']}")

print('Controlled authored-candidate witness. No saved-agent or complete benchmark-gate claim.')
print('Fresh Python 3.12.15 worker reproduced the same twelve native outcomes and exact Bash block.')


codec authored 1 gold released | 6 started | 6 passed | 0 failed | exit 0
codec authored 2 gold restore_url | 15 started | 15 passed | 0 failed | exit 0
codec authored 3 gold restore_both | 24 started | 24 passed | 0 failed | exit 0
codec authored 4 candidate released | 6 started | 6 passed | 0 failed | exit 0
codec authored 5 candidate restore_url | 15 started | 9 passed | 6 failed | exit 1
codec authored 6 candidate restore_both | 24 started | 18 passed | 6 failed | exit 1
codec authored 7 candidate restore_both | 24 started | 18 passed | 6 failed | exit 1
codec authored 8 candidate restore_url | 15 started | 9 passed | 6 failed | exit 1
codec authored 9 candidate released | 6 started | 6 passed | 0 failed | exit 0
codec authored 10 gold restore_both | 24 started | 24 passed | 0 failed | exit 0
codec authored 11 gold restore_url | 15 started | 15 passed | 0 failed | exit 0
codec authored 12 gold released | 6 started | 6 passed | 0 failed | exit 0
Controlled authored-candidate witness

## 2d. Main experiment. Compiler-qualified evaluator repair

This experiment evaluates the repair on two source-exposed tasks with authored incorrect candidates. Seven layouts and two roles per task give 28 compiled fixtures, including 24 transformed fixtures. Each transformed test classfile map equals its role's original map byte for byte. This study compiles with `--release 8 -g:none` equally in every arm, permitting changes to line counts. It does not retroactively change the earlier debug-preserving controls.

The layouts were frozen before Azure execution. They are original source, canonical preceding annotations, inline annotations, multiline annotation arguments, fully qualified annotation names, an intervening block comment and a twenty-blank-line gap. All 28 fixtures qualified. No compilation or equivalence failure was discarded. Commons Codec had already been inspected, so this is source-exposed transfer rather than blind validation.

The scope is externally fixed at two Jsoup methods and eight Codec methods. The released strategy uses the pinned source predicate. A same-line ablation adds a simple-name annotation check on the declaration line. The repair reads compiled runtime annotations and intersects them with the frozen required identities. It replaces annotation classification within this scope. Complete repository-level change detection remains outside the experiment. Unannotated helper methods remain outside the inventory.

A native JUnit listener records `MethodSource` identities with qualified class names, parameter types and unique invocation IDs. The count contract requires exactly the declared number of completed invocations for each required method. The fixtures declare one invocation per ordinary method and nine per Codec parameterized method. Missing or unknown counts, unexpected methods, container errors, skips or aborts produce `INCOMPLETE`. Fully accounted execution produces `FAIL` if an invocation fails and `PASS` otherwise. General dynamic-provider cardinality is not inferred.

| Strategy | Complete task-layouts | Nonempty authored-candidate passes | Empty selections | Complete developer-control passes |
|---|---|---|---|---|
| Released filter | 2/14 | 2/14 | 10/14 | 2/14 |
| Same-line patch | 5/14 | 1/14 | 8/14 | 5/14 |
| Compiled inventory | 14/14 | 0/14 | 0/14 | 14/14 |

There are 84 native executions, from two tasks, seven layouts, two roles and three strategies. The table counts selection completeness once per task-layout rather than double-counting roles. Empty runs are never counted as false passes. Applied to every strategy's transcript, the count contract marks each incomplete run `INCOMPLETE`, including the nonempty authored-candidate passes. The repair then restores all required execution and accepts all developer controls while rejecting all authored-candidate layouts. The same assertion-failure values recur across all complete candidate runs. These dependent, source-exposed counts do not estimate population accuracy or real-agent failure prevalence.

### Crosswalk to Figure 1 in the Writeup

Figure 1 uses the `original` layout and `candidate` role from the Jsoup native-JUnit study described here. The candidate is authored, and its source and compiled classfiles remain fixed across selectors. The released filter runs `ElementTest.invalidCharactersDiscardedInXml` successfully but omits `EntitiesTest.controlCharactersAreEscaped`. The compiled inventory runs both methods, retaining the first pass and exposing the omitted method's assertion failure.

| Figure 1 arm | Started | Passed | Failed | Native verdict | Execution contract |
|---|---|---|---|---|---|
| Released filter | 1 | 1 | 0 | PASS | INCOMPLETE |
| Compiled inventory | 2 | 1 | 1 | FAIL | FAIL |

Both rows belong to the 84-execution study. The baseline contract is INCOMPLETE because the required invocation is absent. The repaired contract is FAIL because complete execution exposes an assertion failure. The earlier Maven experiment is reported separately in Sections 4–5, where test-failure-ignore makes both Maven process exits zero.

The next cell checks embedded saved-record projections offline by default. Set `LAYOUT_RESULT_ARCHIVE` to either sanitized native result archive in the replay kit to additionally verify the frozen report, classfile-map equality, native command receipts, unique invocation identities, transcripts and contract verdicts. The raw mode accepts only the two recorded hashes. Its recheck of saved classfile maps attests to the worker's emitted maps rather than rehashing live classes.

| Record | SHA256 |
|---|---|
| Original private Jsoup layout result (provenance) | `71b0436e102d55635dd93ed1150b28ebb3a36b48a6b3af4f0563bd31093ee77d` |
| Original private Codec layout result (provenance) | `c1826d5e447b5c1ec82bc6f3029347457fa92ef2321108b5b6462f1ba923125e` |
| Sanitized Jsoup native archive in kit | `390f755c3c8b83d78b850f32392b3d2fa82690053f651a0e3f045173cb653705` |
| Sanitized Codec native archive in kit | `5fad770e6c268ed581cd876128e65f38ba74600ee794c58d3553225e6de2f572` |
| Frozen study plan | `8f44f392b30b47bcaebf57396b61688fcc7aef366a950cafc063adb9cf9d6a0c` |
| Sanitized native replay kit | `718ef0b8da7a19c2d11e2a86e724188b15b0ba2da88337a2d352b18e29338433` |

### Native reproduction in three steps

The locally prepared `EVALUATOR_LAYOUT_REPLAY_KIT_v1.zip` contains source fixtures, the Java listener, the repair/count contract, public artifact download instructions, licenses, a saved projection and sanitized native evidence archives. The native archives retain transcript bytes and identity/hash records while omitting generated Java copies and the outer wrapper command. It excludes cloud controllers, provider/account IDs, credentials, private input archives and unrelated upstream TLS key fixtures. The sanitized kit is attached under Project Files in the associated Writeup. Download that exact ZIP for the native commands below, keeping the private companion separate. The notebook itself does not contain the private cloud inputs.

Use Linux amd64, Python 3.12 and a Java 17 JDK. The recorded native runs used Java 17.0.20.1. Run the following commands from the extracted kit root. Run these commands separately from the default saved-record replay.

1. Confirm the toolchain, set `JAVA_HOME` to your installed JDK, and fetch the pinned public dependencies.

```bash
python3.12 --version
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
"$JAVA_HOME/bin/java" -version
"$JAVA_HOME/bin/javac" -version
python3.12 fetch_dependencies.py
```

The example JDK path is the recorded Debian path. Replace it if your Java 17 JDK is installed elsewhere. The download step requires internet access and checks the public files against their frozen lengths and SHA256 hashes.

2. Run the Jsoup case and then Commons Codec, using fresh output directories.

```bash
python3.12 run_study.py --case jsoup --java-home "$JAVA_HOME" --dependencies dependencies --output results-jsoup
python3.12 run_study.py --case codec --java-home "$JAVA_HOME" --dependencies dependencies --output results-codec
```

3. Open both reports and check their qualification, completion, control and error fields. Inspect the per-variant native records when a field is unexpected.

```bash
python3.12 -m json.tool results-jsoup/REPORT.json
python3.12 -m json.tool results-codec/REPORT.json
```

Each case should record 42 native executions, for 84 across both. Require `completed`, `all_variants_qualified` and `all_complete_controls_match` to be true, with no reported errors. The combined expected matrix appears at the top of this notebook and in the table above. For Figure 1 specifically, the original-layout authored Jsoup candidate starts one passing test under the released selector and two tests, one failing, under the compiled inventory. Those are existing observations within the matrix.

The download step fetches eight pinned public JARs and the original TEBench predicate source, verifying lengths and SHA256 digests. Setup fetches the predicate from its pinned public revision. Each native study records compiler output, inventory, transformed source, classfile maps, native command/status records and class-loading logs. Require `completed`, inspect every error and qualification field, and check `all_complete_controls_match`. A driver exit alone is insufficient. Other JDK builds can produce different artifact hashes, so equivalence is qualified within each run rather than assumed across arbitrary compiler versions.

The public kit reproduces the **new scoped layout/repair experiment**. Earlier Jsoup Maven and developer-only Codec recipes still use their separately retained inputs.


In [5]:
import base64
from collections import Counter
from hashlib import sha256
import json
from pathlib import Path
import tarfile

LAYOUT_RESULT_ARCHIVE = None
layout_evidence = {'cases': [{'case': 'jsoup',
            'instance_id': 'jsoup__001752f1',
            'archive_sha256': '390f755c3c8b83d78b850f32392b3d2fa82690053f651a0e3f045173cb653705',
            'report_sha256': '1b0f466725c8638dacd44e564044b2c8bbecac72c209ddef4ecd80690315f8d7',
            'qualified_compiled_fixtures': 14,
            'required_invocations': {'org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()': 1,
                                     'org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()': 1},
            'candidate_failures': 1,
            'rows': [{'case': 'jsoup',
                      'layout': 'original',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 1,
                      'started': 1,
                      'passed': 1,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'original',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 2,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'jsoup',
                      'layout': 'original',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 2,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'jsoup',
                      'layout': 'original',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 1,
                      'started': 1,
                      'passed': 1,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'original',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 1,
                      'failed': 1,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'jsoup',
                      'layout': 'original',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 1,
                      'failed': 1,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'jsoup',
                      'layout': 'canonical',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 2,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'jsoup',
                      'layout': 'canonical',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 2,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'jsoup',
                      'layout': 'canonical',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 2,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'jsoup',
                      'layout': 'canonical',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 1,
                      'failed': 1,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'jsoup',
                      'layout': 'canonical',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 1,
                      'failed': 1,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'jsoup',
                      'layout': 'canonical',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 1,
                      'failed': 1,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'jsoup',
                      'layout': 'inline',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'inline',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 2,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'jsoup',
                      'layout': 'inline',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 2,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'jsoup',
                      'layout': 'inline',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'inline',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 1,
                      'failed': 1,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'jsoup',
                      'layout': 'inline',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 1,
                      'failed': 1,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'jsoup',
                      'layout': 'multiline',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'multiline',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'multiline',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 2,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'jsoup',
                      'layout': 'multiline',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'multiline',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'multiline',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 1,
                      'failed': 1,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'jsoup',
                      'layout': 'qualified',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'qualified',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'qualified',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 2,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'jsoup',
                      'layout': 'qualified',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'qualified',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'qualified',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 1,
                      'failed': 1,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'jsoup',
                      'layout': 'block_comment',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'block_comment',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'block_comment',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 2,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'jsoup',
                      'layout': 'block_comment',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'block_comment',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'block_comment',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 1,
                      'failed': 1,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'jsoup',
                      'layout': 'gap20',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'gap20',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'gap20',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 2,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'jsoup',
                      'layout': 'gap20',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'gap20',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'jsoup',
                      'layout': 'gap20',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 2,
                      'started': 2,
                      'passed': 1,
                      'failed': 1,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'}],
            'original_private_archive_sha256': '71b0436e102d55635dd93ed1150b28ebb3a36b48a6b3af4f0563bd31093ee77d',
            'public_archive_path': 'evidence/jsoup_result.tar.gz'},
           {'case': 'codec',
            'instance_id': 'commons-codec__d594dd25',
            'archive_sha256': '5fad770e6c268ed581cd876128e65f38ba74600ee794c58d3553225e6de2f572',
            'report_sha256': '57d62fdd9cabe93545ae438dc1d640f66551ac349a372cde3c3de37a8f7fd106',
            'qualified_compiled_fixtures': 14,
            'required_invocations': {'org.apache.commons.codec.binary.Base64Test#testDecodeBase64DiffChars()': 1,
                                     'org.apache.commons.codec.binary.Base64Test#testDecodeBase64StandardDiffChars()': 1,
                                     'org.apache.commons.codec.binary.Base64Test#testDecodeBase64UrlDiffChars()': 1,
                                     'org.apache.commons.codec.binary.Base64Test#testIsStringBase64()': 1,
                                     'org.apache.commons.codec.binary.Base64Test#testIsStringBase64Standard()': 1,
                                     'org.apache.commons.codec.binary.Base64Test#testIsStringBase64Url()': 1,
                                     'org.apache.commons.codec.binary.Base64Test#testDecodeEncodeStandard(java.lang.String)': 9,
                                     'org.apache.commons.codec.binary.Base64Test#testDecodeEncodeUrl(java.lang.String)': 9},
            'candidate_failures': 6,
            'rows': [{'case': 'codec',
                      'layout': 'original',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 6,
                      'started': 6,
                      'passed': 6,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'original',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 6,
                      'started': 6,
                      'passed': 6,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'original',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 24,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'codec',
                      'layout': 'original',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 6,
                      'started': 6,
                      'passed': 6,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'original',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 6,
                      'started': 6,
                      'passed': 6,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'original',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 18,
                      'failed': 6,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'codec',
                      'layout': 'canonical',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 24,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'codec',
                      'layout': 'canonical',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 24,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'codec',
                      'layout': 'canonical',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 24,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'codec',
                      'layout': 'canonical',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 18,
                      'failed': 6,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'codec',
                      'layout': 'canonical',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 18,
                      'failed': 6,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'codec',
                      'layout': 'canonical',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 18,
                      'failed': 6,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'codec',
                      'layout': 'inline',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'inline',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 24,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'codec',
                      'layout': 'inline',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 24,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'codec',
                      'layout': 'inline',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'inline',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 18,
                      'failed': 6,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'codec',
                      'layout': 'inline',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 18,
                      'failed': 6,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'codec',
                      'layout': 'multiline',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'multiline',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'multiline',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 24,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'codec',
                      'layout': 'multiline',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'multiline',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'multiline',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 18,
                      'failed': 6,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'codec',
                      'layout': 'qualified',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'qualified',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'qualified',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 24,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'codec',
                      'layout': 'qualified',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'qualified',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'qualified',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 18,
                      'failed': 6,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'codec',
                      'layout': 'block_comment',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'block_comment',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'block_comment',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 24,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'codec',
                      'layout': 'block_comment',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'block_comment',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'block_comment',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 18,
                      'failed': 6,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'},
                     {'case': 'codec',
                      'layout': 'gap20',
                      'role': 'gold',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'gap20',
                      'role': 'gold',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'gap20',
                      'role': 'gold',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 24,
                      'failed': 0,
                      'native_verdict': 'PASS',
                      'contract_verdict': 'PASS'},
                     {'case': 'codec',
                      'layout': 'gap20',
                      'role': 'candidate',
                      'strategy': 'released',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'gap20',
                      'role': 'candidate',
                      'strategy': 'same_line_patch',
                      'selected_methods': 0,
                      'started': 0,
                      'passed': 0,
                      'failed': 0,
                      'native_verdict': 'EMPTY',
                      'contract_verdict': 'INCOMPLETE'},
                     {'case': 'codec',
                      'layout': 'gap20',
                      'role': 'candidate',
                      'strategy': 'compiled_inventory',
                      'selected_methods': 8,
                      'started': 24,
                      'passed': 18,
                      'failed': 6,
                      'native_verdict': 'FAIL',
                      'contract_verdict': 'FAIL'}],
            'original_private_archive_sha256': 'c1826d5e447b5c1ec82bc6f3029347457fa92ef2321108b5b6462f1ba923125e',
            'public_archive_path': 'evidence/codec_result.tar.gz'}]}

strategies = ('released', 'same_line_patch', 'compiled_inventory')
layouts = ('original', 'canonical', 'inline', 'multiline', 'qualified', 'block_comment', 'gap20')
totals = {name: Counter() for name in strategies}
for case in layout_evidence['cases']:
    assert case['qualified_compiled_fixtures'] == 14
    assert len(case['rows']) == 42
    assert [(row['layout'], row['role'], row['strategy']) for row in case['rows']] == [
        (layout, role, strategy) for layout in layouts
        for role in ('gold', 'candidate') for strategy in strategies]
    for row in case['rows']:
        assert row['started'] == row['passed'] + row['failed']
        assert row['native_verdict'] in ('PASS', 'FAIL', 'EMPTY')
        assert row['contract_verdict'] in ('PASS', 'FAIL', 'INCOMPLETE')
        assert row['native_verdict'] == ('EMPTY' if row['started'] == 0 else 'FAIL' if row['failed'] else 'PASS')
        complete = row['started'] == sum(case['required_invocations'].values())
        assert row['contract_verdict'] == ('INCOMPLETE' if not complete else 'FAIL' if row['failed'] else 'PASS')
        count = totals[row['strategy']]
        if row['role'] == 'candidate':
            count['task_layouts'] += 1
            count['complete'] += complete
            count['authored_native_passes'] += row['native_verdict'] == 'PASS'
            count['empty'] += row['native_verdict'] == 'EMPTY'
            count['contract_passes'] += row['contract_verdict'] == 'PASS'
        else:
            count['complete_gold_passes'] += row['contract_verdict'] == 'PASS'
        if row['strategy'] == 'compiled_inventory':
            assert complete
            assert row['contract_verdict'] == ('PASS' if row['role'] == 'gold' else 'FAIL')
            assert row['failed'] == (0 if row['role'] == 'gold' else case['candidate_failures'])

expected = {
    'released': (2, 2, 10, 0, 2),
    'same_line_patch': (5, 1, 8, 0, 5),
    'compiled_inventory': (14, 0, 0, 0, 14),
}
for name in strategies:
    count = totals[name]
    assert count['task_layouts'] == 14
    assert tuple(count[key] for key in ('complete', 'authored_native_passes', 'empty', 'contract_passes', 'complete_gold_passes')) == expected[name]
    print(f"{name} | complete {count['complete']}/14 | authored native passes "
          f"{count['authored_native_passes']}/14 | empty {count['empty']}/14 | "
          f"complete gold passes {count['complete_gold_passes']}/14")

if LAYOUT_RESULT_ARCHIVE is not None:
    path = Path(LAYOUT_RESULT_ARCHIVE)
    assert path.stat().st_size <= 4 * 1024 * 1024
    archive_hash = sha256(path.read_bytes()).hexdigest()
    case = next(case for case in layout_evidence['cases'] if case['archive_sha256'] == archive_hash)
    with tarfile.open(path) as archive:
        wrapper = json.load(archive.extractfile('REPORT.json'))
        assert wrapper['completed'] and wrapper['admitted']
        assert not wrapper['training_performed'] and not wrapper['model_inference_performed']
        raw_bytes = archive.extractfile('study/REPORT.json').read()
        assert sha256(raw_bytes).hexdigest() == case['report_sha256']
        report = json.loads(raw_bytes)
        assert report['completed'] and report['all_variants_qualified'] and report['all_complete_controls_match']
        assert report['native_execution_count'] == 42 and report['errors'] == []
        assert report['required_invocations'] == case['required_invocations']
        baseline_maps = {row['role']: row['classfiles'] for row in report['variants'] if row['layout'] == 'original'}
        projected = []
        for variant in report['variants']:
            assert variant['qualified'] and not variant['errors']
            assert variant['classfiles'] == baseline_maps[variant['role']]
            assert variant['complete_control_matches']
            for arm in variant['arms']:
                prefix = 'study/' + variant['layout'] + '-' + variant['role'] + '/' + arm['strategy']
                stdout = archive.extractfile(prefix + '/native.stdout').read().decode()
                records = []
                for line in stdout.splitlines():
                    fields = line.split('\t')
                    assert len(fields) == 5 and fields[0] == 'RESULT'
                    records.append({'method': fields[1], 'status': fields[2],
                        'uid': base64.b64decode(fields[3], validate=True).decode(),
                        'message': base64.b64decode(fields[4], validate=True).decode()})
                assert records == arm['records']
                assert len({record['uid'] for record in records}) == len(records)
                assert not arm['containers'] and not arm['skipped']
                assert all(record['status'] in ('SUCCESSFUL', 'FAILED') for record in records)
                command = json.load(archive.extractfile(prefix + '/native.json'))
                assert command['returncode'] == arm['native_exit']
                assert not command['timed_out'] and not command['output_limited']
                failed = sum(record['status'] == 'FAILED' for record in records)
                assert failed == arm['failed']
                assert command['returncode'] == (1 if failed else 0)
                actual_counts = Counter(record['method'] for record in records)
                assert dict(actual_counts) == arm['contract']['observed_invocations']
                assert arm['contract']['expected_invocations'] == case['required_invocations']
                complete = dict(actual_counts) == case['required_invocations']
                assert arm['contract']['verdict'] == ('INCOMPLETE' if not complete else 'FAIL' if failed else 'PASS')
                projected.append({'case': case['case'], 'layout': variant['layout'], 'role': variant['role'],
                    'strategy': arm['strategy'], 'selected_methods': len(arm['selected']),
                    'started': arm['started'], 'passed': arm['passed'], 'failed': arm['failed'],
                    'native_verdict': arm['native_verdict'], 'contract_verdict': arm['contract']['verdict']})
        assert projected == case['rows']

print('Two source-exposed tasks, 28 compiled fixtures, 24 transformed fixtures, 84 native executions.')
print('Layouts are dependent. Incorrect candidates are authored. No prevalence or leaderboard claim.')


released | complete 2/14 | authored native passes 2/14 | empty 10/14 | complete gold passes 2/14
same_line_patch | complete 5/14 | authored native passes 1/14 | empty 8/14 | complete gold passes 5/14
compiled_inventory | complete 14/14 | authored native passes 0/14 | empty 0/14 | complete gold passes 14/14
Two source-exposed tasks, 28 compiled fixtures, 24 transformed fixtures, 84 native executions.
Layouts are dependent. Incorrect candidates are authored. No prevalence or leaderboard claim.


## 3. Trace the skipped test and the historical direct control

The raw developer diff includes the required method. The separate update filter scans backward from the preceding line and skips the inline annotation. The released union consequently selects only `ElementTest.invalidCharactersDiscardedInXml`.

| Maven observation | Actual started test | Started | Failed | Recorded test decision |
|---|---|---:|---:|---|
| Repaired reference | `ElementTest.invalidCharactersDiscardedInXml` | 1 | 0 | Pass |
| Authored partial candidate | `ElementTest.invalidCharactersDiscardedInXml` | 1 | 0 | Pass |
| Direct control on the partial candidate | `EntitiesTest.controlCharactersAreEscaped` | 1 | 1 | Fail |

The partial and direct observations share the old-test class hash above and the same production classfile map. The Maven control adds JaCoCo to both POMs and supplies saved source and diff inputs to the released selector. The unchanged-POM attempt started zero tests. The historical direct control selects the omitted test alone. The paired run below keeps the released selection in both arms. Composite score effects remain unmeasured.


## 4. Reproduce the paired execution result from saved records

Both Azure CPU arms staged the same authored partial candidate, adapted POM and compiled classes at the same scratch path. They used the same read-only offline cache. The logged Maven command vectors differ in exactly one argument.

| Arm | `-Dtest` value | Surefire starts | Passed | Failed | Gate |
|---|---|---:|---:|---:|---|
| Released union | `org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml` | 1 | 1 | 0 | Pass |
| Expanded union | `org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml,org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped` | 2 | 1 | 1 | Fail |

`ElementTest.invalidCharactersDiscardedInXml` passed in both arms. The added `EntitiesTest.controlCharactersAreEscaped` failed with the same assertion type and message as the historical direct control. Both Maven invocations used `-Dmaven.test.failure.ignore=true` and returned zero. Surefire emits per-class and aggregate lines that the released gate counts twice. The table uses the fresh XML testcase rows. The gate decisions come from the released evaluator.

A fresh Azure CPU worker ran the notebook command block byte for byte. Its gate report, Maven command records and Surefire method outcomes matched the original pair, including the added assertion’s type and message.

The cell below checks the command difference, the recorded fixture and cache identities, and the Surefire XML counts and decisions. It defaults to an embedded projection of the archived records that retains the XML suite counts, test identities and failure attributes. If you already have the private saved worker records, set `RESULT_ARCHIVE` to either `result.tar.gz` to check its SHA256, archived XML and commands. For the fresh reproduction, the cell also checks the saved notebook script and Python receipt. The saved replay needs no Azure access.

In [6]:
from hashlib import sha256
from pathlib import Path
import json
import tarfile
import xml.etree.ElementTree as ET

RESULT_ARCHIVE = None
ORIGINAL_RESULT_SHA256 = "c3b06af04358a7b3fb2c497767d7dc78e7476cfa5a1232a05b321fe0d3b9ec9d"
FRESH_RESULT_SHA256 = "9188e75ca380dd2194f61a46e85afed253b6e0f6276645b65f6f2d1d08d0d96d"
NOTEBOOK_SCRIPT_SHA256 = "904b43475a67912612e9a53e04a51b99c3bf744621641174536c098fd2acfa5e"
ARMS = ("released_union", "expanded_union")
BASE = "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml"
ADDED = "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped"
saved = json.loads(r'''{
  "report": {
    "case": "jsoup__001752f1",
    "completed": true,
    "admitted": true,
    "arms": {
      "released_union": {
        "selectors": [
          "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml"
        ],
        "returncode": 0,
        "compiler_reused_classes": true,
        "gate": {
          "success": true,
          "summary": {
            "total": 2,
            "passed": 2,
            "failed": 0,
            "errors": 0,
            "skipped": 0
          },
          "failed_tests": [],
          "error": null
        },
        "tests": {
          "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml": null
        },
        "fixture_unchanged": true,
        "cache_unchanged": true
      },
      "expanded_union": {
        "selectors": [
          "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml",
          "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped"
        ],
        "returncode": 0,
        "compiler_reused_classes": true,
        "gate": {
          "success": false,
          "summary": {
            "total": 4,
            "passed": 2,
            "failed": 2,
            "errors": 0,
            "skipped": 0
          },
          "failed_tests": [
            {
              "class": "org.jsoup.nodes.EntitiesTest",
              "method": "controlCharactersAreEscaped",
              "full_name": "org.jsoup.nodes.EntitiesTest.controlCharactersAreEscaped",
              "failure_type": "org.opentest4j.AssertionFailedError",
              "message": "expected: <<a foo=\"&#x1b;esc&#x7;bell\">Text &#x1b; &#x7;</a>> but was: <<a foo=\"escbell\">Text  </a>>"
            }
          ],
          "error": null
        },
        "tests": {
          "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml": null,
          "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped": {
            "message": "expected: <<a foo=\"&#x1b;esc&#x7;bell\">Text &#x1b; &#x7;</a>> but was: <<a foo=\"escbell\">Text  </a>>",
            "type": "org.opentest4j.AssertionFailedError"
          }
        },
        "fixture_unchanged": true,
        "cache_unchanged": true
      }
    },
    "fixture_manifest_sha256": "0a8292b02c83db533c99fa7e188f759f6e600dfd2b220094244385940d845be1",
    "gate_sha256": "81d04fdffa571c83f885d743519660dcff09dd9dd98dc99b71ac0524dd28aca2",
    "historical_report_sha256": "0c034915e9866e0744a493f25c6b56a3bdee52f070b7dca7a56f3b6c944560cd",
    "cache_manifest_sha256": "f0c79beee59e3e049a6f135a56be6b6df37ef3d6949fcd0fc1d5d7164d94c96d"
  },
  "commands": {
    "released_union": {
      "argv": [
        "/tmp/tebench-pair/apache-maven-3.8.7/bin/mvn",
        "test",
        "jacoco:report",
        "-B",
        "-Drat.skip=true",
        "-Denforcer.skip=true",
        "-Dcheckstyle.skip=true",
        "-Dmaven.test.failure.ignore=true",
        "-Dfelix.skip=true",
        "-Dmaven.javadoc.skip=true",
        "-Djacoco.version=0.8.11",
        "-DargLine=-javaagent:${settings.localRepository}/org/jacoco/org.jacoco.agent/0.8.11/org.jacoco.agent-0.8.11-runtime.jar=destfile=${project.build.directory}/jacoco.exec",
        "-Dtest=org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml",
        "-DfailIfNoTests=false",
        "-o",
        "-s",
        "/tmp/tebench-pair/results/pair/empty-settings.xml",
        "-gs",
        "/tmp/tebench-pair/results/pair/empty-settings.xml",
        "-Dmaven.repo.local=/tmp/tebench-pair/repository",
        "-DlastModGranularityMs=2000"
      ],
      "returncode": 0,
      "timed_out": false,
      "output_limited": false,
      "seconds_limit": 50
    },
    "expanded_union": {
      "argv": [
        "/tmp/tebench-pair/apache-maven-3.8.7/bin/mvn",
        "test",
        "jacoco:report",
        "-B",
        "-Drat.skip=true",
        "-Denforcer.skip=true",
        "-Dcheckstyle.skip=true",
        "-Dmaven.test.failure.ignore=true",
        "-Dfelix.skip=true",
        "-Dmaven.javadoc.skip=true",
        "-Djacoco.version=0.8.11",
        "-DargLine=-javaagent:${settings.localRepository}/org/jacoco/org.jacoco.agent/0.8.11/org.jacoco.agent-0.8.11-runtime.jar=destfile=${project.build.directory}/jacoco.exec",
        "-Dtest=org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml,org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped",
        "-DfailIfNoTests=false",
        "-o",
        "-s",
        "/tmp/tebench-pair/results/pair/empty-settings.xml",
        "-gs",
        "/tmp/tebench-pair/results/pair/empty-settings.xml",
        "-Dmaven.repo.local=/tmp/tebench-pair/repository",
        "-DlastModGranularityMs=2000"
      ],
      "returncode": 0,
      "timed_out": false,
      "output_limited": false,
      "seconds_limit": 50
    }
  },
  "xml": {
    "released_union": [
      "<testsuite name=\"org.jsoup.nodes.ElementTest\" tests=\"1\" failures=\"0\" errors=\"0\" skipped=\"0\"><testcase classname=\"org.jsoup.nodes.ElementTest\" name=\"invalidCharactersDiscardedInXml\" /></testsuite>"
    ],
    "expanded_union": [
      "<testsuite name=\"org.jsoup.nodes.ElementTest\" tests=\"1\" failures=\"0\" errors=\"0\" skipped=\"0\"><testcase classname=\"org.jsoup.nodes.ElementTest\" name=\"invalidCharactersDiscardedInXml\" /></testsuite>",
      "<testsuite name=\"org.jsoup.nodes.EntitiesTest\" tests=\"1\" failures=\"1\" errors=\"0\" skipped=\"0\"><testcase classname=\"org.jsoup.nodes.EntitiesTest\" name=\"controlCharactersAreEscaped\"><failure type=\"org.opentest4j.AssertionFailedError\" message=\"expected: &lt;&lt;a foo=&quot;&amp;#x1b;esc&amp;#x7;bell&quot;&gt;Text &amp;#x1b; &amp;#x7;&lt;/a&gt;&gt; but was: &lt;&lt;a foo=&quot;escbell&quot;&gt;Text  &lt;/a&gt;&gt;\" /></testcase></testsuite>"
    ]
  }
}''')
evidence = saved

if RESULT_ARCHIVE is not None:
    path = Path(RESULT_ARCHIVE)
    archive_digest = sha256(path.read_bytes()).hexdigest()
    assert archive_digest in (ORIGINAL_RESULT_SHA256, FRESH_RESULT_SHA256)
    with tarfile.open(path, "r:gz") as archive:
        def read_member(name):
            with archive.extractfile(name) as stream:
                return stream.read()

        if archive_digest == FRESH_RESULT_SHA256:
            status = json.loads(read_member("notebook_status.json"))
            assert status["shell_exit"] == 0
            assert status["image_python"] == "/usr/local/bin/python3"
            assert status["image_python_version"] == "3.12.15"
            assert sha256(read_member("notebook_steps.sh")).hexdigest() == (
                status["script_sha256"]
            ) == NOTEBOOK_SCRIPT_SHA256

        evidence = {
            "report": json.loads(read_member("pair/REPORT.json")),
            "commands": {
                arm: json.loads(read_member(f"pair/{arm}/test.json"))
                for arm in ARMS
            },
            "xml": {
                arm: [
                    read_member(name)
                    for name in archive.getnames()
                    if name.startswith(f"pair/{arm}/TEST-") and name.endswith(".xml")
                ]
                for arm in ARMS
            },
        }
    assert evidence["report"] == saved["report"]
    assert evidence["commands"] == saved["commands"]

report = evidence["report"]
assert report["case"] == "jsoup__001752f1"
assert report["completed"] and report["admitted"]
for field in ("fixture_manifest_sha256", "gate_sha256", "cache_manifest_sha256"):
    assert report[field] == saved["report"][field]

released, expanded = [evidence["commands"][arm]["argv"] for arm in ARMS]
assert len(released) == len(expanded)
assert [(before, after) for before, after in zip(released, expanded) if before != after] == [
    (f"-Dtest={BASE}", f"-Dtest={BASE},{ADDED}")
]
assert "-o" in released and "-Dmaven.test.failure.ignore=true" in released

expected_selectors = {ARMS[0]: [BASE], ARMS[1]: [BASE, ADDED]}
for arm in ARMS:
    observed = report["arms"][arm]
    command = evidence["commands"][arm]
    assert observed["selectors"] == expected_selectors[arm]
    assert observed["fixture_unchanged"] and observed["cache_unchanged"]
    assert observed["compiler_reused_classes"]
    assert command["returncode"] == observed["returncode"] == 0
    assert not command["timed_out"] and not command["output_limited"]

    tests = {}
    for document in evidence["xml"][arm]:
        suite = ET.fromstring(document)
        rows = suite.findall("testcase")
        assert int(suite.attrib["tests"]) == len(rows)
        assert int(suite.attrib["failures"]) == sum(row.find("failure") is not None for row in rows)
        assert int(suite.attrib["errors"]) == int(suite.attrib["skipped"]) == 0
        for row in rows:
            identity = f'{row.attrib["classname"]}#{row.attrib["name"]}'
            assert identity not in tests
            assert row.find("error") is None and row.find("skipped") is None
            failure = row.find("failure")
            tests[identity] = dict(failure.attrib) if failure is not None else None

    assert set(tests) == set(expected_selectors[arm])
    assert tests == observed["tests"]
    assert tests[BASE] is None
    failures = sum(failure is not None for failure in tests.values())
    assert failures == (arm == "expanded_union")
    assert observed["gate"]["success"] == (failures == 0)
    assert observed["gate"]["error"] is None
    if arm == "expanded_union":
        diagnostic = observed["gate"]["failed_tests"]
        assert len(diagnostic) == 1
        assert diagnostic[0]["full_name"] == ADDED.replace("#", ".")
        assert tests[ADDED] == {
            "type": diagnostic[0]["failure_type"],
            "message": diagnostic[0]["message"],
        }
    decision = "pass" if observed["gate"]["success"] else "fail"
    print(f"{arm} | {len(tests)} started | {len(tests) - failures} passed | {failures} failed | gate {decision}")

released_union | 1 started | 1 passed | 0 failed | gate pass
expanded_union | 2 started | 1 passed | 1 failed | gate fail


## 5. Run the full pair with the private Azure input

This section documents the full worker rerun that we performed. It requires the separately retained input `AZURE_PAIR_INPUT_PRIVATE_v6.tar.gz`, with SHA256 `126e6ab4ddb9f94cdffad2282bfb1460d4ec465e6afb39c3a7a35ab5c46d8058`. The packet contains the exact paired runner, 3,019-file Maven cache and pinned toolchain. Use a disposable Azure Linux amd64 worker with one vCPU and one GiB RAM from `mcr.microsoft.com/devcontainers/python@sha256:86d5f7e6b9084cbedd80abeefb3e8b14eb319e28d2c7e29015324400ac754748`. Stage the archive at the path below before starting the check. Give the worker a fresh scratch directory and a bounded lifecycle.

**Python 3.12 is required for this reproduction.** Use the pinned image’s default `/usr/local/bin/python3`, verified as Python 3.12.15. Before running the block, check `command -v python3` and `python3 --version`. If PATH selects the system interpreter, put `/usr/local/bin` first with `export PATH=/usr/local/bin:$PATH`. An initial staging attempt selected the system Python and stopped at archive extraction before either test ran.

Run inside that disposable worker as root. The guest extracts the pinned Debian Java 17.0.20.1 packages inside the worker, verifies the Maven 3.8.7 archive, drops privileges before Maven and blocks Internet sockets for the runner. Calling `run` directly writes the receipt and logs to the worker filesystem.

```bash
set -eu
input=/inputs/AZURE_PAIR_INPUT_PRIVATE_v6.tar.gz
printf '%s  %s
' '126e6ab4ddb9f94cdffad2282bfb1460d4ec465e6afb39c3a7a35ab5c46d8058' "$input" | sha256sum -c -
mkdir -m 755 /tmp/tebench-pair
tar -xzf "$input" -C /tmp/tebench-pair
/usr/bin/timeout -k 8s 275s python3 -I -S -B -c 'import runpy; raise SystemExit(runpy.run_path("/tmp/tebench-pair/azure_guest.py")["run"]())'
```

Retrieve `/tmp/tebench-pair/results/pair/REPORT.json`, the two arms' `test.json` files and their `TEST-*.xml` reports before deleting the worker. The runner restages the same fixture for each arm, checks the cache manifest and fixture bytes before and after execution, and permits only the generated bundle manifest timestamp to differ. Each Maven arm has a 50-second limit. Reproduction succeeds when `admitted` is true and the gate and XML records match the one-pass and one-pass/one-fail rows above.

The confirmed result covers one authored incomplete patch in the JaCoCo 0.8.11 adapted POM. The unchanged-POM execution attempt started zero tests. Composite update scores and population effects remain unmeasured.

| Paired artifact | SHA256 |
|---|---|
| Original worker result archive | `c3b06af04358a7b3fb2c497767d7dc78e7476cfa5a1232a05b321fe0d3b9ec9d` |
| Fresh reproduction archive | `9188e75ca380dd2194f61a46e85afed253b6e0f6276645b65f6f2d1d08d0d96d` |
| Executed notebook Bash block | `904b43475a67912612e9a53e04a51b99c3bf744621641174536c098fd2acfa5e` |
| Exact worker input archive | `126e6ab4ddb9f94cdffad2282bfb1460d4ec465e6afb39c3a7a35ab5c46d8058` |
| Offline cache manifest | `f0c79beee59e3e049a6f135a56be6b6df37ef3d6949fcd0fc1d5d7164d94c96d` |
| Frozen fixture manifest | `0a8292b02c83db533c99fa7e188f759f6e600dfd2b220094244385940d845be1` |
| Released gate source | `81d04fdffa571c83f885d743519660dcff09dd9dd98dc99b71ac0524dd28aca2` |
| Adapted POM | `fc12ef59346c43cebeedb53d77a499a8fe8988220384461ff45d627262ae7ffc` |

## 6. Scope and source identities

The score reversal is measured on the supplied-file Jsoup task. The execution consequence uses an authored partial patch. In Commons Codec, the direct JUnit check restores 18 omitted passing invocations per expanded run. The developer-only arms pass, while identification retains the two added methods. Section 2c separately verifies an authored incorrect Codec test candidate. Restoring its omitted URL test changes native JUnit exit zero to one with six assertion failures. The observation covers native JUnit execution of the authored candidate. Both Codec native Bash blocks reproduce on fresh CPU workers. Section 2c reproduces the same authored candidate. The Commons Collections union replay yields unchanged coverage and uses whole-class coverage as a proxy in its empty-union arm. Section 1b reconstructs the changes depicted in a published illustration. Unavailable raw patches and potentially unshown edits limit its projected grades. Full per-task agent outputs remain unavailable for a leaderboard regrade. Section 2d adds a separate scoped repair study on the same two source-exposed tasks. Its 24 transformed fixtures and 84 native executions are dependent controls. The sanitized kit can rebuild that new study from public source/dependency artifacts without our private cloud inputs.

The following SHA256 digests identify the archived source records for the notebook’s projections.

| Record | SHA256 |
|---|---|
| Eight-output identification | `672803bfe6a6d70afe079c49eb22eb1913c317ad0ed464c5d71137dc7e0defd8` |
| Frozen prompt and domain specification | `6237fbb2d91208e627567a1f4ecf136eb9fa28cdc3a878bb54f65c2cbfe2c48a` |
| Reference extraction and layout record | `fecf687fb5127685f80b23d593f312de1aa22cd233a8666b24da33b4c37f2c5e` |
| Jsoup classfile and assertion controls | `c85c25e2968a78c364df560ae11513b9293b0afbba968f38c9784abc0435bf70` |
| JaCoCo-adapted gate | `0c034915e9866e0744a493f25c6b56a3bdee52f070b7dca7a56f3b6c944560cd` |
| Unchanged-POM attempt | `c17ee81b8cb725df397487df2591c22f75133a2fb6230f13f17e93c5fcfd8b9d` |
| Commons Collections union replay | `389c41822421026b31336221c17748cd8355dd39ef30b9b8c0b99efdc20ade9b` |

The inspected evaluator is TEBench revision `5f607183bd9f29b6c740eae76b0489dd0c7556d5`. [TEBench paper](https://arxiv.org/html/2605.06125) · [Pinned source](https://github.com/iSEngLab/TEBench/tree/5f607183bd9f29b6c740eae76b0489dd0c7556d5) · [TEBench repository](https://github.com/iSEngLab/TEBench) · [Jsoup developer revision](https://github.com/jhy/jsoup/commit/001752f1a0b73ae17668dc9fd4c5942424a6c1e2)

The archived identification evaluator source has SHA256 `e0d03cb23dc915be300b6f77b22fe4dba88d6d56cea9964e1033664ce3911497`. The archived ground-truth extractor has SHA256 `70529a9f865f300a4c5903a8f0abd1169d13b1c6121f615e66138fd659df38cb`.


## 7. Secondary Gemma 4 E4B pilot and exploratory coverage example

Section 2d evaluates the selector repair. This secondary study records the inactive feedback contrast and a post-hoc assertion-adequacy example on one existing task. Agent-learning effects remain unmeasured.

This constrained repair study uses Gemma E4B. Section 1 records the earlier 31B identification run. Official Gemma 4 E4B Q4_0, llama.cpp b11552, two CPU threads, 8192 context, 2048 output-token limit, no thinking, temperature 0.7, top-k 40 and top-p 0.95 are pinned in the kit. Native consumed prompt IDs are retained. Training and GPU inference remain paused.

Five **source-exposed method units from four historical commits** use seeds 1729 and 2718. The two Codec units share one commit. Only Jsoup and Codec exercise known omissions. Lang and JFreeChart are unaffected-selector controls. Signatures, annotations, providers, imports and non-target methods are fixed. Existing methods start from parent bodies and new Codec targets use authored TODO scaffolds. Model-written Java is retained unchanged.

The full ten initial slots contain nine outputs and one timeout. Eight permitted shared repair requests return six outputs and two timeouts. Each final policy has eight available outputs, three compiled native grades and two finite-oracle-qualified outputs. Two missing slots, four admission rejections and one compilation error make up the other seven. The conservative JSON/body policy rejects Java Unicode escapes and scaffold escapes, leaving semantic Java correctness unresolved for those outputs. Both graders provide identical initial feedback in every slot, so their repair calls are shared or both retain the initial response. There is **no activated feedback contrast**, no compute-matched eligible subset and no demonstrated policy advantage.

The oracle invokes only the generated target. It requires complete developer-production PASS and complete declared-fault FAIL. Fixed Gold controls cannot confer adequacy on an empty target. Jsoup/Lang use compatible historical production rollbacks, JFreeChart restores its parent equals body with new APIs retained, and Codec uses explicitly authored wrong-alphabet substitutions. One-fault adequacy is not exhaustive correctness.

The first provisional grader omitted Java `-ea`. All five controls and the exact initial outputs were regraded uniformly with assertions enabled. A deliberately false language assertion must fail, and the runner refuses a JVM without `-ea`. An Azure audit verifies that corrected feedback **exactly equals what the model actually received**, and that six retained raw responses remain unchanged across resumption. Provisional grades stay visible as provisional evidence. The researcher inspected initial oracle outcomes while callbacks ran. Those outcomes never entered feedback, but motivated the following post-hoc example.

### Post-hoc single-class diagnostic

The actual saved Jsoup 2718 body passes both graders but fails to detect the historical Entities behavior. It checks that XML output contains `Text` and leaves removal of invalid XML characters unasserted. Its `expectedXml` variable is never asserted. These Java assertions are enabled.

Before measuring coverage, the exploratory follow-up froze exactly **`org.jsoup.nodes.Entities`**, the changed production class. Developer reference and saved Gemma body each run twice in fresh JVMs on the same debug-enabled production classfiles. All four runs have the **same class ID, full 162-bit probe vector with 73 hits, class/method counters and 40 of 115 covered branches**. Native class origins are bound to the fresh directories. The developer reference passes intended production and fails the historical fault. Gemma passes both. Raw `.exec`, XML, vectors and native records are retained.

This post-hoc counterexample was selected after observing a weak test. It covers one changed production class on an existing task. Execution-trace equality, full-project coverage and complete TEBench scores remain unmeasured. The initial diagnostic’s helper import compilation failure and its two-import operational correction remain preserved.

### Replay saved evidence

The next cell runs offline with an embedded recorded projection. Set `GEMMA_REPLAY_KIT` to `GEMMA_PAIRED_REPAIR_REPLAY_KIT_v2.zip` to verify the complete curated kit, SHA256 manifest, native archives and model-record linkage. Fresh Java execution uses the commands below. Optional inference regeneration is separate from replaying saved responses and must not replace any study observation.

### Fresh native CPU reproduction

Use a fresh disposable Linux amd64 worker, **Python 3.12**, Java 17 at `/usr/lib/jvm/java-17-openjdk-amd64`, one CPU and one GiB RAM. The qualified workers used Python 3.12.15 and Debian Java 17.0.20.1. Exact classfile-map comparisons require the recorded JDK. No cloud SDK, credentials, model or GPU is needed by the scientific kit. Its downloaders verify pinned public Maven artifacts and TEBench source. The research agent runs these checks on Azure CPU.

```bash
set -euo pipefail
unzip GEMMA_PAIRED_REPAIR_REPLAY_KIT_v2.zip -d gemma_replay
cd gemma_replay
export GEMMA_REPLAY_ROOT="$PWD"
python3.12 kit/fetch_dependencies.py --output "$GEMMA_REPLAY_ROOT/jars"
python3.12 kit/fetch_instrumentation.py --output "$GEMMA_REPLAY_ROOT/instrumentation"
python3.12 kit/audit_provenance.py --input "$GEMMA_REPLAY_ROOT/evidence" --output "$GEMMA_REPLAY_ROOT/provenance-replay.json"
python3.12 kit/analyze_study.py --input "$GEMMA_REPLAY_ROOT/evidence" --output "$GEMMA_REPLAY_ROOT/analysis-replay"
python3.12 kit/coverage_diagnostic.py --units jsoup --dependencies "$GEMMA_REPLAY_ROOT/jars" --responses "$GEMMA_REPLAY_ROOT/evidence/INITIAL_RESPONSES.json" --output "$GEMMA_REPLAY_ROOT/coverage-replay"
```

A successful diagnostic sets `completed`, `oracle_contrast`, `all_four_exact_vectors_equal`, `all_four_class_and_method_counters_equal` and `counterexample_admitted` to true in `coverage-replay/REPORT.json`. It must reproduce all four 73-of-162 probe vectors, 40-of-115 branch counts and the distinct fault outcomes. A negative result must not be replaced by an edited body or narrower matching scope.

For full initial/final native grades, follow the kit’s per-unit loops in `README.txt`, preserving all compilation/rejection/failure records. Omitting `--responses` runs reference/stale/vacuous controls. Those commands compile production with `--release 8 -g:none`. The exploratory coverage command uses `-g` equally for both roles. The public kit retains upstream source licenses, raw model records, exact routing, amended methods and scientific native evidence. It excludes provider controllers, private identities, credentials and signed transfer links.


In [7]:
"""Notebook cell template. Embedded evidence is replaced during packaging."""
from hashlib import sha256
import io
import json
from pathlib import Path
import tarfile
import zipfile

# Set this to GEMMA_PAIRED_REPAIR_REPLAY_KIT_v2.zip to verify its saved evidence.
# None uses the embedded recorded projection. Neither mode executes Java or a model.
GEMMA_REPLAY_KIT = None
EXPECTED_GEMMA_KIT_SHA256 = "dc3756c665abc8974d5f5f158e812c31e0b9a8dbf21a7a1776155b9c790ec5b2"
gemma_summary = json.loads('{\n  "completed": true,\n  "model": {\n    "model": "google/gemma-4-E4B-it-qat-q4_0-gguf",\n    "revision": "4b4a2c1d584be7264f87aac328a1bc739ce81b6c",\n    "file": "gemma-4-E4B_q4_0-it.gguf",\n    "bytes": 5154941280,\n    "sha256": "676c35070db6dbe52f93e9c864ee0fba4eddea94b9c875d9cb10daff453fbaee",\n    "url": "https://huggingface.co/google/gemma-4-E4B-it-qat-q4_0-gguf/resolve/4b4a2c1d584be7264f87aac328a1bc739ce81b6c/gemma-4-E4B_q4_0-it.gguf"\n  },\n  "backend": "Azure CPU only, two threads, E4B prospectively substituted before all new model outputs",\n  "paired_seed_observations": 10,\n  "repair_units": 5,\n  "historical_commit_clusters": 4,\n  "counts": {\n    "initial_status_REJECTED": 7,\n    "initial_model_outputs_available": 9,\n    "initial_responses_missing": 1,\n    "initial_qualified": 1,\n    "initial_nonempty_source_false_pass": 0,\n    "released_qualified": 2,\n    "released_status_REJECTED": 6,\n    "released_outputs_available": 8,\n    "released_observations": 10,\n    "released_accepted_unqualified": 1,\n    "released_accepted_developer_failure": 0,\n    "released_crossgraded_source_false_pass": 0,\n    "repaired_qualified": 2,\n    "repaired_status_REJECTED": 6,\n    "repaired_outputs_available": 8,\n    "repaired_observations": 10,\n    "repaired_accepted_unqualified": 1,\n    "repaired_accepted_developer_failure": 0,\n    "repaired_crossgraded_source_false_pass": 0,\n    "both_qualified": 2,\n    "repaired_only": 0,\n    "released_only": 0,\n    "neither_qualified": 8,\n    "initial_status_COMPILE_ERROR": 1,\n    "released_status_COMPILE_ERROR": 1,\n    "repaired_status_COMPILE_ERROR": 1,\n    "released_status_GRADED": 3,\n    "repaired_status_GRADED": 3,\n    "initial_status_GRADED": 2\n  },\n  "initial_calls": 10,\n  "distinct_repair_calls": 8,\n  "actual_repair_outputs": 6,\n  "activated_initial_grader_feedback_contrasts": 0,\n  "java_language_assertions_enabled": true,\n  "primary_policy_extra_calls": {\n    "released": 8,\n    "repaired": 8\n  },\n  "rows": [\n    {\n      "sample": "jsoup-1729",\n      "unit": "jsoup",\n      "seed": 1729,\n      "commit_cluster": "jsoup__001752f1",\n      "initial_output_status": "available",\n      "initial_grade_status": "REJECTED",\n      "initial_feedback_verdicts": {\n        "released": "REJECTED",\n        "repaired": "REJECTED"\n      },\n      "initial_oracle_qualified": false,\n      "arms": {\n        "released": {\n          "status": "REJECTED",\n          "oracle_qualified": false,\n          "model_output_status": "available",\n          "retained_initial": false,\n          "request": "jsoup-1729-repair-released",\n          "final_own_verdict": "REJECTED",\n          "native": {},\n          "oracle": {}\n        },\n        "repaired": {\n          "status": "REJECTED",\n          "oracle_qualified": false,\n          "model_output_status": "available",\n          "retained_initial": false,\n          "request": "jsoup-1729-repair-released",\n          "final_own_verdict": "REJECTED",\n          "native": {},\n          "oracle": {}\n        }\n      }\n    },\n    {\n      "sample": "codec_standard-1729",\n      "unit": "codec_standard",\n      "seed": 1729,\n      "commit_cluster": "commons-codec__d594dd25",\n      "initial_output_status": "available",\n      "initial_grade_status": "COMPILE_ERROR",\n      "initial_feedback_verdicts": {\n        "released": "COMPILE_ERROR",\n        "repaired": "COMPILE_ERROR"\n      },\n      "initial_oracle_qualified": false,\n      "arms": {\n        "released": {\n          "status": "REJECTED",\n          "oracle_qualified": false,\n          "model_output_status": "available",\n          "retained_initial": false,\n          "request": "codec_standard-1729-repair-released",\n          "final_own_verdict": "REJECTED",\n          "native": {},\n          "oracle": {}\n        },\n        "repaired": {\n          "status": "REJECTED",\n          "oracle_qualified": false,\n          "model_output_status": "available",\n          "retained_initial": false,\n          "request": "codec_standard-1729-repair-released",\n          "final_own_verdict": "REJECTED",\n          "native": {},\n          "oracle": {}\n        }\n      }\n    },\n    {\n      "sample": "codec_url-1729",\n      "unit": "codec_url",\n      "seed": 1729,\n      "commit_cluster": "commons-codec__d594dd25",\n      "initial_output_status": "available",\n      "initial_grade_status": "REJECTED",\n      "initial_feedback_verdicts": {\n        "released": "REJECTED",\n        "repaired": "REJECTED"\n      },\n      "initial_oracle_qualified": false,\n      "arms": {\n        "released": {\n          "status": "COMPILE_ERROR",\n          "oracle_qualified": false,\n          "model_output_status": "available",\n          "retained_initial": false,\n          "request": "codec_url-1729-repair-released",\n          "final_own_verdict": "COMPILE_ERROR",\n          "native": {},\n          "oracle": {}\n        },\n        "repaired": {\n          "status": "COMPILE_ERROR",\n          "oracle_qualified": false,\n          "model_output_status": "available",\n          "retained_initial": false,\n          "request": "codec_url-1729-repair-released",\n          "final_own_verdict": "COMPILE_ERROR",\n          "native": {},\n          "oracle": {}\n        }\n      }\n    },\n    {\n      "sample": "lang-1729",\n      "unit": "lang",\n      "seed": 1729,\n      "commit_cluster": "commons-lang__14d34506",\n      "initial_output_status": "available",\n      "initial_grade_status": "REJECTED",\n      "initial_feedback_verdicts": {\n        "released": "REJECTED",\n        "repaired": "REJECTED"\n      },\n      "initial_oracle_qualified": false,\n      "arms": {\n        "released": {\n          "status": "REJECTED",\n          "oracle_qualified": false,\n          "model_output_status": "timeout_no_response",\n          "retained_initial": false,\n          "request": "lang-1729-repair-released",\n          "final_own_verdict": "REJECTED",\n          "native": {},\n          "oracle": {}\n        },\n        "repaired": {\n          "status": "REJECTED",\n          "oracle_qualified": false,\n          "model_output_status": "timeout_no_response",\n          "retained_initial": false,\n          "request": "lang-1729-repair-released",\n          "final_own_verdict": "REJECTED",\n          "native": {},\n          "oracle": {}\n        }\n      }\n    },\n    {\n      "sample": "jfreechart-1729",\n      "unit": "jfreechart",\n      "seed": 1729,\n      "commit_cluster": "jfreechart__c1a6c135",\n      "initial_output_status": "available",\n      "initial_grade_status": "REJECTED",\n      "initial_feedback_verdicts": {\n        "released": "REJECTED",\n        "repaired": "REJECTED"\n      },\n      "initial_oracle_qualified": false,\n      "arms": {\n        "released": {\n          "status": "GRADED",\n          "oracle_qualified": true,\n          "model_output_status": "available",\n          "retained_initial": false,\n          "request": "jfreechart-1729-repair-released",\n          "final_own_verdict": "PASS",\n          "native": {\n            "selected": [\n              "org.jfree.chart.renderer.category.BoxAndWhiskerRendererTest#testEquals()"\n            ],\n            "started": 1,\n            "failed": 0,\n            "native_exit": 0,\n            "contract": {\n              "verdict": "PASS",\n              "reasons": [],\n              "expected_invocations": {\n                "org.jfree.chart.renderer.category.BoxAndWhiskerRendererTest#testEquals()": 1\n              },\n              "observed_invocations": {\n                "org.jfree.chart.renderer.category.BoxAndWhiskerRendererTest#testEquals()": 1\n              },\n              "passed": 1,\n              "failed": 0\n            }\n          },\n          "oracle": {\n            "developer": {\n              "verdict": "PASS",\n              "started": 1,\n              "failed": 0\n            },\n            "fault": {\n              "verdict": "FAIL",\n              "started": 1,\n              "failed": 1\n            }\n          }\n        },\n        "repaired": {\n          "status": "GRADED",\n          "oracle_qualified": true,\n          "model_output_status": "available",\n          "retained_initial": false,\n          "request": "jfreechart-1729-repair-released",\n          "final_own_verdict": "PASS",\n          "native": {\n            "selected": [\n              "org.jfree.chart.renderer.category.BoxAndWhiskerRendererTest#testEquals()"\n            ],\n            "started": 1,\n            "failed": 0,\n            "native_exit": 0,\n            "contract": {\n              "verdict": "PASS",\n              "reasons": [],\n              "expected_invocations": {\n                "org.jfree.chart.renderer.category.BoxAndWhiskerRendererTest#testEquals()": 1\n              },\n              "observed_invocations": {\n                "org.jfree.chart.renderer.category.BoxAndWhiskerRendererTest#testEquals()": 1\n              },\n              "passed": 1,\n              "failed": 0\n            }\n          },\n          "oracle": {\n            "developer": {\n              "verdict": "PASS",\n              "started": 1,\n              "failed": 0\n            },\n            "fault": {\n              "verdict": "FAIL",\n              "started": 1,\n              "failed": 1\n            }\n          }\n        }\n      }\n    },\n    {\n      "sample": "jfreechart-2718",\n      "unit": "jfreechart",\n      "seed": 2718,\n      "commit_cluster": "jfreechart__c1a6c135",\n      "initial_output_status": "available",\n      "initial_grade_status": "GRADED",\n      "initial_feedback_verdicts": {\n        "released": "PASS",\n        "repaired": "PASS"\n      },\n      "initial_oracle_qualified": true,\n      "arms": {\n        "released": {\n          "status": "GRADED",\n          "oracle_qualified": true,\n          "model_output_status": "available",\n          "retained_initial": true,\n          "request": null,\n          "final_own_verdict": "PASS",\n          "native": {\n            "selected": [\n              "org.jfree.chart.renderer.category.BoxAndWhiskerRendererTest#testEquals()"\n            ],\n            "started": 1,\n            "failed": 0,\n            "native_exit": 0,\n            "contract": {\n              "verdict": "PASS",\n              "reasons": [],\n              "expected_invocations": {\n                "org.jfree.chart.renderer.category.BoxAndWhiskerRendererTest#testEquals()": 1\n              },\n              "observed_invocations": {\n                "org.jfree.chart.renderer.category.BoxAndWhiskerRendererTest#testEquals()": 1\n              },\n              "passed": 1,\n              "failed": 0\n            }\n          },\n          "oracle": {\n            "developer": {\n              "verdict": "PASS",\n              "started": 1,\n              "failed": 0\n            },\n            "fault": {\n              "verdict": "FAIL",\n              "started": 1,\n              "failed": 1\n            }\n          }\n        },\n        "repaired": {\n          "status": "GRADED",\n          "oracle_qualified": true,\n          "model_output_status": "available",\n          "retained_initial": true,\n          "request": null,\n          "final_own_verdict": "PASS",\n          "native": {\n            "selected": [\n              "org.jfree.chart.renderer.category.BoxAndWhiskerRendererTest#testEquals()"\n            ],\n            "started": 1,\n            "failed": 0,\n            "native_exit": 0,\n            "contract": {\n              "verdict": "PASS",\n              "reasons": [],\n              "expected_invocations": {\n                "org.jfree.chart.renderer.category.BoxAndWhiskerRendererTest#testEquals()": 1\n              },\n              "observed_invocations": {\n                "org.jfree.chart.renderer.category.BoxAndWhiskerRendererTest#testEquals()": 1\n              },\n              "passed": 1,\n              "failed": 0\n            }\n          },\n          "oracle": {\n            "developer": {\n              "verdict": "PASS",\n              "started": 1,\n              "failed": 0\n            },\n            "fault": {\n              "verdict": "FAIL",\n              "started": 1,\n              "failed": 1\n            }\n          }\n        }\n      }\n    },\n    {\n      "sample": "lang-2718",\n      "unit": "lang",\n      "seed": 2718,\n      "commit_cluster": "commons-lang__14d34506",\n      "initial_output_status": "timeout_no_response",\n      "initial_grade_status": "REJECTED",\n      "initial_feedback_verdicts": {\n        "released": "REJECTED",\n        "repaired": "REJECTED"\n      },\n      "initial_oracle_qualified": false,\n      "arms": {\n        "released": {\n          "status": "REJECTED",\n          "oracle_qualified": false,\n          "model_output_status": "timeout_no_response",\n          "retained_initial": false,\n          "request": "lang-2718-repair-released",\n          "final_own_verdict": "REJECTED",\n          "native": {},\n          "oracle": {}\n        },\n        "repaired": {\n          "status": "REJECTED",\n          "oracle_qualified": false,\n          "model_output_status": "timeout_no_response",\n          "retained_initial": false,\n          "request": "lang-2718-repair-released",\n          "final_own_verdict": "REJECTED",\n          "native": {},\n          "oracle": {}\n        }\n      }\n    },\n    {\n      "sample": "codec_url-2718",\n      "unit": "codec_url",\n      "seed": 2718,\n      "commit_cluster": "commons-codec__d594dd25",\n      "initial_output_status": "available",\n      "initial_grade_status": "REJECTED",\n      "initial_feedback_verdicts": {\n        "released": "REJECTED",\n        "repaired": "REJECTED"\n      },\n      "initial_oracle_qualified": false,\n      "arms": {\n        "released": {\n          "status": "REJECTED",\n          "oracle_qualified": false,\n          "model_output_status": "available",\n          "retained_initial": false,\n          "request": "codec_url-2718-repair-released",\n          "final_own_verdict": "REJECTED",\n          "native": {},\n          "oracle": {}\n        },\n        "repaired": {\n          "status": "REJECTED",\n          "oracle_qualified": false,\n          "model_output_status": "available",\n          "retained_initial": false,\n          "request": "codec_url-2718-repair-released",\n          "final_own_verdict": "REJECTED",\n          "native": {},\n          "oracle": {}\n        }\n      }\n    },\n    {\n      "sample": "codec_standard-2718",\n      "unit": "codec_standard",\n      "seed": 2718,\n      "commit_cluster": "commons-codec__d594dd25",\n      "initial_output_status": "available",\n      "initial_grade_status": "REJECTED",\n      "initial_feedback_verdicts": {\n        "released": "REJECTED",\n        "repaired": "REJECTED"\n      },\n      "initial_oracle_qualified": false,\n      "arms": {\n        "released": {\n          "status": "REJECTED",\n          "oracle_qualified": false,\n          "model_output_status": "available",\n          "retained_initial": false,\n          "request": "codec_standard-2718-repair-released",\n          "final_own_verdict": "REJECTED",\n          "native": {},\n          "oracle": {}\n        },\n        "repaired": {\n          "status": "REJECTED",\n          "oracle_qualified": false,\n          "model_output_status": "available",\n          "retained_initial": false,\n          "request": "codec_standard-2718-repair-released",\n          "final_own_verdict": "REJECTED",\n          "native": {},\n          "oracle": {}\n        }\n      }\n    },\n    {\n      "sample": "jsoup-2718",\n      "unit": "jsoup",\n      "seed": 2718,\n      "commit_cluster": "jsoup__001752f1",\n      "initial_output_status": "available",\n      "initial_grade_status": "GRADED",\n      "initial_feedback_verdicts": {\n        "released": "PASS",\n        "repaired": "PASS"\n      },\n      "initial_oracle_qualified": false,\n      "arms": {\n        "released": {\n          "status": "GRADED",\n          "oracle_qualified": false,\n          "model_output_status": "available",\n          "retained_initial": true,\n          "request": null,\n          "final_own_verdict": "PASS",\n          "native": {\n            "selected": [\n              "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()"\n            ],\n            "started": 1,\n            "failed": 0,\n            "native_exit": 0,\n            "contract": {\n              "verdict": "INCOMPLETE",\n              "reasons": [\n                "invocation_count:org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()"\n              ],\n              "expected_invocations": {\n                "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1,\n                "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()": 1\n              },\n              "observed_invocations": {\n                "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()": 1\n              },\n              "passed": 1,\n              "failed": 0\n            }\n          },\n          "oracle": {\n            "developer": {\n              "verdict": "PASS",\n              "started": 1,\n              "failed": 0\n            },\n            "fault": {\n              "verdict": "PASS",\n              "started": 1,\n              "failed": 0\n            }\n          }\n        },\n        "repaired": {\n          "status": "GRADED",\n          "oracle_qualified": false,\n          "model_output_status": "available",\n          "retained_initial": true,\n          "request": null,\n          "final_own_verdict": "PASS",\n          "native": {\n            "selected": [\n              "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()",\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()"\n            ],\n            "started": 2,\n            "failed": 0,\n            "native_exit": 0,\n            "contract": {\n              "verdict": "PASS",\n              "reasons": [],\n              "expected_invocations": {\n                "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1,\n                "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()": 1\n              },\n              "observed_invocations": {\n                "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()": 1,\n                "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n              },\n              "passed": 2,\n              "failed": 0\n            }\n          },\n          "oracle": {\n            "developer": {\n              "verdict": "PASS",\n              "started": 1,\n              "failed": 0\n            },\n            "fault": {\n              "verdict": "PASS",\n              "started": 1,\n              "failed": 0\n            }\n          }\n        }\n      }\n    }\n  ],\n  "limits": [\n    "All historical tasks were source-exposed. Public-source memorization is not excluded.",\n    "Two Codec units share a commit. Seeds, layouts, invocations and methods are not independent tasks.",\n    "Qualification is a finite one-fault test-adequacy oracle, not exhaustive semantic correctness.",\n    "Missing model responses are explicit availability failures. They are not fabricated patches or demonstrated semantic code errors.",\n    "Primary policy changes feedback and stopping jointly. Matched ablation is conditional on initial developer-grader discordance.",\n    "The conservative admission policy rejects Java Unicode escapes and method-scaffold escapes. Admission failure does not establish semantically incorrect Java.",\n    "No post-training, swarm improvement, population accuracy, significance, leaderboard rank or competition-winning effect is established."\n  ]\n}\n')
gemma_coverage = json.loads('{\n  "completed": true,\n  "exploratory_posthoc_single_case": true,\n  "model_sample": "jsoup-2718",\n  "exact_scope": "org/jsoup/nodes/Entities",\n  "training": false,\n  "model_calls": 0,\n  "runs": [\n    {\n      "role": "gold",\n      "repetition": 0,\n      "class_id": "9cbd3281e37819ff",\n      "probe_bits": "000000000000000000000000000011111111000011001111110000000001011000000010000000000000000100110000010101000011010010011111101111111111111111111111111111010101111111",\n      "covered_probes": 73,\n      "total_probes": 162,\n      "class_counters": {\n        "INSTRUCTION": {\n          "missed": 347,\n          "covered": 341\n        },\n        "BRANCH": {\n          "missed": 75,\n          "covered": 40\n        },\n        "LINE": {\n          "missed": 77,\n          "covered": 78\n        },\n        "COMPLEXITY": {\n          "missed": 64,\n          "covered": 20\n        },\n        "METHOD": {\n          "missed": 14,\n          "covered": 9\n        },\n        "CLASS": {\n          "missed": 0,\n          "covered": 1\n        }\n      },\n      "method_counters": {\n        "isNamedEntity(Ljava/lang/String;)Z": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "isBaseNamedEntity(Ljava/lang/String;)Z": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "getByName(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 30,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 7,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 3,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "codepointsForName(Ljava/lang/String;[I)I": {\n          "INSTRUCTION": {\n            "missed": 36,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 10,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 3,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "findPrefix(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 19,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 4,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 3,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escape(Ljava/lang/String;Lorg/jsoup/nodes/Document$OutputSettings;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escape(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 6,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escapeString(Ljava/lang/String;Lorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;Ljava/nio/charset/Charset;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 23,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 8,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escape(Ljava/lang/Appendable;Ljava/lang/String;Lorg/jsoup/nodes/Document$OutputSettings;I)V": {\n          "INSTRUCTION": {\n            "missed": 0,\n            "covered": 11\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 2\n          },\n          "COMPLEXITY": {\n            "missed": 0,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "doEscape(Ljava/lang/String;Ljava/lang/Appendable;Lorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;Ljava/nio/charset/Charset;I)V": {\n          "INSTRUCTION": {\n            "missed": 13,\n            "covered": 70\n          },\n          "BRANCH": {\n            "missed": 6,\n            "covered": 10\n          },\n          "LINE": {\n            "missed": 4,\n            "covered": 21\n          },\n          "COMPLEXITY": {\n            "missed": 5,\n            "covered": 4\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "appendEscaped(ILjava/lang/Appendable;ILorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;Lorg/jsoup/nodes/Entities$CoreCharset;Ljava/nio/charset/CharsetEncoder;)V": {\n          "INSTRUCTION": {\n            "missed": 92,\n            "covered": 34\n          },\n          "BRANCH": {\n            "missed": 17,\n            "covered": 9\n          },\n          "LINE": {\n            "missed": 24,\n            "covered": 8\n          },\n          "COMPLEXITY": {\n            "missed": 13,\n            "covered": 4\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "appendNbsp(Ljava/lang/Appendable;Lorg/jsoup/nodes/Entities$EscapeMode;)V": {\n          "INSTRUCTION": {\n            "missed": 13,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 3,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "appendLt(Ljava/lang/Appendable;ILorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;)V": {\n          "INSTRUCTION": {\n            "missed": 20,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 6,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 3,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 4,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "appendApos(Ljava/lang/Appendable;ILorg/jsoup/nodes/Entities$EscapeMode;)V": {\n          "INSTRUCTION": {\n            "missed": 26,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 6,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 5,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 4,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "appendEncoded(Ljava/lang/Appendable;Lorg/jsoup/nodes/Entities$EscapeMode;I)V": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 18\n          },\n          "BRANCH": {\n            "missed": 1,\n            "covered": 1\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 4\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "unescape(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "unescape(Ljava/lang/String;Z)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "canEncode(Lorg/jsoup/nodes/Entities$CoreCharset;CLjava/nio/charset/CharsetEncoder;)Z": {\n          "INSTRUCTION": {\n            "missed": 15,\n            "covered": 11\n          },\n          "BRANCH": {\n            "missed": 7,\n            "covered": 2\n          },\n          "LINE": {\n            "missed": 2,\n            "covered": 2\n          },\n          "COMPLEXITY": {\n            "missed": 5,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "isValidXmlChar(I)Z": {\n          "INSTRUCTION": {\n            "missed": 6,\n            "covered": 25\n          },\n          "BRANCH": {\n            "missed": 10,\n            "covered": 8\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 1\n          },\n          "COMPLEXITY": {\n            "missed": 8,\n            "covered": 2\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "encoderFor(Ljava/nio/charset/Charset;)Ljava/nio/charset/CharsetEncoder;": {\n          "INSTRUCTION": {\n            "missed": 0,\n            "covered": 19\n          },\n          "BRANCH": {\n            "missed": 1,\n            "covered": 3\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 5\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 2\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "load(Lorg/jsoup/nodes/Entities$EscapeMode;Ljava/lang/String;I)V": {\n          "INSTRUCTION": {\n            "missed": 1,\n            "covered": 125\n          },\n          "BRANCH": {\n            "missed": 1,\n            "covered": 7\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 30\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 4\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "lambda$static$0()[C": {\n          "INSTRUCTION": {\n            "missed": 3,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "<clinit>()V": {\n          "INSTRUCTION": {\n            "missed": 0,\n            "covered": 28\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 5\n          },\n          "COMPLEXITY": {\n            "missed": 0,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        }\n      },\n      "native": {\n        "verdict": "PASS",\n        "reasons": [],\n        "expected_invocations": {\n          "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n        },\n        "observed_invocations": {\n          "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n        },\n        "passed": 1,\n        "failed": 0\n      },\n      "class_origins_verified": true\n    },\n    {\n      "role": "gold",\n      "repetition": 1,\n      "class_id": "9cbd3281e37819ff",\n      "probe_bits": "000000000000000000000000000011111111000011001111110000000001011000000010000000000000000100110000010101000011010010011111101111111111111111111111111111010101111111",\n      "covered_probes": 73,\n      "total_probes": 162,\n      "class_counters": {\n        "INSTRUCTION": {\n          "missed": 347,\n          "covered": 341\n        },\n        "BRANCH": {\n          "missed": 75,\n          "covered": 40\n        },\n        "LINE": {\n          "missed": 77,\n          "covered": 78\n        },\n        "COMPLEXITY": {\n          "missed": 64,\n          "covered": 20\n        },\n        "METHOD": {\n          "missed": 14,\n          "covered": 9\n        },\n        "CLASS": {\n          "missed": 0,\n          "covered": 1\n        }\n      },\n      "method_counters": {\n        "isNamedEntity(Ljava/lang/String;)Z": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "isBaseNamedEntity(Ljava/lang/String;)Z": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "getByName(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 30,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 7,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 3,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "codepointsForName(Ljava/lang/String;[I)I": {\n          "INSTRUCTION": {\n            "missed": 36,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 10,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 3,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "findPrefix(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 19,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 4,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 3,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escape(Ljava/lang/String;Lorg/jsoup/nodes/Document$OutputSettings;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escape(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 6,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escapeString(Ljava/lang/String;Lorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;Ljava/nio/charset/Charset;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 23,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 8,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escape(Ljava/lang/Appendable;Ljava/lang/String;Lorg/jsoup/nodes/Document$OutputSettings;I)V": {\n          "INSTRUCTION": {\n            "missed": 0,\n            "covered": 11\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 2\n          },\n          "COMPLEXITY": {\n            "missed": 0,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "doEscape(Ljava/lang/String;Ljava/lang/Appendable;Lorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;Ljava/nio/charset/Charset;I)V": {\n          "INSTRUCTION": {\n            "missed": 13,\n            "covered": 70\n          },\n          "BRANCH": {\n            "missed": 6,\n            "covered": 10\n          },\n          "LINE": {\n            "missed": 4,\n            "covered": 21\n          },\n          "COMPLEXITY": {\n            "missed": 5,\n            "covered": 4\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "appendEscaped(ILjava/lang/Appendable;ILorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;Lorg/jsoup/nodes/Entities$CoreCharset;Ljava/nio/charset/CharsetEncoder;)V": {\n          "INSTRUCTION": {\n            "missed": 92,\n            "covered": 34\n          },\n          "BRANCH": {\n            "missed": 17,\n            "covered": 9\n          },\n          "LINE": {\n            "missed": 24,\n            "covered": 8\n          },\n          "COMPLEXITY": {\n            "missed": 13,\n            "covered": 4\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "appendNbsp(Ljava/lang/Appendable;Lorg/jsoup/nodes/Entities$EscapeMode;)V": {\n          "INSTRUCTION": {\n            "missed": 13,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 3,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "appendLt(Ljava/lang/Appendable;ILorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;)V": {\n          "INSTRUCTION": {\n            "missed": 20,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 6,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 3,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 4,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "appendApos(Ljava/lang/Appendable;ILorg/jsoup/nodes/Entities$EscapeMode;)V": {\n          "INSTRUCTION": {\n            "missed": 26,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 6,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 5,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 4,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "appendEncoded(Ljava/lang/Appendable;Lorg/jsoup/nodes/Entities$EscapeMode;I)V": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 18\n          },\n          "BRANCH": {\n            "missed": 1,\n            "covered": 1\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 4\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "unescape(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "unescape(Ljava/lang/String;Z)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "canEncode(Lorg/jsoup/nodes/Entities$CoreCharset;CLjava/nio/charset/CharsetEncoder;)Z": {\n          "INSTRUCTION": {\n            "missed": 15,\n            "covered": 11\n          },\n          "BRANCH": {\n            "missed": 7,\n            "covered": 2\n          },\n          "LINE": {\n            "missed": 2,\n            "covered": 2\n          },\n          "COMPLEXITY": {\n            "missed": 5,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "isValidXmlChar(I)Z": {\n          "INSTRUCTION": {\n            "missed": 6,\n            "covered": 25\n          },\n          "BRANCH": {\n            "missed": 10,\n            "covered": 8\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 1\n          },\n          "COMPLEXITY": {\n            "missed": 8,\n            "covered": 2\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "encoderFor(Ljava/nio/charset/Charset;)Ljava/nio/charset/CharsetEncoder;": {\n          "INSTRUCTION": {\n            "missed": 0,\n            "covered": 19\n          },\n          "BRANCH": {\n            "missed": 1,\n            "covered": 3\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 5\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 2\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "load(Lorg/jsoup/nodes/Entities$EscapeMode;Ljava/lang/String;I)V": {\n          "INSTRUCTION": {\n            "missed": 1,\n            "covered": 125\n          },\n          "BRANCH": {\n            "missed": 1,\n            "covered": 7\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 30\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 4\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "lambda$static$0()[C": {\n          "INSTRUCTION": {\n            "missed": 3,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "<clinit>()V": {\n          "INSTRUCTION": {\n            "missed": 0,\n            "covered": 28\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 5\n          },\n          "COMPLEXITY": {\n            "missed": 0,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        }\n      },\n      "native": {\n        "verdict": "PASS",\n        "reasons": [],\n        "expected_invocations": {\n          "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n        },\n        "observed_invocations": {\n          "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n        },\n        "passed": 1,\n        "failed": 0\n      },\n      "class_origins_verified": true\n    },\n    {\n      "role": "gemma",\n      "repetition": 0,\n      "class_id": "9cbd3281e37819ff",\n      "probe_bits": "000000000000000000000000000011111111000011001111110000000001011000000010000000000000000100110000010101000011010010011111101111111111111111111111111111010101111111",\n      "covered_probes": 73,\n      "total_probes": 162,\n      "class_counters": {\n        "INSTRUCTION": {\n          "missed": 347,\n          "covered": 341\n        },\n        "BRANCH": {\n          "missed": 75,\n          "covered": 40\n        },\n        "LINE": {\n          "missed": 77,\n          "covered": 78\n        },\n        "COMPLEXITY": {\n          "missed": 64,\n          "covered": 20\n        },\n        "METHOD": {\n          "missed": 14,\n          "covered": 9\n        },\n        "CLASS": {\n          "missed": 0,\n          "covered": 1\n        }\n      },\n      "method_counters": {\n        "isNamedEntity(Ljava/lang/String;)Z": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "isBaseNamedEntity(Ljava/lang/String;)Z": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "getByName(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 30,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 7,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 3,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "codepointsForName(Ljava/lang/String;[I)I": {\n          "INSTRUCTION": {\n            "missed": 36,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 10,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 3,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "findPrefix(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 19,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 4,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 3,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escape(Ljava/lang/String;Lorg/jsoup/nodes/Document$OutputSettings;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escape(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 6,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escapeString(Ljava/lang/String;Lorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;Ljava/nio/charset/Charset;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 23,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 8,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escape(Ljava/lang/Appendable;Ljava/lang/String;Lorg/jsoup/nodes/Document$OutputSettings;I)V": {\n          "INSTRUCTION": {\n            "missed": 0,\n            "covered": 11\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 2\n          },\n          "COMPLEXITY": {\n            "missed": 0,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "doEscape(Ljava/lang/String;Ljava/lang/Appendable;Lorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;Ljava/nio/charset/Charset;I)V": {\n          "INSTRUCTION": {\n            "missed": 13,\n            "covered": 70\n          },\n          "BRANCH": {\n            "missed": 6,\n            "covered": 10\n          },\n          "LINE": {\n            "missed": 4,\n            "covered": 21\n          },\n          "COMPLEXITY": {\n            "missed": 5,\n            "covered": 4\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "appendEscaped(ILjava/lang/Appendable;ILorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;Lorg/jsoup/nodes/Entities$CoreCharset;Ljava/nio/charset/CharsetEncoder;)V": {\n          "INSTRUCTION": {\n            "missed": 92,\n            "covered": 34\n          },\n          "BRANCH": {\n            "missed": 17,\n            "covered": 9\n          },\n          "LINE": {\n            "missed": 24,\n            "covered": 8\n          },\n          "COMPLEXITY": {\n            "missed": 13,\n            "covered": 4\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "appendNbsp(Ljava/lang/Appendable;Lorg/jsoup/nodes/Entities$EscapeMode;)V": {\n          "INSTRUCTION": {\n            "missed": 13,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 3,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "appendLt(Ljava/lang/Appendable;ILorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;)V": {\n          "INSTRUCTION": {\n            "missed": 20,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 6,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 3,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 4,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "appendApos(Ljava/lang/Appendable;ILorg/jsoup/nodes/Entities$EscapeMode;)V": {\n          "INSTRUCTION": {\n            "missed": 26,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 6,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 5,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 4,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "appendEncoded(Ljava/lang/Appendable;Lorg/jsoup/nodes/Entities$EscapeMode;I)V": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 18\n          },\n          "BRANCH": {\n            "missed": 1,\n            "covered": 1\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 4\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "unescape(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "unescape(Ljava/lang/String;Z)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "canEncode(Lorg/jsoup/nodes/Entities$CoreCharset;CLjava/nio/charset/CharsetEncoder;)Z": {\n          "INSTRUCTION": {\n            "missed": 15,\n            "covered": 11\n          },\n          "BRANCH": {\n            "missed": 7,\n            "covered": 2\n          },\n          "LINE": {\n            "missed": 2,\n            "covered": 2\n          },\n          "COMPLEXITY": {\n            "missed": 5,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "isValidXmlChar(I)Z": {\n          "INSTRUCTION": {\n            "missed": 6,\n            "covered": 25\n          },\n          "BRANCH": {\n            "missed": 10,\n            "covered": 8\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 1\n          },\n          "COMPLEXITY": {\n            "missed": 8,\n            "covered": 2\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "encoderFor(Ljava/nio/charset/Charset;)Ljava/nio/charset/CharsetEncoder;": {\n          "INSTRUCTION": {\n            "missed": 0,\n            "covered": 19\n          },\n          "BRANCH": {\n            "missed": 1,\n            "covered": 3\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 5\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 2\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "load(Lorg/jsoup/nodes/Entities$EscapeMode;Ljava/lang/String;I)V": {\n          "INSTRUCTION": {\n            "missed": 1,\n            "covered": 125\n          },\n          "BRANCH": {\n            "missed": 1,\n            "covered": 7\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 30\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 4\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "lambda$static$0()[C": {\n          "INSTRUCTION": {\n            "missed": 3,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "<clinit>()V": {\n          "INSTRUCTION": {\n            "missed": 0,\n            "covered": 28\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 5\n          },\n          "COMPLEXITY": {\n            "missed": 0,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        }\n      },\n      "native": {\n        "verdict": "PASS",\n        "reasons": [],\n        "expected_invocations": {\n          "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n        },\n        "observed_invocations": {\n          "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n        },\n        "passed": 1,\n        "failed": 0\n      },\n      "class_origins_verified": true\n    },\n    {\n      "role": "gemma",\n      "repetition": 1,\n      "class_id": "9cbd3281e37819ff",\n      "probe_bits": "000000000000000000000000000011111111000011001111110000000001011000000010000000000000000100110000010101000011010010011111101111111111111111111111111111010101111111",\n      "covered_probes": 73,\n      "total_probes": 162,\n      "class_counters": {\n        "INSTRUCTION": {\n          "missed": 347,\n          "covered": 341\n        },\n        "BRANCH": {\n          "missed": 75,\n          "covered": 40\n        },\n        "LINE": {\n          "missed": 77,\n          "covered": 78\n        },\n        "COMPLEXITY": {\n          "missed": 64,\n          "covered": 20\n        },\n        "METHOD": {\n          "missed": 14,\n          "covered": 9\n        },\n        "CLASS": {\n          "missed": 0,\n          "covered": 1\n        }\n      },\n      "method_counters": {\n        "isNamedEntity(Ljava/lang/String;)Z": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "isBaseNamedEntity(Ljava/lang/String;)Z": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "getByName(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 30,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 7,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 3,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "codepointsForName(Ljava/lang/String;[I)I": {\n          "INSTRUCTION": {\n            "missed": 36,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 10,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 3,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "findPrefix(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 19,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 4,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 3,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escape(Ljava/lang/String;Lorg/jsoup/nodes/Document$OutputSettings;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escape(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 6,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escapeString(Ljava/lang/String;Lorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;Ljava/nio/charset/Charset;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 23,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 8,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "escape(Ljava/lang/Appendable;Ljava/lang/String;Lorg/jsoup/nodes/Document$OutputSettings;I)V": {\n          "INSTRUCTION": {\n            "missed": 0,\n            "covered": 11\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 2\n          },\n          "COMPLEXITY": {\n            "missed": 0,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "doEscape(Ljava/lang/String;Ljava/lang/Appendable;Lorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;Ljava/nio/charset/Charset;I)V": {\n          "INSTRUCTION": {\n            "missed": 13,\n            "covered": 70\n          },\n          "BRANCH": {\n            "missed": 6,\n            "covered": 10\n          },\n          "LINE": {\n            "missed": 4,\n            "covered": 21\n          },\n          "COMPLEXITY": {\n            "missed": 5,\n            "covered": 4\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "appendEscaped(ILjava/lang/Appendable;ILorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;Lorg/jsoup/nodes/Entities$CoreCharset;Ljava/nio/charset/CharsetEncoder;)V": {\n          "INSTRUCTION": {\n            "missed": 92,\n            "covered": 34\n          },\n          "BRANCH": {\n            "missed": 17,\n            "covered": 9\n          },\n          "LINE": {\n            "missed": 24,\n            "covered": 8\n          },\n          "COMPLEXITY": {\n            "missed": 13,\n            "covered": 4\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "appendNbsp(Ljava/lang/Appendable;Lorg/jsoup/nodes/Entities$EscapeMode;)V": {\n          "INSTRUCTION": {\n            "missed": 13,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 2,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 3,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 2,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "appendLt(Ljava/lang/Appendable;ILorg/jsoup/nodes/Entities$EscapeMode;Lorg/jsoup/nodes/Document$OutputSettings$Syntax;)V": {\n          "INSTRUCTION": {\n            "missed": 20,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 6,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 3,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 4,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "appendApos(Ljava/lang/Appendable;ILorg/jsoup/nodes/Entities$EscapeMode;)V": {\n          "INSTRUCTION": {\n            "missed": 26,\n            "covered": 0\n          },\n          "BRANCH": {\n            "missed": 6,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 5,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 4,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "appendEncoded(Ljava/lang/Appendable;Lorg/jsoup/nodes/Entities$EscapeMode;I)V": {\n          "INSTRUCTION": {\n            "missed": 9,\n            "covered": 18\n          },\n          "BRANCH": {\n            "missed": 1,\n            "covered": 1\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 4\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "unescape(Ljava/lang/String;)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "unescape(Ljava/lang/String;Z)Ljava/lang/String;": {\n          "INSTRUCTION": {\n            "missed": 4,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "canEncode(Lorg/jsoup/nodes/Entities$CoreCharset;CLjava/nio/charset/CharsetEncoder;)Z": {\n          "INSTRUCTION": {\n            "missed": 15,\n            "covered": 11\n          },\n          "BRANCH": {\n            "missed": 7,\n            "covered": 2\n          },\n          "LINE": {\n            "missed": 2,\n            "covered": 2\n          },\n          "COMPLEXITY": {\n            "missed": 5,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "isValidXmlChar(I)Z": {\n          "INSTRUCTION": {\n            "missed": 6,\n            "covered": 25\n          },\n          "BRANCH": {\n            "missed": 10,\n            "covered": 8\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 1\n          },\n          "COMPLEXITY": {\n            "missed": 8,\n            "covered": 2\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "encoderFor(Ljava/nio/charset/Charset;)Ljava/nio/charset/CharsetEncoder;": {\n          "INSTRUCTION": {\n            "missed": 0,\n            "covered": 19\n          },\n          "BRANCH": {\n            "missed": 1,\n            "covered": 3\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 5\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 2\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "load(Lorg/jsoup/nodes/Entities$EscapeMode;Ljava/lang/String;I)V": {\n          "INSTRUCTION": {\n            "missed": 1,\n            "covered": 125\n          },\n          "BRANCH": {\n            "missed": 1,\n            "covered": 7\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 30\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 4\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        },\n        "lambda$static$0()[C": {\n          "INSTRUCTION": {\n            "missed": 3,\n            "covered": 0\n          },\n          "LINE": {\n            "missed": 1,\n            "covered": 0\n          },\n          "COMPLEXITY": {\n            "missed": 1,\n            "covered": 0\n          },\n          "METHOD": {\n            "missed": 1,\n            "covered": 0\n          }\n        },\n        "<clinit>()V": {\n          "INSTRUCTION": {\n            "missed": 0,\n            "covered": 28\n          },\n          "LINE": {\n            "missed": 0,\n            "covered": 5\n          },\n          "COMPLEXITY": {\n            "missed": 0,\n            "covered": 1\n          },\n          "METHOD": {\n            "missed": 0,\n            "covered": 1\n          }\n        }\n      },\n      "native": {\n        "verdict": "PASS",\n        "reasons": [],\n        "expected_invocations": {\n          "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n        },\n        "observed_invocations": {\n          "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n        },\n        "passed": 1,\n        "failed": 0\n      },\n      "class_origins_verified": true\n    }\n  ],\n  "errors": [],\n  "normalized_model_response_sha256": "46b1c224ddb8c117a9dbc71d5d36ee59757283713ac1ade3a563018fb827ac1e",\n  "grades": {\n    "gold": {\n      "unit": "jsoup",\n      "status": "GRADED",\n      "oracle_qualified": true,\n      "compile": {\n        "returncode": 0,\n        "timed_out": false,\n        "output_limited": false\n      },\n      "arms": {\n        "released": {\n          "selected": [\n            "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()"\n          ],\n          "started": 1,\n          "failed": 0,\n          "records": [\n            {\n              "method": "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()",\n              "status": "SUCCESSFUL",\n              "uid": "[engine:junit-jupiter]/[class:org.jsoup.nodes.ElementTest]/[method:invalidCharactersDiscardedInXml()]",\n              "message": ""\n            }\n          ],\n          "containers": [],\n          "skipped": [],\n          "native_exit": 0,\n          "contract": {\n            "verdict": "INCOMPLETE",\n            "reasons": [\n              "invocation_count:org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()"\n            ],\n            "expected_invocations": {\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1,\n              "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()": 1\n            },\n            "observed_invocations": {\n              "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()": 1\n            },\n            "passed": 1,\n            "failed": 0\n          },\n          "verdict": "PASS"\n        },\n        "repaired": {\n          "selected": [\n            "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()",\n            "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()"\n          ],\n          "started": 2,\n          "failed": 0,\n          "records": [\n            {\n              "method": "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()",\n              "status": "SUCCESSFUL",\n              "uid": "[engine:junit-jupiter]/[class:org.jsoup.nodes.ElementTest]/[method:invalidCharactersDiscardedInXml()]",\n              "message": ""\n            },\n            {\n              "method": "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()",\n              "status": "SUCCESSFUL",\n              "uid": "[engine:junit-jupiter]/[class:org.jsoup.nodes.EntitiesTest]/[method:controlCharactersAreEscaped()]",\n              "message": ""\n            }\n          ],\n          "containers": [],\n          "skipped": [],\n          "native_exit": 0,\n          "contract": {\n            "verdict": "PASS",\n            "reasons": [],\n            "expected_invocations": {\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1,\n              "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()": 1\n            },\n            "observed_invocations": {\n              "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()": 1,\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n            },\n            "passed": 2,\n            "failed": 0\n          },\n          "verdict": "PASS"\n        }\n      },\n      "oracle": {\n        "developer": {\n          "selected": [\n            "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()"\n          ],\n          "started": 1,\n          "failed": 0,\n          "records": [\n            {\n              "method": "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()",\n              "status": "SUCCESSFUL",\n              "uid": "[engine:junit-jupiter]/[class:org.jsoup.nodes.EntitiesTest]/[method:controlCharactersAreEscaped()]",\n              "message": ""\n            }\n          ],\n          "containers": [],\n          "skipped": [],\n          "native_exit": 0,\n          "contract": {\n            "verdict": "PASS",\n            "reasons": [],\n            "expected_invocations": {\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n            },\n            "observed_invocations": {\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n            },\n            "passed": 1,\n            "failed": 0\n          }\n        },\n        "fault": {\n          "selected": [\n            "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()"\n          ],\n          "started": 1,\n          "failed": 1,\n          "records": [\n            {\n              "method": "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()",\n              "status": "FAILED",\n              "uid": "[engine:junit-jupiter]/[class:org.jsoup.nodes.EntitiesTest]/[method:controlCharactersAreEscaped()]",\n              "message": "org.opentest4j.AssertionFailedError: expected: <<a foo=\\"escbell\\">Text  </a>> but was: <<a foo=\\"&#x1b;esc&#x7;bell\\">Text &#x1b; &#x7;</a>>"\n            }\n          ],\n          "containers": [],\n          "skipped": [],\n          "native_exit": 1,\n          "contract": {\n            "verdict": "FAIL",\n            "reasons": [],\n            "expected_invocations": {\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n            },\n            "observed_invocations": {\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n            },\n            "passed": 0,\n            "failed": 1\n          }\n        }\n      },\n      "feedback": {\n        "released": {\n          "verdict": "PASS",\n          "selected": [\n            "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()"\n          ],\n          "started": 1,\n          "failed": 0,\n          "failures": []\n        },\n        "repaired": {\n          "verdict": "PASS",\n          "selected": [\n            "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()",\n            "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()"\n          ],\n          "started": 2,\n          "failed": 0,\n          "failures": []\n        }\n      },\n      "classfiles": {\n        "org/jsoup/MultiLocaleExtension$MultiLocaleTest.class": "75e5b71aa7b80314ddc933b66f429d7583033cf18fe73f579f4ed0748c53485a",\n        "org/jsoup/MultiLocaleExtension.class": "c2896cd3ebff36ae2f3ca54e4d494054eded9093a10a3036227c81c933bdd626",\n        "org/jsoup/TextUtil.class": "adb791e4f637d40e1cec5e7111831b682aef717d373547e6ecfa019781419607",\n        "org/jsoup/nodes/ElementTest$1.class": "d3af70ba5d443380ef06ffc840e03b620d9d99daf6ae618fe7d6b9c11dc28ed0",\n        "org/jsoup/nodes/ElementTest$2.class": "dc80c6c7ca772d76461e9a6cef1c2cb0708a6bbf3b88124c976455c36b5bdc05",\n        "org/jsoup/nodes/ElementTest.class": "d40dbd4b9cccd4393075144d63dedc3c6ae29fc124acee614aaf9851978d4ac3",\n        "org/jsoup/nodes/EntitiesTest.class": "18b6613308fa89e7e48be009669dd83c4f6109e9b7844b9bd59aa2955846c141",\n        "org/jsoup/nodes/NodeIteratorTest.class": "263a945fe9ab6ed51eb09f1dc3e10edf935a2193a1d500ad8848d56f47d63d8d",\n        "org/jsoup/select/EvaluatorDebug$SexprVisitor.class": "1db7c9c61ecc09b9f18b47035620da3f992da0f87b7e1f262b4ae30d326899ac",\n        "org/jsoup/select/EvaluatorDebug.class": "7d92b7200e1077c7156799a6d6849f853288b6c12e87cd6708c90c1eec5ddf76",\n        "org/jsoup/select/SelectorTest$1.class": "44d95bd292f8ea722f23728c4f19bd191f159cdfe4746de407708e293a727f80",\n        "org/jsoup/select/SelectorTest$ResetTracker.class": "71952b2c70b6104238dbc0b32e2814b94f1b0ab3a17b4de8126add140551ddbf",\n        "org/jsoup/select/SelectorTest.class": "5446cdbd677f069d68c584c65cb930ce8a6fe4795d29e7b780fdf3d55d867445"\n      }\n    },\n    "gemma": {\n      "unit": "jsoup",\n      "status": "GRADED",\n      "oracle_qualified": false,\n      "compile": {\n        "returncode": 0,\n        "timed_out": false,\n        "output_limited": false\n      },\n      "arms": {\n        "released": {\n          "selected": [\n            "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()"\n          ],\n          "started": 1,\n          "failed": 0,\n          "records": [\n            {\n              "method": "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()",\n              "status": "SUCCESSFUL",\n              "uid": "[engine:junit-jupiter]/[class:org.jsoup.nodes.ElementTest]/[method:invalidCharactersDiscardedInXml()]",\n              "message": ""\n            }\n          ],\n          "containers": [],\n          "skipped": [],\n          "native_exit": 0,\n          "contract": {\n            "verdict": "INCOMPLETE",\n            "reasons": [\n              "invocation_count:org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()"\n            ],\n            "expected_invocations": {\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1,\n              "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()": 1\n            },\n            "observed_invocations": {\n              "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()": 1\n            },\n            "passed": 1,\n            "failed": 0\n          },\n          "verdict": "PASS"\n        },\n        "repaired": {\n          "selected": [\n            "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()",\n            "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()"\n          ],\n          "started": 2,\n          "failed": 0,\n          "records": [\n            {\n              "method": "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()",\n              "status": "SUCCESSFUL",\n              "uid": "[engine:junit-jupiter]/[class:org.jsoup.nodes.ElementTest]/[method:invalidCharactersDiscardedInXml()]",\n              "message": ""\n            },\n            {\n              "method": "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()",\n              "status": "SUCCESSFUL",\n              "uid": "[engine:junit-jupiter]/[class:org.jsoup.nodes.EntitiesTest]/[method:controlCharactersAreEscaped()]",\n              "message": ""\n            }\n          ],\n          "containers": [],\n          "skipped": [],\n          "native_exit": 0,\n          "contract": {\n            "verdict": "PASS",\n            "reasons": [],\n            "expected_invocations": {\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1,\n              "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()": 1\n            },\n            "observed_invocations": {\n              "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()": 1,\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n            },\n            "passed": 2,\n            "failed": 0\n          },\n          "verdict": "PASS"\n        }\n      },\n      "oracle": {\n        "developer": {\n          "selected": [\n            "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()"\n          ],\n          "started": 1,\n          "failed": 0,\n          "records": [\n            {\n              "method": "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()",\n              "status": "SUCCESSFUL",\n              "uid": "[engine:junit-jupiter]/[class:org.jsoup.nodes.EntitiesTest]/[method:controlCharactersAreEscaped()]",\n              "message": ""\n            }\n          ],\n          "containers": [],\n          "skipped": [],\n          "native_exit": 0,\n          "contract": {\n            "verdict": "PASS",\n            "reasons": [],\n            "expected_invocations": {\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n            },\n            "observed_invocations": {\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n            },\n            "passed": 1,\n            "failed": 0\n          }\n        },\n        "fault": {\n          "selected": [\n            "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()"\n          ],\n          "started": 1,\n          "failed": 0,\n          "records": [\n            {\n              "method": "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()",\n              "status": "SUCCESSFUL",\n              "uid": "[engine:junit-jupiter]/[class:org.jsoup.nodes.EntitiesTest]/[method:controlCharactersAreEscaped()]",\n              "message": ""\n            }\n          ],\n          "containers": [],\n          "skipped": [],\n          "native_exit": 0,\n          "contract": {\n            "verdict": "PASS",\n            "reasons": [],\n            "expected_invocations": {\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n            },\n            "observed_invocations": {\n              "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()": 1\n            },\n            "passed": 1,\n            "failed": 0\n          }\n        }\n      },\n      "feedback": {\n        "released": {\n          "verdict": "PASS",\n          "selected": [\n            "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()"\n          ],\n          "started": 1,\n          "failed": 0,\n          "failures": []\n        },\n        "repaired": {\n          "verdict": "PASS",\n          "selected": [\n            "org.jsoup.nodes.ElementTest#invalidCharactersDiscardedInXml()",\n            "org.jsoup.nodes.EntitiesTest#controlCharactersAreEscaped()"\n          ],\n          "started": 2,\n          "failed": 0,\n          "failures": []\n        }\n      },\n      "classfiles": {\n        "org/jsoup/MultiLocaleExtension$MultiLocaleTest.class": "75e5b71aa7b80314ddc933b66f429d7583033cf18fe73f579f4ed0748c53485a",\n        "org/jsoup/MultiLocaleExtension.class": "c2896cd3ebff36ae2f3ca54e4d494054eded9093a10a3036227c81c933bdd626",\n        "org/jsoup/TextUtil.class": "adb791e4f637d40e1cec5e7111831b682aef717d373547e6ecfa019781419607",\n        "org/jsoup/nodes/ElementTest$1.class": "d3af70ba5d443380ef06ffc840e03b620d9d99daf6ae618fe7d6b9c11dc28ed0",\n        "org/jsoup/nodes/ElementTest$2.class": "dc80c6c7ca772d76461e9a6cef1c2cb0708a6bbf3b88124c976455c36b5bdc05",\n        "org/jsoup/nodes/ElementTest.class": "d40dbd4b9cccd4393075144d63dedc3c6ae29fc124acee614aaf9851978d4ac3",\n        "org/jsoup/nodes/EntitiesTest.class": "cc9a4b1d9eb22f6f40b0d8c26ed4e91f3a0a554e101a38473e873b076f3a02f2",\n        "org/jsoup/nodes/NodeIteratorTest.class": "263a945fe9ab6ed51eb09f1dc3e10edf935a2193a1d500ad8848d56f47d63d8d",\n        "org/jsoup/select/EvaluatorDebug$SexprVisitor.class": "1db7c9c61ecc09b9f18b47035620da3f992da0f87b7e1f262b4ae30d326899ac",\n        "org/jsoup/select/EvaluatorDebug.class": "7d92b7200e1077c7156799a6d6849f853288b6c12e87cd6708c90c1eec5ddf76",\n        "org/jsoup/select/SelectorTest$1.class": "44d95bd292f8ea722f23728c4f19bd191f159cdfe4746de407708e293a727f80",\n        "org/jsoup/select/SelectorTest$ResetTracker.class": "71952b2c70b6104238dbc0b32e2814b94f1b0ab3a17b4de8126add140551ddbf",\n        "org/jsoup/select/SelectorTest.class": "5446cdbd677f069d68c584c65cb930ce8a6fe4795d29e7b780fdf3d55d867445"\n      }\n    }\n  },\n  "oracle_contrast": true,\n  "all_four_exact_vectors_equal": true,\n  "all_four_class_and_method_counters_equal": true,\n  "counterexample_admitted": true,\n  "production_classfiles": {\n    "developer": {\n      "org/jsoup/Connection$Base.class": "46903aee725945ec84e23d2d673ecd88cea4d7aae0963649b048c26d5883f98a",\n      "org/jsoup/Connection$KeyVal.class": "91748caceaf292c55c153eab6aa3db11d11f6dd606e5a7bc53462abaa14e5fbb",\n      "org/jsoup/Connection$Method.class": "a4156ab800562ca78fbf53d294df81fc543043ce9b11ddd474cddb530346e87b",\n      "org/jsoup/Connection$Request.class": "cc39b1ad42dd66edea641c22b2b29fb57aa35f2cc5a699e533f4ae96ec130961",\n      "org/jsoup/Connection$Response.class": "2f82eeba8c7392413813e9040d405e601c9ac5a6d561ba509d6678b9b7ba7aa6",\n      "org/jsoup/Connection.class": "25cea3ffc828f4982685d167f0272d1ddf26c075feb68feef6da06f9899c0d6e",\n      "org/jsoup/HttpStatusException.class": "3599c5812d0a5a3c27183ae62e5056c0ca4a9122d27e2ae3b715aff4c108c54a",\n      "org/jsoup/Jsoup.class": "39687db85faef3a051039716ae168b6cc9e48cb6b0254e89f5e674da6f44fbd6",\n      "org/jsoup/Progress.class": "4560bd5aa49e7fabb7a7029d8c6917af1088ad6b90b1741d00c5ca4e3e0e0305",\n      "org/jsoup/SerializationException.class": "bb0f694d579fa793c6803dd75397b400a802de696255a6b24a0181821e920486",\n      "org/jsoup/UnsupportedMimeTypeException.class": "ce38059772f80b371f6cc1c69447c4de8e7d897db5aa7c8cce12c649388888c8",\n      "org/jsoup/examples/HtmlToPlainText$1.class": "ece3d33b24a7c65d0f20f165ddcbb5f4c540edd449e10adebe4c07d8830e7551",\n      "org/jsoup/examples/HtmlToPlainText$FormattingVisitor.class": "67ffe8beee9bfd209c7b5cd9e0895c60340309de2c92b26a94b9f1146cf865b6",\n      "org/jsoup/examples/HtmlToPlainText.class": "25823d787c8579f3916b8b487e2307998b14c31fa48904c2d8c13d2e25cd729a",\n      "org/jsoup/examples/ListLinks.class": "f970fecdbf384ae89d081a2f4b319bd82847ffc02cce9c8a171c665100f87302",\n      "org/jsoup/examples/Wikipedia.class": "b5effe19f3f3190ead7c36e91a0c172b1b2b8e399f614952a3416be7a347aa13",\n      "org/jsoup/helper/AuthenticationHandler$AuthShim.class": "a1212e9288eb545e8bc18a02753374542ec6dabd586cf633571b7bb8261c4a9a",\n      "org/jsoup/helper/AuthenticationHandler$GlobalHandler.class": "66cac442429d23dbabfbbd844dc8e7f8e58430f033335896fef5895054248801",\n      "org/jsoup/helper/AuthenticationHandler.class": "8b07d1a2315ba92ffca0fcb7ec4dd68c1700fd3c536f6b6dbaff4fd0aa297a74",\n      "org/jsoup/helper/ChangeNotifyingArrayList.class": "5ce9f33b512fe57e1140a5397640b03839678264b853c55f03e60bb0eb4375db",\n      "org/jsoup/helper/CookieUtil.class": "d34ec7e142bf4fd69b29753e2dcd5f0bb801de2bf146af02f96161eda9096d10",\n      "org/jsoup/helper/DataUtil$CharsetDoc.class": "a66dca54be3c4983ebadb13c471902a781b217760126685f62678641f272e1e2",\n      "org/jsoup/helper/DataUtil.class": "000d0c0063446315db3ddaa9129964df478f6669700d2436a4a8d023fd9e394f",\n      "org/jsoup/helper/HttpConnection$1.class": "356bffc36479fe4d4f2b4ca827f6d926379da157dc7585d042eed92699b5a96c",\n      "org/jsoup/helper/HttpConnection$Base.class": "9ed2d5a699f12da3e7ab6d3e6c22f94ac51a654dbb0e80640f98232d4976d958",\n      "org/jsoup/helper/HttpConnection$KeyVal.class": "a5380f821f165714f5482692dfa9f153b4557d0fbd120bb05c27e49a9f2aaae2",\n      "org/jsoup/helper/HttpConnection$Request.class": "1b2684a13375afee800c11a8330d8dc1c13e7281fb2e3324d8b906c5d2614742",\n      "org/jsoup/helper/HttpConnection$Response.class": "2d63b81aaf62572d4ef90602d81b54c9e7dd74fedab78ae09d215b2246f2d542",\n      "org/jsoup/helper/HttpConnection.class": "bfe2a82a89187aa928b133f40130dd0dc04814009e2f8279f5bbc08977505350",\n      "org/jsoup/helper/RequestAuthenticator$Context.class": "4fa9001174b49601805a46d7ee36935cf4c39000411366ba556e03483bd642a4",\n      "org/jsoup/helper/RequestAuthenticator.class": "6b1b00079c6d3ee34384f2f9bbf2e179be7822f744238609f936bd780afb8084",\n      "org/jsoup/helper/RequestDispatch.class": "c3a5b7dc05d3a4ae52a3d519f2ced3166484f07d950435fe96729f1a056f2d45",\n      "org/jsoup/helper/RequestExecutor.class": "8c72ea3ec45b3f4fff288a7f7b9a88903fd5aa52f27b8965ff12a02373103c39",\n      "org/jsoup/helper/UrlBuilder.class": "d91ce6517232d615e6e9dafdab3b5f9d0ae43f6cd2ed89f10f35535b0727aeec",\n      "org/jsoup/helper/UrlConnectionExecutor.class": "bebf451a08e3e5061e6221267e7f73ac23797034293b69c327a8ca9a1e39308c",\n      "org/jsoup/helper/Validate.class": "b168d0a7859769238923101803bcefdfb400d48fa451279e77057fa000b8a5e8",\n      "org/jsoup/helper/ValidationException.class": "b3bd2a4aea69e7585f7c156d0bdb2bccea8c06859e7caf1f628e3908d8e5c016",\n      "org/jsoup/helper/W3CDom$W3CBuilder.class": "398c4b47e95681081d798992d1332d3bd4a9f3e79e982ee2353381c7cac8ebb2",\n      "org/jsoup/helper/W3CDom.class": "6ae881fd6760ca9431221b9de7809d094c86694eefa820410c6d3d04f0179cdb",\n      "org/jsoup/helper/package-info.class": "838331b3ecf370c9428c70e47145ecca0faff05d7a64c227dc9808f9c20c0647",\n      "org/jsoup/internal/ControllableInputStream.class": "1ce9ca524115517178032df0f287f8b0f4618d071f4383976a06119ea79176d9",\n      "org/jsoup/internal/Functions.class": "1bdf9e70c16c967f5adfcf7dd922fdbf11f158edd446620c81b4f23adc18f806",\n      "org/jsoup/internal/Normalizer.class": "af4b2d9d02c9821d9b5134a565c6f185c90cb50785cc352f28c19b579f1a6855",\n      "org/jsoup/internal/SharedConstants.class": "d060df166d0b07a818b3bbf8ef17dc3fb96d6c6440cd8462b2aecee931d9c159",\n      "org/jsoup/internal/SimpleBufferedInput.class": "d33f49a7d4b1c6239aee83259e38f6a42e0f035659c941a9326f3b1f947d8c94",\n      "org/jsoup/internal/SoftPool.class": "9ac52cce7b6f1c4108c6d20378c970bbffd2d4a2f85dfb35f11e955f71f87ad0",\n      "org/jsoup/internal/StringUtil$StringJoiner.class": "fdb3ac6163f2c5b0657836cf37ae2f611887c547ec126614263d1e4d3ddaf9b9",\n      "org/jsoup/internal/StringUtil.class": "f568c77b4def55664c169ccb7ef9c8a3b4526e55a18c5d4b97b1f691bea6c483",\n      "org/jsoup/internal/package-info.class": "48f68f412bf20bebbf75069f334ec0a36ba23c887dae36272b61fb2ceeabee54",\n      "org/jsoup/nodes/Attribute.class": "d738b75dd6c7b8b7d317bc5c750313b05ac467ea234ef1ec819814f083fc8710",\n      "org/jsoup/nodes/Attributes$1.class": "29f7e3a9b72dcb17155cf884fb7e5b9ed737b4888d650ff5812a1de1494a3a27",\n      "org/jsoup/nodes/Attributes$Dataset$DatasetIterator.class": "390d9c4962a03940243d85776f0bfa2c9fc6d8e956c4c9b2993adad3960e988b",\n      "org/jsoup/nodes/Attributes$Dataset$EntrySet.class": "c8f1242ef2059c6b02ffc11182ef7bcfca588ae7c1cd8af8114632edbf7286fb",\n      "org/jsoup/nodes/Attributes$Dataset.class": "e07cf9088654df7225d85456fead1dbbc44f0ae072fd104ae910c95ff2d6d2e7",\n      "org/jsoup/nodes/Attributes.class": "72fd2145df28a2b7bc19fe738c1d007c6e3eb5084a1706d9833fb42c615c8a7c",\n      "org/jsoup/nodes/CDataNode.class": "67f348f46b7c4e942dde41e678b19b74524d0516117885ca13dc20c01e759b3a",\n      "org/jsoup/nodes/Comment.class": "e23908900da901915bc9c3caaa032f4df868c35139721b0576ccd4c57898db1c",\n      "org/jsoup/nodes/DataNode.class": "80cc120787bdbcf87eab3b909c13dd558272a8c3dd7c0ec07337678da9194eed",\n      "org/jsoup/nodes/Document$OutputSettings$Syntax.class": "7e804a00fb3d868b0e20e2a87be911b5a805a11f5360a62468a207cd23a28a6f",\n      "org/jsoup/nodes/Document$OutputSettings.class": "20836331a21665abf0e5e09712228a0119777da096eac548a003f4c2d3ec98a9",\n      "org/jsoup/nodes/Document$QuirksMode.class": "0d33f9ea3a1231d3ff3f57a51043693ea8181ddc0c0969ca9f0b9f4aee3bf4ab",\n      "org/jsoup/nodes/Document.class": "fd6cf580574401a375af6c006ac8b7a47b4380faa75ce0e8da222dd187a4ff1a",\n      "org/jsoup/nodes/DocumentType.class": "730360dcef32b366bfd2b91c4d3692e290d9efc58fea2573041451798ec0ef0a",\n      "org/jsoup/nodes/Element$NodeList.class": "b0ea8de6b11cdcf0b18095861cacc52e1b3ef2ce05a8e3ae3b391dfa18c9dd41",\n      "org/jsoup/nodes/Element$TextAccumulator.class": "54c938bcf65d818a2e5af8f85caf32b121e39cb6aba025a66e9d8df9d1eb97dd",\n      "org/jsoup/nodes/Element.class": "389114379404fa0ba9203dadd5085497445062af45c3cca35f115289f563e41b",\n      "org/jsoup/nodes/Entities$1.class": "0b6a8d94254bc353bd269e3e65745d6d11099717542d8ea4e8886177491b28b2",\n      "org/jsoup/nodes/Entities$CoreCharset.class": "51bd3fa786636bfd871cb836339b7a9ec723a241de5f8e19ac1f9c7875aa53e5",\n      "org/jsoup/nodes/Entities$EscapeMode.class": "ea5561cda5742ed20d9ef2627f3ce12419214e31f51e36679806aa1109f6f627",\n      "org/jsoup/nodes/Entities.class": "5ac6a626f8ffc36f0a7860cbc6117a24a1dea6d2e873f678248d8be74144bc84",\n      "org/jsoup/nodes/EntitiesData.class": "165e45f79154ed561fa616061063bffa7ef9862de9d5e4d8af1143f92a59dfd6",\n      "org/jsoup/nodes/FormElement.class": "52d236f25d14af1d98603f4ac3e7818de68bf616c8f7f08b89553622cce03fc7",\n      "org/jsoup/nodes/LeafNode.class": "df27c0f731a7ad92f72827d4cc49d310cc829f8888b471064e42800094824463",\n      "org/jsoup/nodes/Node$OuterHtmlVisitor.class": "be28e49e9d9ebbeeb6d65b9d33e5517bc215e5ba28decbdd4f76bf8ce831ff2c",\n      "org/jsoup/nodes/Node.class": "408cf6a9e6fe3fee51fe65f851d8f70ba46aa33d135b86304aa7c165e8fb2e25",\n      "org/jsoup/nodes/NodeIterator.class": "a644da9226957a094af4ceb6d1638b31daeb3aae01b185558cdfe7d554d23c0d",\n      "org/jsoup/nodes/NodeUtils.class": "d3e4051940d6622f0e39d269c967e96e0f45b6a50b241164f0b4ab8139d535ae",\n      "org/jsoup/nodes/PseudoTextElement.class": "abc3cf68189a17c038d97fed789956aa0dd58b45ed517efc261987e81e1c53ab",\n      "org/jsoup/nodes/Range$AttributeRange.class": "61d22f3124862789e4daf1da3a0d4622df2a2f099b1d253808e6a6476ff2a1a8",\n      "org/jsoup/nodes/Range$Position.class": "857944298608df161b2758a6298e3ebeb37cea68aabf3cb383d6c187b9c9abed",\n      "org/jsoup/nodes/Range.class": "7151cc0b4738d208d2628791725f3905e58b67dbb1a416b1184d8e6786a83308",\n      "org/jsoup/nodes/TextNode.class": "1eca735c1f74daf12874989f837ce8f0840c66fd7c7e8f9021e87853daf01647",\n      "org/jsoup/nodes/XmlDeclaration.class": "03d4a83a973bb6ad069140f23ee221beff0d0293d1b29c0ea30779bd3e1930f9",\n      "org/jsoup/nodes/package-info.class": "a8e4fde222a492a110534b823849a741d32fdfbf778cf7737e47f2e8fec64a55",\n      "org/jsoup/package-info.class": "5ac66a48d014c872a2d4972a11cf5d384379c3de082fa32d862b2e9e336df17b",\n      "org/jsoup/parser/CharacterReader.class": "9953646ad4b36fba55ffa0a703015370b4b42eea8f8653a63adf5a56da987255",\n      "org/jsoup/parser/HtmlTreeBuilder.class": "c5a0bda2ecfbf4cd812570f4744ed1143da866a058ce4aff45cf00af048bd18e",\n      "org/jsoup/parser/HtmlTreeBuilderState$1.class": "956b6165d4b4623b035000ca3cc4e25826a177772d6c4c268302704ddd33d45b",\n      "org/jsoup/parser/HtmlTreeBuilderState$10.class": "f753a6889c0c15df66e2c2ce6517c33db8ff1f7aa2a99bff6bfc3e13b2d67c47",\n      "org/jsoup/parser/HtmlTreeBuilderState$11.class": "27169d882dcaf36f3d84b2dbffe849df4f10eea73e20679d64bd322c898df703",\n      "org/jsoup/parser/HtmlTreeBuilderState$12.class": "66b8a9073872899c9e556246171ede1d70fc442e1164b7cbcc486e7d204f7248",\n      "org/jsoup/parser/HtmlTreeBuilderState$13.class": "427bbd0e4339c0db3a19b234d66c04456d764e705d9a5578392de1986fe5b081",\n      "org/jsoup/parser/HtmlTreeBuilderState$14.class": "1b66b3145feda12f43143f8c5a9ef051f56701a56927feb58a1cf03a62678780",\n      "org/jsoup/parser/HtmlTreeBuilderState$15.class": "65fa4e976099ab34f92456a32fe8a552934d56a73a3f751a57bdd6ebdaeade0d",\n      "org/jsoup/parser/HtmlTreeBuilderState$16.class": "1605dd55e5d6559af005a9f71104be983d747e3c89f96eb6744a7896f9e297fb",\n      "org/jsoup/parser/HtmlTreeBuilderState$17.class": "fc0d986b4bba07b9d227085ea31966e48209fb94fc7542cb1d7d5cb4f24e0dea",\n      "org/jsoup/parser/HtmlTreeBuilderState$18.class": "4267fca764d2e612bece28e28eecfab690a33b2faacd0588a6a1bc80ffdd8ec7",\n      "org/jsoup/parser/HtmlTreeBuilderState$19.class": "32a8a908ac59540c8533282ddde680a234b0b00922d9bcc6e3ba396f54bbd3c8",\n      "org/jsoup/parser/HtmlTreeBuilderState$2.class": "b6bdad5c784590de795a89bd13331c4592cdded13ed8ae8ae0488532d3973112",\n      "org/jsoup/parser/HtmlTreeBuilderState$20.class": "15839d3615643fd04fb00e5990e73711660d25e22d2e13862783ea4680a956ee",\n      "org/jsoup/parser/HtmlTreeBuilderState$21.class": "1f25a39afa1fd5ca22b48d6caf36f86d083a60c0c489025e850cadafcaab2beb",\n      "org/jsoup/parser/HtmlTreeBuilderState$22.class": "df96fb436c29232e80b4b1d0f821d20c3a26d13fd43df0c32efc66ff1b987704",\n      "org/jsoup/parser/HtmlTreeBuilderState$23.class": "a363079d7dbbc39508394a4e741e94e07d3167cd33a9e27bac8e10f8f5a65760",\n      "org/jsoup/parser/HtmlTreeBuilderState$24.class": "2b417adb803940f6afee4faafa7b54114cbdc35fd97cc9d798ab145e24d75eec",\n      "org/jsoup/parser/HtmlTreeBuilderState$25.class": "ee2b7baf8c4ec41d2e794e5699e234be8357e589144f2a19eb0e40540e5763ac",\n      "org/jsoup/parser/HtmlTreeBuilderState$3.class": "4027ad7d0651168fdbc191cdb273549a9917b6b6a29010ada38bff500373ea59",\n      "org/jsoup/parser/HtmlTreeBuilderState$4.class": "d577631ea025aa2ddc9035ab2fae74a4af43a22bc7ab234a912a18fc141410d2",\n      "org/jsoup/parser/HtmlTreeBuilderState$5.class": "8cc31d128e8e385a418c2c16cc9aed5505cc5b3ca7760fe13de6706f02ddec4a",\n      "org/jsoup/parser/HtmlTreeBuilderState$6.class": "baf52be2f21068890c69854c093a48e770e7c850c5f1f6674192182b4a7bf331",\n      "org/jsoup/parser/HtmlTreeBuilderState$7.class": "d3c39e1f11848e0029644e9e36e387adebdea6af0193e458e7e64d38c489f014",\n      "org/jsoup/parser/HtmlTreeBuilderState$8.class": "b70db5296d48570dfac1b61a9467ac20f12e75acdd4d8d4f9554cfcd7c1c331b",\n      "org/jsoup/parser/HtmlTreeBuilderState$9.class": "7bcb330d68918869a4c761f7d4a8ce142427d416676178919ef5adf7381790f3",\n      "org/jsoup/parser/HtmlTreeBuilderState$Constants.class": "56fd4d9c1f694fd22c5772adad37a019bcb9b29e7b81a985c6a7e602756f42a4",\n      "org/jsoup/parser/HtmlTreeBuilderState.class": "bd090c1b70e1821d5a9b52b4ec29a0b92c28fdbb3fa6358643c058e0f8efddef",\n      "org/jsoup/parser/ParseError.class": "b17e81388786963aa3629df917578329336e66764d27e6864f3165568d56097a",\n      "org/jsoup/parser/ParseErrorList.class": "bbd92f18c6e0ec8667391aed5113e115a6f7085a03971118a30cd5eb671f73f4",\n      "org/jsoup/parser/ParseSettings.class": "7ddcbb73897b0c78126ca5c5a9de9b775ee402f0dbfb78eb2cdd9a6e96001b9d",\n      "org/jsoup/parser/Parser.class": "36c84975b7d7a2a341a88fdcee553ccc363b0b68398563665648e8eb22c19aa8",\n      "org/jsoup/parser/StreamParser$ElementIterator.class": "7e0ff0418effb4bc56c9468a7b61387d2f4b02de6fd04002a2e343df897ee448",\n      "org/jsoup/parser/StreamParser.class": "c9e4738d3f7abb8faee265d37080679a40aafc18b6fa77f6093b5398b21d7d22",\n      "org/jsoup/parser/Tag.class": "dfa1d297b9621a51b34fd9d73afffbe2bb72dc9d90b891b2ecc13747695c3c81",\n      "org/jsoup/parser/Token$1.class": "6f3adb9b7d967e2bae68a292d3fe58e5423343e1d351a2978c95cb46cb74a6a9",\n      "org/jsoup/parser/Token$CData.class": "4be17063bbec4c6740d1093b314080c18173da46f9f2a0c501e488ecdaf9e0f8",\n      "org/jsoup/parser/Token$Character.class": "6f0a8e782c5fc448df1f50d561022f6be523e3c599f7d81ae57819c49eec79d7",\n      "org/jsoup/parser/Token$Comment.class": "80495b873e6e3a819457d9af00f15885970eaaa6e5dbecc32ac3a0cf1160734f",\n      "org/jsoup/parser/Token$Doctype.class": "6607aa58375431b4e0a3a9549a60beb8651d0b31b7fb4be3ded7bdd7201e1269",\n      "org/jsoup/parser/Token$EOF.class": "9d14f8ba552f216e8a5fec8e24c474933fb0aba1a7d48f65c96f57c668a03fe7",\n      "org/jsoup/parser/Token$EndTag.class": "c429b96b70d47d284ccfcd2c91c0bc58a6987b37534ea9bf6117d6f4cff1fcaf",\n      "org/jsoup/parser/Token$StartTag.class": "423eb434a3c400673c23b87d40f6032d8d56b55d64f6a254d06383b8530c9695",\n      "org/jsoup/parser/Token$Tag.class": "01a5d7455a97609b0ad1c6f9ff32a27683dedc473a703c8e2fe8e9ee952d1422",\n      "org/jsoup/parser/Token$TokenType.class": "f031ea564463210da2656d91d11eb6f50c49826968b5dada110ca65114446ccd",\n      "org/jsoup/parser/Token$XmlDecl.class": "7b3f6998e94f0899c85ad7caefcb5668d80476669a23a9134344f2eab3f5c6e5",\n      "org/jsoup/parser/Token.class": "175633ec0017ea28cd0b91033f27a75471b5ef0158ca4458d038c146d3f87c98",\n      "org/jsoup/parser/TokenQueue.class": "4c4924a73be7c276f784c710633c7c7f43334a695ea3f3320481d6d94adf4188",\n      "org/jsoup/parser/Tokeniser.class": "45d424919ea06d078ead4c23f0119cc5e3bfbc0d11be5c63529c8eda164997d8",\n      "org/jsoup/parser/TokeniserState$1.class": "4c11edbcbb8137cd60d7c901f14c87833accd006a2036852effc45754713ed8d",\n      "org/jsoup/parser/TokeniserState$10.class": "17c50fc3f3dc0c5ab1f554bc9b757bc9e722ba867aaad85b643006f1f25b331f",\n      "org/jsoup/parser/TokeniserState$11.class": "d99be7c5b0d559d1d4f914d7af1416effe34aeca1b6fc61324aaa6f38ab59784",\n      "org/jsoup/parser/TokeniserState$12.class": "44c8a2bcb87b298fdcdbe0f699d716b8dcca6200cdaaeebc71a794549c6ed51e",\n      "org/jsoup/parser/TokeniserState$13.class": "09184874ce15f194c30f5169215bb1e635d6ae647cf9df23d72f55b1e1df1193",\n      "org/jsoup/parser/TokeniserState$14.class": "bad31aad08eb7373e470913be16d121ce296e3d87ac3f27cce2b01377b344806",\n      "org/jsoup/parser/TokeniserState$15.class": "eb5c1fc2bbf15351d730dd3e97b8600e29ff028c3953929e58b0715b7e6f8ca5",\n      "org/jsoup/parser/TokeniserState$16.class": "3e23254f076f2f418293d7d3cf13f3ad4831178b6ed33269e4af7717fd141bdc",\n      "org/jsoup/parser/TokeniserState$17.class": "ea23496874ded86b3d258d34e80ef7a808e3a97cf358ca8716c18ad8c61427d9",\n      "org/jsoup/parser/TokeniserState$18.class": "bcb021181175c79a66912a867e244ef4e65f7731ef575a35c9ef3c0fdb7b19ad",\n      "org/jsoup/parser/TokeniserState$19.class": "ca13a6df4313ab12b59452a25df06c3a8b2baaa10c5aef31a6ceba45adc39992",\n      "org/jsoup/parser/TokeniserState$2.class": "63e8cf53410ca7c2c4aef9660ce06ef9f3dfcd39169c1f73dd5280539817e5e2",\n      "org/jsoup/parser/TokeniserState$20.class": "a19f68c14f0194fad1afa2ff198b0acf37155afa5c05b5e94af0c2d168d9e9d5",\n      "org/jsoup/parser/TokeniserState$21.class": "a9a66a6f4117259cb6086f035d2ad01789a61c37dab6c9267d824498374df9d7",\n      "org/jsoup/parser/TokeniserState$22.class": "740f2b9b3ffde19d7eb85e3111454ac79278460d16251cf4b69b9106f4e3b14e",\n      "org/jsoup/parser/TokeniserState$23.class": "17e0e6f666bac80af73dddc42ba5116de5d302d0369c1f62bccdedcf7c7cf831",\n      "org/jsoup/parser/TokeniserState$24.class": "61badabcd7cf24eef1a837a4629bf09c7c6fb0d231ccecc08d060799ab8a96e4",\n      "org/jsoup/parser/TokeniserState$25.class": "ad913746ed929969defad6ae968fe90620ebfcdd7349ecdb2151fe18f14fa03b",\n      "org/jsoup/parser/TokeniserState$26.class": "d7777501c5d67a8971b524371a3e63d7b9e82cf93e43718119d830659cc77ede",\n      "org/jsoup/parser/TokeniserState$27.class": "faf8f2f35ccd30ef479e1fd664ad9198f34bb7550080823fdb69ca0603e259d5",\n      "org/jsoup/parser/TokeniserState$28.class": "6ae8d0fbb1bdab2fb223d4d5477c47e937225b82e0d46018ffbd262c10929e90",\n      "org/jsoup/parser/TokeniserState$29.class": "5715fb6e39914ada6bedc37b2f52f9dff75a3410876b2f04c718925710b7ade8",\n      "org/jsoup/parser/TokeniserState$3.class": "1974f1808d22445ac400ddd81174c8fbb38b27551508088603f20ea3a74008bc",\n      "org/jsoup/parser/TokeniserState$30.class": "2bacc2db3b9fe1cbd45b21a4e289ab90092a8e00d51c083c1f8087f91577e301",\n      "org/jsoup/parser/TokeniserState$31.class": "a8b06398f05094a1bc7ec85812f006fe847a9ddc2b1dc17b4c34713a9fae34c6",\n      "org/jsoup/parser/TokeniserState$32.class": "e525147c288eaafe4f19b964c54519b46e1ca2b169baa37477c7f8fad89025e1",\n      "org/jsoup/parser/TokeniserState$33.class": "81dd686af692105f8e21f18fe3c73f7b98eb85171b506fcd5cf43f07be2660fd",\n      "org/jsoup/parser/TokeniserState$34.class": "af0d7d117038bb25e0b30bf15dfd49cfb9e5dc53426119b163882d4c4ce7c1b2",\n      "org/jsoup/parser/TokeniserState$35.class": "5d22e2e8029fb47b0063a1e1ae87d2425d4b7e3c59df1c933e3727bf14683b84",\n      "org/jsoup/parser/TokeniserState$36.class": "5f83c30e168cb4746c7c0589bfe25e2b33633ff4ea73607a9681c3f84a50ef28",\n      "org/jsoup/parser/TokeniserState$37.class": "acc50b21456a98a61e74b0d504ed9a029827e7a0f861854dc7197817d52539f9",\n      "org/jsoup/parser/TokeniserState$38.class": "747ef3ec1f08a2f2259f78e6b52d2469025832261911f4f84d0b2e1bbaa6a23b",\n      "org/jsoup/parser/TokeniserState$39.class": "acea39b9b9e932b282b5f2dad19541aa2f29d44798aee6bb3a9c67c053887f83",\n      "org/jsoup/parser/TokeniserState$4.class": "00a5922311138cb506c0f808d5afe50feed464454c85514031d5207b0ba81f33",\n      "org/jsoup/parser/TokeniserState$40.class": "a2ffd0da3e2adc52bf2de7a7e025095699962dcd1a931ba0b350cec7e26fad8d",\n      "org/jsoup/parser/TokeniserState$41.class": "0e95b65be52931e241e06d55354f0515a7bd0488770948e5268430ed98be5355",\n      "org/jsoup/parser/TokeniserState$42.class": "eff711b32108dac5fa2498525fd2a633357bc10e8e4c5eba52df453bd3f0dd44",\n      "org/jsoup/parser/TokeniserState$43.class": "4da35ccbc6bdd136a2eb4f26614b88c5fb4d70beb976a340029b351046cf51ac",\n      "org/jsoup/parser/TokeniserState$44.class": "8ed3b557cab9ef5526de949736fe7b3afbfce8ed4ca039f637b09f5c60c7272e",\n      "org/jsoup/parser/TokeniserState$45.class": "5fafe74c4e219bb778abffd29a7afbb8bac4bb02725dcb5fd5fc8c18a5aef64a",\n      "org/jsoup/parser/TokeniserState$46.class": "ed7d5f348114cb27f1b82c23a96a265706c864e3b56e3059d5bb8bb313482fe4",\n      "org/jsoup/parser/TokeniserState$47.class": "22ba0429ea2b725a9bf370543a5ec7c02fca93d59ac1e3c0942073948e70a513",\n      "org/jsoup/parser/TokeniserState$48.class": "490712bce7a159c83b663ef8939047c6da731a766f2e87895d69e80e216b3958",\n      "org/jsoup/parser/TokeniserState$49.class": "09919dbacb04ab9c4284c57a06314a24dab93ba650595af6c943d5696f7b7e6e",\n      "org/jsoup/parser/TokeniserState$5.class": "4e8920dd33c325494801ba85be8b42cec0d4fefbda21e799a8f10c22eda0bebf",\n      "org/jsoup/parser/TokeniserState$50.class": "c16d14831aeadc9aee7ae6cefbf1a4e1a32f5019e26a08e2c1858c2cf9b1b019",\n      "org/jsoup/parser/TokeniserState$51.class": "b4f6879e19ed4cf5f598375244d8d9821911acdb223add60689a669f85343d5b",\n      "org/jsoup/parser/TokeniserState$52.class": "4c7019ac3cb6752b07ac6cc87482366d80e7bc4c3f69408a1701461ffcab7432",\n      "org/jsoup/parser/TokeniserState$53.class": "663b4388d61d5aefb613a9472194a92050febba6f11a6e790f3bdcfa3a0b1ebd",\n      "org/jsoup/parser/TokeniserState$54.class": "3ea3e4a60e81cb99fbc88797b6607abfcec55e1d762ff9d473c02585a26e020b",\n      "org/jsoup/parser/TokeniserState$55.class": "71a68c7309ba796389fcc4772a90e2fdb28410c35a345631cbf97fc713c865c5",\n      "org/jsoup/parser/TokeniserState$56.class": "a701004f42cdb1d2d31aa6c2e0877882987470f6a4a7181f1c5ea1890688acf6",\n      "org/jsoup/parser/TokeniserState$57.class": "313e1043e62ef3d69adfc807e204a99813f00d55423f4bbfd7447d61f91abe5f",\n      "org/jsoup/parser/TokeniserState$58.class": "938f5bae78ea8c7f8735e843559902023bf058da985a17c24486af9e1fd9b794",\n      "org/jsoup/parser/TokeniserState$59.class": "5253fd1a948b2debffd96f8cb7ce02ee125e89b37c042d601b4ee4bc8c9b22b4",\n      "org/jsoup/parser/TokeniserState$6.class": "cbf555f52f4ee696f18c3890d7d6d22c8b7bd3d168cf768035f00bb52edfbcf4",\n      "org/jsoup/parser/TokeniserState$60.class": "029beb9ad9bb7fea7651fc099dd6500588474b70f84f19c7a50d725a8fb96932",\n      "org/jsoup/parser/TokeniserState$61.class": "cbf6decfcf63599d7dce33de4ea8924a52f594eaf9c650f181cd082ef3f46c5c",\n      "org/jsoup/parser/TokeniserState$62.class": "53e4b18ae5ffd18a30d5254d6a76c49eb29e0f613bbe3cd3f3075ee9d65d61d5",\n      "org/jsoup/parser/TokeniserState$63.class": "9bd07158f7bf61ac819d348d36f37038a609e1c73bce68c4f4429195b4dfe0b6",\n      "org/jsoup/parser/TokeniserState$64.class": "84301625db4da93284974af96453d27eb28c0404cff2037ba27b968535ae357d",\n      "org/jsoup/parser/TokeniserState$65.class": "12c805ef0fd3f952195143e9b050d45fff2eddf73a555ec25e5009b231ba2290",\n      "org/jsoup/parser/TokeniserState$66.class": "0992e99496d8398f12395f8d1128805e5160a8fd745afa8f8ef47dcb029a35e5",\n      "org/jsoup/parser/TokeniserState$67.class": "164b75ec1e4462e95106fc7bd36eab24f4e74882cac63f0cb6a39a8156ac4bb3",\n      "org/jsoup/parser/TokeniserState$68.class": "94fcf62633a65ecb61bbeab3721a4b4d63656d7a03ec662f59ded3ca51d5f766",\n      "org/jsoup/parser/TokeniserState$7.class": "9f205d50c657915b777271e2040e5312cb4398b5d3fab484eaa3aac19083e15c",\n      "org/jsoup/parser/TokeniserState$8.class": "0063b8acd3ef7bb672eace99a96c996562fc4716bedef54202440342decc15a8",\n      "org/jsoup/parser/TokeniserState$9.class": "d844b4688cfc22ee7c7f070d1bf1ec29cad26a404033cebf4fdc47ca4b825b83",\n      "org/jsoup/parser/TokeniserState.class": "304510bede635b82d9362306b6e90e56d66f6bb0b0c66c26fdb640e1b7c96403",\n      "org/jsoup/parser/TreeBuilder.class": "325180718a259fc4525b5c58ca47a644b1117a927d6c2d8b394e9fe08027597f",\n      "org/jsoup/parser/XmlTreeBuilder$1.class": "2bca133a99f4d51c59c48f5cb8c2d2ba080f4e798f8d9db62802ee6039bf5825",\n      "org/jsoup/parser/XmlTreeBuilder.class": "20ef1840be3138253a972e6aa0f8c7c5fbb646c0bd57bcf54b48c4703d9cb4b4",\n      "org/jsoup/parser/package-info.class": "2f2a1aebb16a42706de12956a6046ffd6a2e4e065a7a949f8aba9cc982261d86",\n      "org/jsoup/safety/Cleaner$1.class": "7c27cf8d818fa956bf3a7fae85e041b43ac3d24dc9daf1808335a8ce8be052fa",\n      "org/jsoup/safety/Cleaner$CleaningVisitor.class": "6b3c7740e68eec4cec21e408d33ae699f524c781ba6daa27afee6c5b993a4314",\n      "org/jsoup/safety/Cleaner$ElementMeta.class": "b61714c21232f13a3b34d8a054f9ba7baa30430653d9311bc92389d384ba2a8d",\n      "org/jsoup/safety/Cleaner.class": "ad89c3f8a580ff2a0d32d3844e9c4ff6783e4a54de0d5a19676d7f12f7b3edbb",\n      "org/jsoup/safety/Safelist$AttributeKey.class": "081541f182f73ebe4bf87a2c81bc0d2f48d91137064a1bbca57c73c9887149dd",\n      "org/jsoup/safety/Safelist$AttributeValue.class": "06171b5af9222a7aefb9399e966dc26215ed8ceb62332bcc4b9eec721135fc24",\n      "org/jsoup/safety/Safelist$Protocol.class": "e0dca079c815c48a3cf4e5b86e8eed37cf7789823c56a4bf1b60886f7982fc79",\n      "org/jsoup/safety/Safelist$TagName.class": "ba0e4102efa628f2d6bb83ba9717f1bffd9d649e980ab770931d8b9bedf5cdea",\n      "org/jsoup/safety/Safelist$TypedValue.class": "b596c6eebd7a49aef7e93df2b8a1f3dc020a5a3bad6d222065af628cf52e368f",\n      "org/jsoup/safety/Safelist.class": "e22c09ed92d3c80c77d242ac105dca2a76a6cb27c9a6ddc5f9289052fe8e5806",\n      "org/jsoup/safety/package-info.class": "5a5cdd21cc8eeafc016b7a4ee66227b36b85ba32262356214be3a183b0873ac6",\n      "org/jsoup/select/Collector.class": "6c4f17e6a288f46c40009fffa44d41b05557f4e06915e78d698a9798d4af610b",\n      "org/jsoup/select/CombiningEvaluator$And.class": "7bb8fc39bdbaaf1892b4200c4bcbffe4501c6922e9fd7d7cc6ea59cf638302ad",\n      "org/jsoup/select/CombiningEvaluator$Or.class": "6d377e581497023687df5ef5d6d72e555370b5308f9fd6b002fda4132ce42b9f",\n      "org/jsoup/select/CombiningEvaluator.class": "a93c66d8af9b18395386a015671739a327a74e78c4bdddbbcdec9a0a40cb8243",\n      "org/jsoup/select/Elements.class": "c60ae7e2ffb719a3c4c8700efe915fe10c73586014ddbfc468435176d71235f1",\n      "org/jsoup/select/Evaluator$AllElements.class": "83901abe162f9ab2e2cce08b371c89a82e48890747e87b337a27287ebccf1a51",\n      "org/jsoup/select/Evaluator$Attribute.class": "c0d3f41d964b35a98e3695a7c0fbdceb7b3e83d4c83eb2557e0bfe12d74fd4d8",\n      "org/jsoup/select/Evaluator$AttributeKeyPair.class": "0ad54afa07a269e4dae6674a34e858021f45fc92a37a7218d0e49ab847f635d6",\n      "org/jsoup/select/Evaluator$AttributeStarting.class": "bf82e47d72b72719ff85cc13971169ebb2c1c266f0a8ff9ba554245504f3285c",\n      "org/jsoup/select/Evaluator$AttributeWithValue.class": "9a2a379f3baa3f512cff846814982920acc88b26b6a6e96912bc7154c9daf0ea",\n      "org/jsoup/select/Evaluator$AttributeWithValueContaining.class": "0571f501810b24b376cd6993ea2fc67088005274b39d8b28e8ae3ceea94778c3",\n      "org/jsoup/select/Evaluator$AttributeWithValueEnding.class": "4698aceac0eae8e401d85849602e76c8b642a95f3d218ddbbfc11986f7753b63",\n      "org/jsoup/select/Evaluator$AttributeWithValueMatching.class": "60fab4ff1f42b8a295c17468a381a544e97088cd8a61da71795f574b4dd7205f",\n      "org/jsoup/select/Evaluator$AttributeWithValueNot.class": "40e963651ddea8e73e65f1a97c49dd8274154fa830b80b345f66601f499e4a20",\n      "org/jsoup/select/Evaluator$AttributeWithValueStarting.class": "f3b09e2639a5651ef230d209671534b8f0d169f6dbaaf7b30a1bee32210dfee7",\n      "org/jsoup/select/Evaluator$Class.class": "026ac5df8537e93dcf6f114c9f643abc644c52ab7418462ea4a8c134fed99fa3",\n      "org/jsoup/select/Evaluator$ContainsData.class": "f7738d9e26f0ca240427d4e193b15e89090f5cf98f0d24f675377f7028d87ea0",\n      "org/jsoup/select/Evaluator$ContainsOwnText.class": "5d5992d8297047b9a871ccb3217d5fa6599040baa24cf233e00326940a92e136",\n      "org/jsoup/select/Evaluator$ContainsText.class": "9c914d6f3f7e6a926e756a3c6c88b7739fe7d0fec854cc0d9a835b2d2579d26e",\n      "org/jsoup/select/Evaluator$ContainsWholeOwnText.class": "2d134aa64f9213406db36d062c03f91ccf40f50fae9126e8ebbbef520e595ced",\n      "org/jsoup/select/Evaluator$ContainsWholeText.class": "5581ae8d80a55cd44eb209831756e8213dd260c0c2f9188311e32c564486ab71",\n      "org/jsoup/select/Evaluator$CssNthEvaluator.class": "e249cf7a128b0488a93906d1c676244abccd68a6129ab2a6c619892b0b2725fe",\n      "org/jsoup/select/Evaluator$Id.class": "3c6ed23ad1284e5aa37e32e7c1502700139e41f8b2b88af729614b663d98b0ab",\n      "org/jsoup/select/Evaluator$IndexEquals.class": "8c0a545059094d8b816b3402f0ab5decb90567793372385a257439ccd5d8c40e",\n      "org/jsoup/select/Evaluator$IndexEvaluator.class": "f5afed3ec3d42c9cd97a1aab900cf18db24456fd14f1eac130c5af57f2e5bd82",\n      "org/jsoup/select/Evaluator$IndexGreaterThan.class": "e7715be4d4c6830dc0360b87163341606ca392fe83996bce9700ee7c64809e21",\n      "org/jsoup/select/Evaluator$IndexLessThan.class": "fdb567d97d3442e2fbc64e1ec30425a1751992e1b99eebfcf345a3f5feb85869",\n      "org/jsoup/select/Evaluator$IsEmpty.class": "d398b955c655fcbec9d7b2e5a049ed01d1952da1526833b00cc42dd3c4a6d237",\n      "org/jsoup/select/Evaluator$IsFirstChild.class": "e2f85d066b18c72e24417d81eb16209a9000253ac19400d87de40f61d42fb5e8",\n      "org/jsoup/select/Evaluator$IsFirstOfType.class": "860d72fd0cb75e2ce1c4024310c773c2a905c0a4c918c65b5508765aecf14b85",\n      "org/jsoup/select/Evaluator$IsLastChild.class": "273203e5fd518dbdb40abab86944f9ee6fb99d78b225f22397cead4c4c4cbf83",\n      "org/jsoup/select/Evaluator$IsLastOfType.class": "57ffc4eaabb9c45e5802350aa2d644430554fc4a972673f1801cbd626c72d186",\n      "org/jsoup/select/Evaluator$IsNthChild.class": "aec8f606e5e8d6f05d7faf97ea48a548dc21493f86e8e680c8dd8e974e299655",\n      "org/jsoup/select/Evaluator$IsNthLastChild.class": "411497b15cf1ad554cda7b699ccfd2c6013f2e76a2b1ef66a4afd55f99887088",\n      "org/jsoup/select/Evaluator$IsNthLastOfType.class": "3e450b4d9b0962df4e702a2e61975d20907574934a2e1b765036b6ebdc5011f4",\n      "org/jsoup/select/Evaluator$IsNthOfType.class": "1645f0a0efd50c1028adac71d281ed6a55ff88f1f70ca8a55a57c60b5f2da3d5",\n      "org/jsoup/select/Evaluator$IsOnlyChild.class": "021a7f1da2dc061423b581a9626f1df0b143cb15efbe2be1a8d2b0157c9f0791",\n      "org/jsoup/select/Evaluator$IsOnlyOfType.class": "0dd0ee3c1750aa919354a413ef6965f3a9a7767bc714369bb762d517b411ff75",\n      "org/jsoup/select/Evaluator$IsRoot.class": "85ef7462afe5bce4fd4ae77cfbe5eec71d261bc38a5ae26ea42fa5f225b6ec22",\n      "org/jsoup/select/Evaluator$MatchText.class": "ed9056e7bf28f68db449bc6bbf04d68887b0a065f1ede5ccc084975559217fa0",\n      "org/jsoup/select/Evaluator$Matches.class": "c83f09f7dd52853efb5a852df885de0a25ccee86741498083f96916d979fc3ff",\n      "org/jsoup/select/Evaluator$MatchesOwn.class": "9434d3c357173b8c7d466ffca80ab9394eb83443d68dedf6713dc26c1aef7c4e",\n      "org/jsoup/select/Evaluator$MatchesWholeOwnText.class": "93bcb91bcd3c0fb8dee0fb1f20b75d05c836b479b572a01c8972769c1ee3a21a",\n      "org/jsoup/select/Evaluator$MatchesWholeText.class": "0d8b64b857a3e7eb428ebb1161f15357eda65a0231ecbf91e3234c53350af301",\n      "org/jsoup/select/Evaluator$Tag.class": "c5bd6c0a504121eb69cf703c7a28c837f2747997deff31f7024b1c5a3a8d1838",\n      "org/jsoup/select/Evaluator$TagEndsWith.class": "1a3771f6089d88e75523c1eebb78202561aa1528dbf87f2db605a768d64b999c",\n      "org/jsoup/select/Evaluator$TagStartsWith.class": "a651ab87de836faeb11f30c5c1c2aab32e02b6efd11311a09f376b076149d2b6",\n      "org/jsoup/select/Evaluator.class": "ef349107b93df7ea4cbc0d088d656a005811205addfdeaefb34d0df9315f8108",\n      "org/jsoup/select/NodeFilter$FilterResult.class": "6e3bc3a1608e24748e09890edee3bc443311634cd16734fdb9ab15ddf6570b45",\n      "org/jsoup/select/NodeFilter.class": "657a89041f912f3ae1f0de7794b0dc1f4cf4480a4c84db72042cf3511fed849f",\n      "org/jsoup/select/NodeTraversor.class": "0b7f20e65dbec362ad238559da12ac1f0d1da5f3eb434a2cb87b5647d4e4830e",\n      "org/jsoup/select/NodeVisitor.class": "5d6b56c222695b681569e87cd1a16bcee409ff75f36d9b62983347a618e53250",\n      "org/jsoup/select/QueryParser.class": "4c044b9b66288288723f3c0e3a7b35c5e38b8d37abffa8a942b5dbe83757100b",\n      "org/jsoup/select/Selector$SelectorParseException.class": "f3f5e6677b845fbd210bbf8c66b4b55fd209abb1815affcd5ea380b2178a3392",\n      "org/jsoup/select/Selector.class": "1172c4e40aae9c64fec77db123f6cda28b9b36258e5407971f1f2f308c90ab09",\n      "org/jsoup/select/StructuralEvaluator$Ancestor.class": "c659685ba074778bce4d1825693de84a5ae6ede345731c56cd23d1528f55d4d6",\n      "org/jsoup/select/StructuralEvaluator$Has.class": "3250673186f532e2bfdf6b91c908438f718d4f6d93f0443330ad1a4bd7343b2c",\n      "org/jsoup/select/StructuralEvaluator$ImmediateParentRun.class": "47b296c7ff24a622c4209dac275fae7aa1746e8414540c3068d0600110077dd9",\n      "org/jsoup/select/StructuralEvaluator$ImmediatePreviousSibling.class": "92b4bf7ad1792db2456d670de195c3161ef95be442ef5e79ab37089e656d7d2d",\n      "org/jsoup/select/StructuralEvaluator$Is.class": "79b2e73c1a1465baca15150dccd0b371ddc78777efc2c130fc5c173c6196c2bc",\n      "org/jsoup/select/StructuralEvaluator$Not.class": "8cf27eacef4a6a909130b2f78c7053c9dc1b33f6bb22a6f9a98435e874c229ca",\n      "org/jsoup/select/StructuralEvaluator$PreviousSibling.class": "5187721ed303854ff6cd9da054c0941286e2c14c5c112e25b762ca250c54e434",\n      "org/jsoup/select/StructuralEvaluator$Root.class": "d188b8b88070c952d5d4b208ff88e55688c3721c71d692c741091c362f654fd9",\n      "org/jsoup/select/StructuralEvaluator.class": "67dbf10469fe3d225abbdfefeef504740a25821de42ac158bd743a089b31528e",\n      "org/jsoup/select/package-info.class": "44832debbe2daa2f30d92c91c8749c8ead966f24040e763de8ef6f5e4655d627"\n    },\n    "fault": {\n      "org/jsoup/Connection$Base.class": "46903aee725945ec84e23d2d673ecd88cea4d7aae0963649b048c26d5883f98a",\n      "org/jsoup/Connection$KeyVal.class": "91748caceaf292c55c153eab6aa3db11d11f6dd606e5a7bc53462abaa14e5fbb",\n      "org/jsoup/Connection$Method.class": "a4156ab800562ca78fbf53d294df81fc543043ce9b11ddd474cddb530346e87b",\n      "org/jsoup/Connection$Request.class": "cc39b1ad42dd66edea641c22b2b29fb57aa35f2cc5a699e533f4ae96ec130961",\n      "org/jsoup/Connection$Response.class": "2f82eeba8c7392413813e9040d405e601c9ac5a6d561ba509d6678b9b7ba7aa6",\n      "org/jsoup/Connection.class": "25cea3ffc828f4982685d167f0272d1ddf26c075feb68feef6da06f9899c0d6e",\n      "org/jsoup/HttpStatusException.class": "3599c5812d0a5a3c27183ae62e5056c0ca4a9122d27e2ae3b715aff4c108c54a",\n      "org/jsoup/Jsoup.class": "39687db85faef3a051039716ae168b6cc9e48cb6b0254e89f5e674da6f44fbd6",\n      "org/jsoup/Progress.class": "4560bd5aa49e7fabb7a7029d8c6917af1088ad6b90b1741d00c5ca4e3e0e0305",\n      "org/jsoup/SerializationException.class": "bb0f694d579fa793c6803dd75397b400a802de696255a6b24a0181821e920486",\n      "org/jsoup/UnsupportedMimeTypeException.class": "ce38059772f80b371f6cc1c69447c4de8e7d897db5aa7c8cce12c649388888c8",\n      "org/jsoup/examples/HtmlToPlainText$1.class": "ece3d33b24a7c65d0f20f165ddcbb5f4c540edd449e10adebe4c07d8830e7551",\n      "org/jsoup/examples/HtmlToPlainText$FormattingVisitor.class": "67ffe8beee9bfd209c7b5cd9e0895c60340309de2c92b26a94b9f1146cf865b6",\n      "org/jsoup/examples/HtmlToPlainText.class": "25823d787c8579f3916b8b487e2307998b14c31fa48904c2d8c13d2e25cd729a",\n      "org/jsoup/examples/ListLinks.class": "f970fecdbf384ae89d081a2f4b319bd82847ffc02cce9c8a171c665100f87302",\n      "org/jsoup/examples/Wikipedia.class": "b5effe19f3f3190ead7c36e91a0c172b1b2b8e399f614952a3416be7a347aa13",\n      "org/jsoup/helper/AuthenticationHandler$AuthShim.class": "a1212e9288eb545e8bc18a02753374542ec6dabd586cf633571b7bb8261c4a9a",\n      "org/jsoup/helper/AuthenticationHandler$GlobalHandler.class": "66cac442429d23dbabfbbd844dc8e7f8e58430f033335896fef5895054248801",\n      "org/jsoup/helper/AuthenticationHandler.class": "8b07d1a2315ba92ffca0fcb7ec4dd68c1700fd3c536f6b6dbaff4fd0aa297a74",\n      "org/jsoup/helper/ChangeNotifyingArrayList.class": "5ce9f33b512fe57e1140a5397640b03839678264b853c55f03e60bb0eb4375db",\n      "org/jsoup/helper/CookieUtil.class": "d34ec7e142bf4fd69b29753e2dcd5f0bb801de2bf146af02f96161eda9096d10",\n      "org/jsoup/helper/DataUtil$CharsetDoc.class": "a66dca54be3c4983ebadb13c471902a781b217760126685f62678641f272e1e2",\n      "org/jsoup/helper/DataUtil.class": "000d0c0063446315db3ddaa9129964df478f6669700d2436a4a8d023fd9e394f",\n      "org/jsoup/helper/HttpConnection$1.class": "356bffc36479fe4d4f2b4ca827f6d926379da157dc7585d042eed92699b5a96c",\n      "org/jsoup/helper/HttpConnection$Base.class": "9ed2d5a699f12da3e7ab6d3e6c22f94ac51a654dbb0e80640f98232d4976d958",\n      "org/jsoup/helper/HttpConnection$KeyVal.class": "a5380f821f165714f5482692dfa9f153b4557d0fbd120bb05c27e49a9f2aaae2",\n      "org/jsoup/helper/HttpConnection$Request.class": "1b2684a13375afee800c11a8330d8dc1c13e7281fb2e3324d8b906c5d2614742",\n      "org/jsoup/helper/HttpConnection$Response.class": "2d63b81aaf62572d4ef90602d81b54c9e7dd74fedab78ae09d215b2246f2d542",\n      "org/jsoup/helper/HttpConnection.class": "bfe2a82a89187aa928b133f40130dd0dc04814009e2f8279f5bbc08977505350",\n      "org/jsoup/helper/RequestAuthenticator$Context.class": "4fa9001174b49601805a46d7ee36935cf4c39000411366ba556e03483bd642a4",\n      "org/jsoup/helper/RequestAuthenticator.class": "6b1b00079c6d3ee34384f2f9bbf2e179be7822f744238609f936bd780afb8084",\n      "org/jsoup/helper/RequestDispatch.class": "c3a5b7dc05d3a4ae52a3d519f2ced3166484f07d950435fe96729f1a056f2d45",\n      "org/jsoup/helper/RequestExecutor.class": "8c72ea3ec45b3f4fff288a7f7b9a88903fd5aa52f27b8965ff12a02373103c39",\n      "org/jsoup/helper/UrlBuilder.class": "d91ce6517232d615e6e9dafdab3b5f9d0ae43f6cd2ed89f10f35535b0727aeec",\n      "org/jsoup/helper/UrlConnectionExecutor.class": "bebf451a08e3e5061e6221267e7f73ac23797034293b69c327a8ca9a1e39308c",\n      "org/jsoup/helper/Validate.class": "b168d0a7859769238923101803bcefdfb400d48fa451279e77057fa000b8a5e8",\n      "org/jsoup/helper/ValidationException.class": "b3bd2a4aea69e7585f7c156d0bdb2bccea8c06859e7caf1f628e3908d8e5c016",\n      "org/jsoup/helper/W3CDom$W3CBuilder.class": "398c4b47e95681081d798992d1332d3bd4a9f3e79e982ee2353381c7cac8ebb2",\n      "org/jsoup/helper/W3CDom.class": "6ae881fd6760ca9431221b9de7809d094c86694eefa820410c6d3d04f0179cdb",\n      "org/jsoup/helper/package-info.class": "838331b3ecf370c9428c70e47145ecca0faff05d7a64c227dc9808f9c20c0647",\n      "org/jsoup/internal/ControllableInputStream.class": "1ce9ca524115517178032df0f287f8b0f4618d071f4383976a06119ea79176d9",\n      "org/jsoup/internal/Functions.class": "1bdf9e70c16c967f5adfcf7dd922fdbf11f158edd446620c81b4f23adc18f806",\n      "org/jsoup/internal/Normalizer.class": "af4b2d9d02c9821d9b5134a565c6f185c90cb50785cc352f28c19b579f1a6855",\n      "org/jsoup/internal/SharedConstants.class": "d060df166d0b07a818b3bbf8ef17dc3fb96d6c6440cd8462b2aecee931d9c159",\n      "org/jsoup/internal/SimpleBufferedInput.class": "d33f49a7d4b1c6239aee83259e38f6a42e0f035659c941a9326f3b1f947d8c94",\n      "org/jsoup/internal/SoftPool.class": "9ac52cce7b6f1c4108c6d20378c970bbffd2d4a2f85dfb35f11e955f71f87ad0",\n      "org/jsoup/internal/StringUtil$StringJoiner.class": "fdb3ac6163f2c5b0657836cf37ae2f611887c547ec126614263d1e4d3ddaf9b9",\n      "org/jsoup/internal/StringUtil.class": "f568c77b4def55664c169ccb7ef9c8a3b4526e55a18c5d4b97b1f691bea6c483",\n      "org/jsoup/internal/package-info.class": "48f68f412bf20bebbf75069f334ec0a36ba23c887dae36272b61fb2ceeabee54",\n      "org/jsoup/nodes/Attribute.class": "d738b75dd6c7b8b7d317bc5c750313b05ac467ea234ef1ec819814f083fc8710",\n      "org/jsoup/nodes/Attributes$1.class": "29f7e3a9b72dcb17155cf884fb7e5b9ed737b4888d650ff5812a1de1494a3a27",\n      "org/jsoup/nodes/Attributes$Dataset$DatasetIterator.class": "390d9c4962a03940243d85776f0bfa2c9fc6d8e956c4c9b2993adad3960e988b",\n      "org/jsoup/nodes/Attributes$Dataset$EntrySet.class": "c8f1242ef2059c6b02ffc11182ef7bcfca588ae7c1cd8af8114632edbf7286fb",\n      "org/jsoup/nodes/Attributes$Dataset.class": "e07cf9088654df7225d85456fead1dbbc44f0ae072fd104ae910c95ff2d6d2e7",\n      "org/jsoup/nodes/Attributes.class": "72fd2145df28a2b7bc19fe738c1d007c6e3eb5084a1706d9833fb42c615c8a7c",\n      "org/jsoup/nodes/CDataNode.class": "67f348f46b7c4e942dde41e678b19b74524d0516117885ca13dc20c01e759b3a",\n      "org/jsoup/nodes/Comment.class": "e23908900da901915bc9c3caaa032f4df868c35139721b0576ccd4c57898db1c",\n      "org/jsoup/nodes/DataNode.class": "80cc120787bdbcf87eab3b909c13dd558272a8c3dd7c0ec07337678da9194eed",\n      "org/jsoup/nodes/Document$OutputSettings$Syntax.class": "7e804a00fb3d868b0e20e2a87be911b5a805a11f5360a62468a207cd23a28a6f",\n      "org/jsoup/nodes/Document$OutputSettings.class": "20836331a21665abf0e5e09712228a0119777da096eac548a003f4c2d3ec98a9",\n      "org/jsoup/nodes/Document$QuirksMode.class": "0d33f9ea3a1231d3ff3f57a51043693ea8181ddc0c0969ca9f0b9f4aee3bf4ab",\n      "org/jsoup/nodes/Document.class": "fd6cf580574401a375af6c006ac8b7a47b4380faa75ce0e8da222dd187a4ff1a",\n      "org/jsoup/nodes/DocumentType.class": "730360dcef32b366bfd2b91c4d3692e290d9efc58fea2573041451798ec0ef0a",\n      "org/jsoup/nodes/Element$NodeList.class": "b0ea8de6b11cdcf0b18095861cacc52e1b3ef2ce05a8e3ae3b391dfa18c9dd41",\n      "org/jsoup/nodes/Element$TextAccumulator.class": "54c938bcf65d818a2e5af8f85caf32b121e39cb6aba025a66e9d8df9d1eb97dd",\n      "org/jsoup/nodes/Element.class": "389114379404fa0ba9203dadd5085497445062af45c3cca35f115289f563e41b",\n      "org/jsoup/nodes/Entities$1.class": "c67a6cfc6c5f47bbb2ec5c71e9ce2e87c14b9801a5ad807eeb0bb7f1ebdeff63",\n      "org/jsoup/nodes/Entities$CoreCharset.class": "abb6c5396df68c55e8dcb940bd1a07522fdff10fece15d14d6ec5b1017616d02",\n      "org/jsoup/nodes/Entities$EscapeMode.class": "ea5561cda5742ed20d9ef2627f3ce12419214e31f51e36679806aa1109f6f627",\n      "org/jsoup/nodes/Entities.class": "d5179d423088f0c4ce619c510f81a7ab904be38ddbb7c9cd65b40f79a8614e29",\n      "org/jsoup/nodes/EntitiesData.class": "165e45f79154ed561fa616061063bffa7ef9862de9d5e4d8af1143f92a59dfd6",\n      "org/jsoup/nodes/FormElement.class": "52d236f25d14af1d98603f4ac3e7818de68bf616c8f7f08b89553622cce03fc7",\n      "org/jsoup/nodes/LeafNode.class": "df27c0f731a7ad92f72827d4cc49d310cc829f8888b471064e42800094824463",\n      "org/jsoup/nodes/Node$OuterHtmlVisitor.class": "be28e49e9d9ebbeeb6d65b9d33e5517bc215e5ba28decbdd4f76bf8ce831ff2c",\n      "org/jsoup/nodes/Node.class": "408cf6a9e6fe3fee51fe65f851d8f70ba46aa33d135b86304aa7c165e8fb2e25",\n      "org/jsoup/nodes/NodeIterator.class": "a644da9226957a094af4ceb6d1638b31daeb3aae01b185558cdfe7d554d23c0d",\n      "org/jsoup/nodes/NodeUtils.class": "d3e4051940d6622f0e39d269c967e96e0f45b6a50b241164f0b4ab8139d535ae",\n      "org/jsoup/nodes/PseudoTextElement.class": "abc3cf68189a17c038d97fed789956aa0dd58b45ed517efc261987e81e1c53ab",\n      "org/jsoup/nodes/Range$AttributeRange.class": "61d22f3124862789e4daf1da3a0d4622df2a2f099b1d253808e6a6476ff2a1a8",\n      "org/jsoup/nodes/Range$Position.class": "857944298608df161b2758a6298e3ebeb37cea68aabf3cb383d6c187b9c9abed",\n      "org/jsoup/nodes/Range.class": "7151cc0b4738d208d2628791725f3905e58b67dbb1a416b1184d8e6786a83308",\n      "org/jsoup/nodes/TextNode.class": "1eca735c1f74daf12874989f837ce8f0840c66fd7c7e8f9021e87853daf01647",\n      "org/jsoup/nodes/XmlDeclaration.class": "03d4a83a973bb6ad069140f23ee221beff0d0293d1b29c0ea30779bd3e1930f9",\n      "org/jsoup/nodes/package-info.class": "a8e4fde222a492a110534b823849a741d32fdfbf778cf7737e47f2e8fec64a55",\n      "org/jsoup/package-info.class": "5ac66a48d014c872a2d4972a11cf5d384379c3de082fa32d862b2e9e336df17b",\n      "org/jsoup/parser/CharacterReader.class": "9953646ad4b36fba55ffa0a703015370b4b42eea8f8653a63adf5a56da987255",\n      "org/jsoup/parser/HtmlTreeBuilder.class": "c5a0bda2ecfbf4cd812570f4744ed1143da866a058ce4aff45cf00af048bd18e",\n      "org/jsoup/parser/HtmlTreeBuilderState$1.class": "956b6165d4b4623b035000ca3cc4e25826a177772d6c4c268302704ddd33d45b",\n      "org/jsoup/parser/HtmlTreeBuilderState$10.class": "f753a6889c0c15df66e2c2ce6517c33db8ff1f7aa2a99bff6bfc3e13b2d67c47",\n      "org/jsoup/parser/HtmlTreeBuilderState$11.class": "27169d882dcaf36f3d84b2dbffe849df4f10eea73e20679d64bd322c898df703",\n      "org/jsoup/parser/HtmlTreeBuilderState$12.class": "66b8a9073872899c9e556246171ede1d70fc442e1164b7cbcc486e7d204f7248",\n      "org/jsoup/parser/HtmlTreeBuilderState$13.class": "427bbd0e4339c0db3a19b234d66c04456d764e705d9a5578392de1986fe5b081",\n      "org/jsoup/parser/HtmlTreeBuilderState$14.class": "1b66b3145feda12f43143f8c5a9ef051f56701a56927feb58a1cf03a62678780",\n      "org/jsoup/parser/HtmlTreeBuilderState$15.class": "65fa4e976099ab34f92456a32fe8a552934d56a73a3f751a57bdd6ebdaeade0d",\n      "org/jsoup/parser/HtmlTreeBuilderState$16.class": "1605dd55e5d6559af005a9f71104be983d747e3c89f96eb6744a7896f9e297fb",\n      "org/jsoup/parser/HtmlTreeBuilderState$17.class": "fc0d986b4bba07b9d227085ea31966e48209fb94fc7542cb1d7d5cb4f24e0dea",\n      "org/jsoup/parser/HtmlTreeBuilderState$18.class": "4267fca764d2e612bece28e28eecfab690a33b2faacd0588a6a1bc80ffdd8ec7",\n      "org/jsoup/parser/HtmlTreeBuilderState$19.class": "32a8a908ac59540c8533282ddde680a234b0b00922d9bcc6e3ba396f54bbd3c8",\n      "org/jsoup/parser/HtmlTreeBuilderState$2.class": "b6bdad5c784590de795a89bd13331c4592cdded13ed8ae8ae0488532d3973112",\n      "org/jsoup/parser/HtmlTreeBuilderState$20.class": "15839d3615643fd04fb00e5990e73711660d25e22d2e13862783ea4680a956ee",\n      "org/jsoup/parser/HtmlTreeBuilderState$21.class": "1f25a39afa1fd5ca22b48d6caf36f86d083a60c0c489025e850cadafcaab2beb",\n      "org/jsoup/parser/HtmlTreeBuilderState$22.class": "df96fb436c29232e80b4b1d0f821d20c3a26d13fd43df0c32efc66ff1b987704",\n      "org/jsoup/parser/HtmlTreeBuilderState$23.class": "a363079d7dbbc39508394a4e741e94e07d3167cd33a9e27bac8e10f8f5a65760",\n      "org/jsoup/parser/HtmlTreeBuilderState$24.class": "2b417adb803940f6afee4faafa7b54114cbdc35fd97cc9d798ab145e24d75eec",\n      "org/jsoup/parser/HtmlTreeBuilderState$25.class": "ee2b7baf8c4ec41d2e794e5699e234be8357e589144f2a19eb0e40540e5763ac",\n      "org/jsoup/parser/HtmlTreeBuilderState$3.class": "4027ad7d0651168fdbc191cdb273549a9917b6b6a29010ada38bff500373ea59",\n      "org/jsoup/parser/HtmlTreeBuilderState$4.class": "d577631ea025aa2ddc9035ab2fae74a4af43a22bc7ab234a912a18fc141410d2",\n      "org/jsoup/parser/HtmlTreeBuilderState$5.class": "8cc31d128e8e385a418c2c16cc9aed5505cc5b3ca7760fe13de6706f02ddec4a",\n      "org/jsoup/parser/HtmlTreeBuilderState$6.class": "baf52be2f21068890c69854c093a48e770e7c850c5f1f6674192182b4a7bf331",\n      "org/jsoup/parser/HtmlTreeBuilderState$7.class": "d3c39e1f11848e0029644e9e36e387adebdea6af0193e458e7e64d38c489f014",\n      "org/jsoup/parser/HtmlTreeBuilderState$8.class": "b70db5296d48570dfac1b61a9467ac20f12e75acdd4d8d4f9554cfcd7c1c331b",\n      "org/jsoup/parser/HtmlTreeBuilderState$9.class": "7bcb330d68918869a4c761f7d4a8ce142427d416676178919ef5adf7381790f3",\n      "org/jsoup/parser/HtmlTreeBuilderState$Constants.class": "56fd4d9c1f694fd22c5772adad37a019bcb9b29e7b81a985c6a7e602756f42a4",\n      "org/jsoup/parser/HtmlTreeBuilderState.class": "bd090c1b70e1821d5a9b52b4ec29a0b92c28fdbb3fa6358643c058e0f8efddef",\n      "org/jsoup/parser/ParseError.class": "b17e81388786963aa3629df917578329336e66764d27e6864f3165568d56097a",\n      "org/jsoup/parser/ParseErrorList.class": "bbd92f18c6e0ec8667391aed5113e115a6f7085a03971118a30cd5eb671f73f4",\n      "org/jsoup/parser/ParseSettings.class": "7ddcbb73897b0c78126ca5c5a9de9b775ee402f0dbfb78eb2cdd9a6e96001b9d",\n      "org/jsoup/parser/Parser.class": "36c84975b7d7a2a341a88fdcee553ccc363b0b68398563665648e8eb22c19aa8",\n      "org/jsoup/parser/StreamParser$ElementIterator.class": "7e0ff0418effb4bc56c9468a7b61387d2f4b02de6fd04002a2e343df897ee448",\n      "org/jsoup/parser/StreamParser.class": "c9e4738d3f7abb8faee265d37080679a40aafc18b6fa77f6093b5398b21d7d22",\n      "org/jsoup/parser/Tag.class": "dfa1d297b9621a51b34fd9d73afffbe2bb72dc9d90b891b2ecc13747695c3c81",\n      "org/jsoup/parser/Token$1.class": "6f3adb9b7d967e2bae68a292d3fe58e5423343e1d351a2978c95cb46cb74a6a9",\n      "org/jsoup/parser/Token$CData.class": "4be17063bbec4c6740d1093b314080c18173da46f9f2a0c501e488ecdaf9e0f8",\n      "org/jsoup/parser/Token$Character.class": "6f0a8e782c5fc448df1f50d561022f6be523e3c599f7d81ae57819c49eec79d7",\n      "org/jsoup/parser/Token$Comment.class": "80495b873e6e3a819457d9af00f15885970eaaa6e5dbecc32ac3a0cf1160734f",\n      "org/jsoup/parser/Token$Doctype.class": "6607aa58375431b4e0a3a9549a60beb8651d0b31b7fb4be3ded7bdd7201e1269",\n      "org/jsoup/parser/Token$EOF.class": "9d14f8ba552f216e8a5fec8e24c474933fb0aba1a7d48f65c96f57c668a03fe7",\n      "org/jsoup/parser/Token$EndTag.class": "c429b96b70d47d284ccfcd2c91c0bc58a6987b37534ea9bf6117d6f4cff1fcaf",\n      "org/jsoup/parser/Token$StartTag.class": "423eb434a3c400673c23b87d40f6032d8d56b55d64f6a254d06383b8530c9695",\n      "org/jsoup/parser/Token$Tag.class": "01a5d7455a97609b0ad1c6f9ff32a27683dedc473a703c8e2fe8e9ee952d1422",\n      "org/jsoup/parser/Token$TokenType.class": "f031ea564463210da2656d91d11eb6f50c49826968b5dada110ca65114446ccd",\n      "org/jsoup/parser/Token$XmlDecl.class": "7b3f6998e94f0899c85ad7caefcb5668d80476669a23a9134344f2eab3f5c6e5",\n      "org/jsoup/parser/Token.class": "175633ec0017ea28cd0b91033f27a75471b5ef0158ca4458d038c146d3f87c98",\n      "org/jsoup/parser/TokenQueue.class": "4c4924a73be7c276f784c710633c7c7f43334a695ea3f3320481d6d94adf4188",\n      "org/jsoup/parser/Tokeniser.class": "45d424919ea06d078ead4c23f0119cc5e3bfbc0d11be5c63529c8eda164997d8",\n      "org/jsoup/parser/TokeniserState$1.class": "4c11edbcbb8137cd60d7c901f14c87833accd006a2036852effc45754713ed8d",\n      "org/jsoup/parser/TokeniserState$10.class": "17c50fc3f3dc0c5ab1f554bc9b757bc9e722ba867aaad85b643006f1f25b331f",\n      "org/jsoup/parser/TokeniserState$11.class": "d99be7c5b0d559d1d4f914d7af1416effe34aeca1b6fc61324aaa6f38ab59784",\n      "org/jsoup/parser/TokeniserState$12.class": "44c8a2bcb87b298fdcdbe0f699d716b8dcca6200cdaaeebc71a794549c6ed51e",\n      "org/jsoup/parser/TokeniserState$13.class": "09184874ce15f194c30f5169215bb1e635d6ae647cf9df23d72f55b1e1df1193",\n      "org/jsoup/parser/TokeniserState$14.class": "bad31aad08eb7373e470913be16d121ce296e3d87ac3f27cce2b01377b344806",\n      "org/jsoup/parser/TokeniserState$15.class": "eb5c1fc2bbf15351d730dd3e97b8600e29ff028c3953929e58b0715b7e6f8ca5",\n      "org/jsoup/parser/TokeniserState$16.class": "3e23254f076f2f418293d7d3cf13f3ad4831178b6ed33269e4af7717fd141bdc",\n      "org/jsoup/parser/TokeniserState$17.class": "ea23496874ded86b3d258d34e80ef7a808e3a97cf358ca8716c18ad8c61427d9",\n      "org/jsoup/parser/TokeniserState$18.class": "bcb021181175c79a66912a867e244ef4e65f7731ef575a35c9ef3c0fdb7b19ad",\n      "org/jsoup/parser/TokeniserState$19.class": "ca13a6df4313ab12b59452a25df06c3a8b2baaa10c5aef31a6ceba45adc39992",\n      "org/jsoup/parser/TokeniserState$2.class": "63e8cf53410ca7c2c4aef9660ce06ef9f3dfcd39169c1f73dd5280539817e5e2",\n      "org/jsoup/parser/TokeniserState$20.class": "a19f68c14f0194fad1afa2ff198b0acf37155afa5c05b5e94af0c2d168d9e9d5",\n      "org/jsoup/parser/TokeniserState$21.class": "a9a66a6f4117259cb6086f035d2ad01789a61c37dab6c9267d824498374df9d7",\n      "org/jsoup/parser/TokeniserState$22.class": "740f2b9b3ffde19d7eb85e3111454ac79278460d16251cf4b69b9106f4e3b14e",\n      "org/jsoup/parser/TokeniserState$23.class": "17e0e6f666bac80af73dddc42ba5116de5d302d0369c1f62bccdedcf7c7cf831",\n      "org/jsoup/parser/TokeniserState$24.class": "61badabcd7cf24eef1a837a4629bf09c7c6fb0d231ccecc08d060799ab8a96e4",\n      "org/jsoup/parser/TokeniserState$25.class": "ad913746ed929969defad6ae968fe90620ebfcdd7349ecdb2151fe18f14fa03b",\n      "org/jsoup/parser/TokeniserState$26.class": "d7777501c5d67a8971b524371a3e63d7b9e82cf93e43718119d830659cc77ede",\n      "org/jsoup/parser/TokeniserState$27.class": "faf8f2f35ccd30ef479e1fd664ad9198f34bb7550080823fdb69ca0603e259d5",\n      "org/jsoup/parser/TokeniserState$28.class": "6ae8d0fbb1bdab2fb223d4d5477c47e937225b82e0d46018ffbd262c10929e90",\n      "org/jsoup/parser/TokeniserState$29.class": "5715fb6e39914ada6bedc37b2f52f9dff75a3410876b2f04c718925710b7ade8",\n      "org/jsoup/parser/TokeniserState$3.class": "1974f1808d22445ac400ddd81174c8fbb38b27551508088603f20ea3a74008bc",\n      "org/jsoup/parser/TokeniserState$30.class": "2bacc2db3b9fe1cbd45b21a4e289ab90092a8e00d51c083c1f8087f91577e301",\n      "org/jsoup/parser/TokeniserState$31.class": "a8b06398f05094a1bc7ec85812f006fe847a9ddc2b1dc17b4c34713a9fae34c6",\n      "org/jsoup/parser/TokeniserState$32.class": "e525147c288eaafe4f19b964c54519b46e1ca2b169baa37477c7f8fad89025e1",\n      "org/jsoup/parser/TokeniserState$33.class": "81dd686af692105f8e21f18fe3c73f7b98eb85171b506fcd5cf43f07be2660fd",\n      "org/jsoup/parser/TokeniserState$34.class": "af0d7d117038bb25e0b30bf15dfd49cfb9e5dc53426119b163882d4c4ce7c1b2",\n      "org/jsoup/parser/TokeniserState$35.class": "5d22e2e8029fb47b0063a1e1ae87d2425d4b7e3c59df1c933e3727bf14683b84",\n      "org/jsoup/parser/TokeniserState$36.class": "5f83c30e168cb4746c7c0589bfe25e2b33633ff4ea73607a9681c3f84a50ef28",\n      "org/jsoup/parser/TokeniserState$37.class": "acc50b21456a98a61e74b0d504ed9a029827e7a0f861854dc7197817d52539f9",\n      "org/jsoup/parser/TokeniserState$38.class": "747ef3ec1f08a2f2259f78e6b52d2469025832261911f4f84d0b2e1bbaa6a23b",\n      "org/jsoup/parser/TokeniserState$39.class": "acea39b9b9e932b282b5f2dad19541aa2f29d44798aee6bb3a9c67c053887f83",\n      "org/jsoup/parser/TokeniserState$4.class": "00a5922311138cb506c0f808d5afe50feed464454c85514031d5207b0ba81f33",\n      "org/jsoup/parser/TokeniserState$40.class": "a2ffd0da3e2adc52bf2de7a7e025095699962dcd1a931ba0b350cec7e26fad8d",\n      "org/jsoup/parser/TokeniserState$41.class": "0e95b65be52931e241e06d55354f0515a7bd0488770948e5268430ed98be5355",\n      "org/jsoup/parser/TokeniserState$42.class": "eff711b32108dac5fa2498525fd2a633357bc10e8e4c5eba52df453bd3f0dd44",\n      "org/jsoup/parser/TokeniserState$43.class": "4da35ccbc6bdd136a2eb4f26614b88c5fb4d70beb976a340029b351046cf51ac",\n      "org/jsoup/parser/TokeniserState$44.class": "8ed3b557cab9ef5526de949736fe7b3afbfce8ed4ca039f637b09f5c60c7272e",\n      "org/jsoup/parser/TokeniserState$45.class": "5fafe74c4e219bb778abffd29a7afbb8bac4bb02725dcb5fd5fc8c18a5aef64a",\n      "org/jsoup/parser/TokeniserState$46.class": "ed7d5f348114cb27f1b82c23a96a265706c864e3b56e3059d5bb8bb313482fe4",\n      "org/jsoup/parser/TokeniserState$47.class": "22ba0429ea2b725a9bf370543a5ec7c02fca93d59ac1e3c0942073948e70a513",\n      "org/jsoup/parser/TokeniserState$48.class": "490712bce7a159c83b663ef8939047c6da731a766f2e87895d69e80e216b3958",\n      "org/jsoup/parser/TokeniserState$49.class": "09919dbacb04ab9c4284c57a06314a24dab93ba650595af6c943d5696f7b7e6e",\n      "org/jsoup/parser/TokeniserState$5.class": "4e8920dd33c325494801ba85be8b42cec0d4fefbda21e799a8f10c22eda0bebf",\n      "org/jsoup/parser/TokeniserState$50.class": "c16d14831aeadc9aee7ae6cefbf1a4e1a32f5019e26a08e2c1858c2cf9b1b019",\n      "org/jsoup/parser/TokeniserState$51.class": "b4f6879e19ed4cf5f598375244d8d9821911acdb223add60689a669f85343d5b",\n      "org/jsoup/parser/TokeniserState$52.class": "4c7019ac3cb6752b07ac6cc87482366d80e7bc4c3f69408a1701461ffcab7432",\n      "org/jsoup/parser/TokeniserState$53.class": "663b4388d61d5aefb613a9472194a92050febba6f11a6e790f3bdcfa3a0b1ebd",\n      "org/jsoup/parser/TokeniserState$54.class": "3ea3e4a60e81cb99fbc88797b6607abfcec55e1d762ff9d473c02585a26e020b",\n      "org/jsoup/parser/TokeniserState$55.class": "71a68c7309ba796389fcc4772a90e2fdb28410c35a345631cbf97fc713c865c5",\n      "org/jsoup/parser/TokeniserState$56.class": "a701004f42cdb1d2d31aa6c2e0877882987470f6a4a7181f1c5ea1890688acf6",\n      "org/jsoup/parser/TokeniserState$57.class": "313e1043e62ef3d69adfc807e204a99813f00d55423f4bbfd7447d61f91abe5f",\n      "org/jsoup/parser/TokeniserState$58.class": "938f5bae78ea8c7f8735e843559902023bf058da985a17c24486af9e1fd9b794",\n      "org/jsoup/parser/TokeniserState$59.class": "5253fd1a948b2debffd96f8cb7ce02ee125e89b37c042d601b4ee4bc8c9b22b4",\n      "org/jsoup/parser/TokeniserState$6.class": "cbf555f52f4ee696f18c3890d7d6d22c8b7bd3d168cf768035f00bb52edfbcf4",\n      "org/jsoup/parser/TokeniserState$60.class": "029beb9ad9bb7fea7651fc099dd6500588474b70f84f19c7a50d725a8fb96932",\n      "org/jsoup/parser/TokeniserState$61.class": "cbf6decfcf63599d7dce33de4ea8924a52f594eaf9c650f181cd082ef3f46c5c",\n      "org/jsoup/parser/TokeniserState$62.class": "53e4b18ae5ffd18a30d5254d6a76c49eb29e0f613bbe3cd3f3075ee9d65d61d5",\n      "org/jsoup/parser/TokeniserState$63.class": "9bd07158f7bf61ac819d348d36f37038a609e1c73bce68c4f4429195b4dfe0b6",\n      "org/jsoup/parser/TokeniserState$64.class": "84301625db4da93284974af96453d27eb28c0404cff2037ba27b968535ae357d",\n      "org/jsoup/parser/TokeniserState$65.class": "12c805ef0fd3f952195143e9b050d45fff2eddf73a555ec25e5009b231ba2290",\n      "org/jsoup/parser/TokeniserState$66.class": "0992e99496d8398f12395f8d1128805e5160a8fd745afa8f8ef47dcb029a35e5",\n      "org/jsoup/parser/TokeniserState$67.class": "164b75ec1e4462e95106fc7bd36eab24f4e74882cac63f0cb6a39a8156ac4bb3",\n      "org/jsoup/parser/TokeniserState$68.class": "94fcf62633a65ecb61bbeab3721a4b4d63656d7a03ec662f59ded3ca51d5f766",\n      "org/jsoup/parser/TokeniserState$7.class": "9f205d50c657915b777271e2040e5312cb4398b5d3fab484eaa3aac19083e15c",\n      "org/jsoup/parser/TokeniserState$8.class": "0063b8acd3ef7bb672eace99a96c996562fc4716bedef54202440342decc15a8",\n      "org/jsoup/parser/TokeniserState$9.class": "d844b4688cfc22ee7c7f070d1bf1ec29cad26a404033cebf4fdc47ca4b825b83",\n      "org/jsoup/parser/TokeniserState.class": "304510bede635b82d9362306b6e90e56d66f6bb0b0c66c26fdb640e1b7c96403",\n      "org/jsoup/parser/TreeBuilder.class": "325180718a259fc4525b5c58ca47a644b1117a927d6c2d8b394e9fe08027597f",\n      "org/jsoup/parser/XmlTreeBuilder$1.class": "2bca133a99f4d51c59c48f5cb8c2d2ba080f4e798f8d9db62802ee6039bf5825",\n      "org/jsoup/parser/XmlTreeBuilder.class": "20ef1840be3138253a972e6aa0f8c7c5fbb646c0bd57bcf54b48c4703d9cb4b4",\n      "org/jsoup/parser/package-info.class": "2f2a1aebb16a42706de12956a6046ffd6a2e4e065a7a949f8aba9cc982261d86",\n      "org/jsoup/safety/Cleaner$1.class": "7c27cf8d818fa956bf3a7fae85e041b43ac3d24dc9daf1808335a8ce8be052fa",\n      "org/jsoup/safety/Cleaner$CleaningVisitor.class": "6b3c7740e68eec4cec21e408d33ae699f524c781ba6daa27afee6c5b993a4314",\n      "org/jsoup/safety/Cleaner$ElementMeta.class": "b61714c21232f13a3b34d8a054f9ba7baa30430653d9311bc92389d384ba2a8d",\n      "org/jsoup/safety/Cleaner.class": "ad89c3f8a580ff2a0d32d3844e9c4ff6783e4a54de0d5a19676d7f12f7b3edbb",\n      "org/jsoup/safety/Safelist$AttributeKey.class": "081541f182f73ebe4bf87a2c81bc0d2f48d91137064a1bbca57c73c9887149dd",\n      "org/jsoup/safety/Safelist$AttributeValue.class": "06171b5af9222a7aefb9399e966dc26215ed8ceb62332bcc4b9eec721135fc24",\n      "org/jsoup/safety/Safelist$Protocol.class": "e0dca079c815c48a3cf4e5b86e8eed37cf7789823c56a4bf1b60886f7982fc79",\n      "org/jsoup/safety/Safelist$TagName.class": "ba0e4102efa628f2d6bb83ba9717f1bffd9d649e980ab770931d8b9bedf5cdea",\n      "org/jsoup/safety/Safelist$TypedValue.class": "b596c6eebd7a49aef7e93df2b8a1f3dc020a5a3bad6d222065af628cf52e368f",\n      "org/jsoup/safety/Safelist.class": "e22c09ed92d3c80c77d242ac105dca2a76a6cb27c9a6ddc5f9289052fe8e5806",\n      "org/jsoup/safety/package-info.class": "5a5cdd21cc8eeafc016b7a4ee66227b36b85ba32262356214be3a183b0873ac6",\n      "org/jsoup/select/Collector.class": "6c4f17e6a288f46c40009fffa44d41b05557f4e06915e78d698a9798d4af610b",\n      "org/jsoup/select/CombiningEvaluator$And.class": "7bb8fc39bdbaaf1892b4200c4bcbffe4501c6922e9fd7d7cc6ea59cf638302ad",\n      "org/jsoup/select/CombiningEvaluator$Or.class": "6d377e581497023687df5ef5d6d72e555370b5308f9fd6b002fda4132ce42b9f",\n      "org/jsoup/select/CombiningEvaluator.class": "a93c66d8af9b18395386a015671739a327a74e78c4bdddbbcdec9a0a40cb8243",\n      "org/jsoup/select/Elements.class": "c60ae7e2ffb719a3c4c8700efe915fe10c73586014ddbfc468435176d71235f1",\n      "org/jsoup/select/Evaluator$AllElements.class": "83901abe162f9ab2e2cce08b371c89a82e48890747e87b337a27287ebccf1a51",\n      "org/jsoup/select/Evaluator$Attribute.class": "c0d3f41d964b35a98e3695a7c0fbdceb7b3e83d4c83eb2557e0bfe12d74fd4d8",\n      "org/jsoup/select/Evaluator$AttributeKeyPair.class": "0ad54afa07a269e4dae6674a34e858021f45fc92a37a7218d0e49ab847f635d6",\n      "org/jsoup/select/Evaluator$AttributeStarting.class": "bf82e47d72b72719ff85cc13971169ebb2c1c266f0a8ff9ba554245504f3285c",\n      "org/jsoup/select/Evaluator$AttributeWithValue.class": "9a2a379f3baa3f512cff846814982920acc88b26b6a6e96912bc7154c9daf0ea",\n      "org/jsoup/select/Evaluator$AttributeWithValueContaining.class": "0571f501810b24b376cd6993ea2fc67088005274b39d8b28e8ae3ceea94778c3",\n      "org/jsoup/select/Evaluator$AttributeWithValueEnding.class": "4698aceac0eae8e401d85849602e76c8b642a95f3d218ddbbfc11986f7753b63",\n      "org/jsoup/select/Evaluator$AttributeWithValueMatching.class": "60fab4ff1f42b8a295c17468a381a544e97088cd8a61da71795f574b4dd7205f",\n      "org/jsoup/select/Evaluator$AttributeWithValueNot.class": "40e963651ddea8e73e65f1a97c49dd8274154fa830b80b345f66601f499e4a20",\n      "org/jsoup/select/Evaluator$AttributeWithValueStarting.class": "f3b09e2639a5651ef230d209671534b8f0d169f6dbaaf7b30a1bee32210dfee7",\n      "org/jsoup/select/Evaluator$Class.class": "026ac5df8537e93dcf6f114c9f643abc644c52ab7418462ea4a8c134fed99fa3",\n      "org/jsoup/select/Evaluator$ContainsData.class": "f7738d9e26f0ca240427d4e193b15e89090f5cf98f0d24f675377f7028d87ea0",\n      "org/jsoup/select/Evaluator$ContainsOwnText.class": "5d5992d8297047b9a871ccb3217d5fa6599040baa24cf233e00326940a92e136",\n      "org/jsoup/select/Evaluator$ContainsText.class": "9c914d6f3f7e6a926e756a3c6c88b7739fe7d0fec854cc0d9a835b2d2579d26e",\n      "org/jsoup/select/Evaluator$ContainsWholeOwnText.class": "2d134aa64f9213406db36d062c03f91ccf40f50fae9126e8ebbbef520e595ced",\n      "org/jsoup/select/Evaluator$ContainsWholeText.class": "5581ae8d80a55cd44eb209831756e8213dd260c0c2f9188311e32c564486ab71",\n      "org/jsoup/select/Evaluator$CssNthEvaluator.class": "e249cf7a128b0488a93906d1c676244abccd68a6129ab2a6c619892b0b2725fe",\n      "org/jsoup/select/Evaluator$Id.class": "3c6ed23ad1284e5aa37e32e7c1502700139e41f8b2b88af729614b663d98b0ab",\n      "org/jsoup/select/Evaluator$IndexEquals.class": "8c0a545059094d8b816b3402f0ab5decb90567793372385a257439ccd5d8c40e",\n      "org/jsoup/select/Evaluator$IndexEvaluator.class": "f5afed3ec3d42c9cd97a1aab900cf18db24456fd14f1eac130c5af57f2e5bd82",\n      "org/jsoup/select/Evaluator$IndexGreaterThan.class": "e7715be4d4c6830dc0360b87163341606ca392fe83996bce9700ee7c64809e21",\n      "org/jsoup/select/Evaluator$IndexLessThan.class": "fdb567d97d3442e2fbc64e1ec30425a1751992e1b99eebfcf345a3f5feb85869",\n      "org/jsoup/select/Evaluator$IsEmpty.class": "d398b955c655fcbec9d7b2e5a049ed01d1952da1526833b00cc42dd3c4a6d237",\n      "org/jsoup/select/Evaluator$IsFirstChild.class": "e2f85d066b18c72e24417d81eb16209a9000253ac19400d87de40f61d42fb5e8",\n      "org/jsoup/select/Evaluator$IsFirstOfType.class": "860d72fd0cb75e2ce1c4024310c773c2a905c0a4c918c65b5508765aecf14b85",\n      "org/jsoup/select/Evaluator$IsLastChild.class": "273203e5fd518dbdb40abab86944f9ee6fb99d78b225f22397cead4c4c4cbf83",\n      "org/jsoup/select/Evaluator$IsLastOfType.class": "57ffc4eaabb9c45e5802350aa2d644430554fc4a972673f1801cbd626c72d186",\n      "org/jsoup/select/Evaluator$IsNthChild.class": "aec8f606e5e8d6f05d7faf97ea48a548dc21493f86e8e680c8dd8e974e299655",\n      "org/jsoup/select/Evaluator$IsNthLastChild.class": "411497b15cf1ad554cda7b699ccfd2c6013f2e76a2b1ef66a4afd55f99887088",\n      "org/jsoup/select/Evaluator$IsNthLastOfType.class": "3e450b4d9b0962df4e702a2e61975d20907574934a2e1b765036b6ebdc5011f4",\n      "org/jsoup/select/Evaluator$IsNthOfType.class": "1645f0a0efd50c1028adac71d281ed6a55ff88f1f70ca8a55a57c60b5f2da3d5",\n      "org/jsoup/select/Evaluator$IsOnlyChild.class": "021a7f1da2dc061423b581a9626f1df0b143cb15efbe2be1a8d2b0157c9f0791",\n      "org/jsoup/select/Evaluator$IsOnlyOfType.class": "0dd0ee3c1750aa919354a413ef6965f3a9a7767bc714369bb762d517b411ff75",\n      "org/jsoup/select/Evaluator$IsRoot.class": "85ef7462afe5bce4fd4ae77cfbe5eec71d261bc38a5ae26ea42fa5f225b6ec22",\n      "org/jsoup/select/Evaluator$MatchText.class": "ed9056e7bf28f68db449bc6bbf04d68887b0a065f1ede5ccc084975559217fa0",\n      "org/jsoup/select/Evaluator$Matches.class": "c83f09f7dd52853efb5a852df885de0a25ccee86741498083f96916d979fc3ff",\n      "org/jsoup/select/Evaluator$MatchesOwn.class": "9434d3c357173b8c7d466ffca80ab9394eb83443d68dedf6713dc26c1aef7c4e",\n      "org/jsoup/select/Evaluator$MatchesWholeOwnText.class": "93bcb91bcd3c0fb8dee0fb1f20b75d05c836b479b572a01c8972769c1ee3a21a",\n      "org/jsoup/select/Evaluator$MatchesWholeText.class": "0d8b64b857a3e7eb428ebb1161f15357eda65a0231ecbf91e3234c53350af301",\n      "org/jsoup/select/Evaluator$Tag.class": "c5bd6c0a504121eb69cf703c7a28c837f2747997deff31f7024b1c5a3a8d1838",\n      "org/jsoup/select/Evaluator$TagEndsWith.class": "1a3771f6089d88e75523c1eebb78202561aa1528dbf87f2db605a768d64b999c",\n      "org/jsoup/select/Evaluator$TagStartsWith.class": "a651ab87de836faeb11f30c5c1c2aab32e02b6efd11311a09f376b076149d2b6",\n      "org/jsoup/select/Evaluator.class": "ef349107b93df7ea4cbc0d088d656a005811205addfdeaefb34d0df9315f8108",\n      "org/jsoup/select/NodeFilter$FilterResult.class": "6e3bc3a1608e24748e09890edee3bc443311634cd16734fdb9ab15ddf6570b45",\n      "org/jsoup/select/NodeFilter.class": "657a89041f912f3ae1f0de7794b0dc1f4cf4480a4c84db72042cf3511fed849f",\n      "org/jsoup/select/NodeTraversor.class": "0b7f20e65dbec362ad238559da12ac1f0d1da5f3eb434a2cb87b5647d4e4830e",\n      "org/jsoup/select/NodeVisitor.class": "5d6b56c222695b681569e87cd1a16bcee409ff75f36d9b62983347a618e53250",\n      "org/jsoup/select/QueryParser.class": "4c044b9b66288288723f3c0e3a7b35c5e38b8d37abffa8a942b5dbe83757100b",\n      "org/jsoup/select/Selector$SelectorParseException.class": "f3f5e6677b845fbd210bbf8c66b4b55fd209abb1815affcd5ea380b2178a3392",\n      "org/jsoup/select/Selector.class": "1172c4e40aae9c64fec77db123f6cda28b9b36258e5407971f1f2f308c90ab09",\n      "org/jsoup/select/StructuralEvaluator$Ancestor.class": "c659685ba074778bce4d1825693de84a5ae6ede345731c56cd23d1528f55d4d6",\n      "org/jsoup/select/StructuralEvaluator$Has.class": "3250673186f532e2bfdf6b91c908438f718d4f6d93f0443330ad1a4bd7343b2c",\n      "org/jsoup/select/StructuralEvaluator$ImmediateParentRun.class": "47b296c7ff24a622c4209dac275fae7aa1746e8414540c3068d0600110077dd9",\n      "org/jsoup/select/StructuralEvaluator$ImmediatePreviousSibling.class": "92b4bf7ad1792db2456d670de195c3161ef95be442ef5e79ab37089e656d7d2d",\n      "org/jsoup/select/StructuralEvaluator$Is.class": "79b2e73c1a1465baca15150dccd0b371ddc78777efc2c130fc5c173c6196c2bc",\n      "org/jsoup/select/StructuralEvaluator$Not.class": "8cf27eacef4a6a909130b2f78c7053c9dc1b33f6bb22a6f9a98435e874c229ca",\n      "org/jsoup/select/StructuralEvaluator$PreviousSibling.class": "5187721ed303854ff6cd9da054c0941286e2c14c5c112e25b762ca250c54e434",\n      "org/jsoup/select/StructuralEvaluator$Root.class": "d188b8b88070c952d5d4b208ff88e55688c3721c71d692c741091c362f654fd9",\n      "org/jsoup/select/StructuralEvaluator.class": "67dbf10469fe3d225abbdfefeef504740a25821de42ac158bd743a089b31528e",\n      "org/jsoup/select/package-info.class": "44832debbe2daa2f30d92c91c8749c8ead966f24040e763de8ef6f5e4655d627"\n    }\n  }\n}\n')
gemma_provenance = json.loads('{\n  "verified": true,\n  "initial_slots": 10,\n  "origins": {\n    "jsoup-1729": {\n      "phase": "parent_collection",\n      "available": true\n    },\n    "codec_standard-1729": {\n      "phase": "parent_collection",\n      "available": true\n    },\n    "codec_url-1729": {\n      "phase": "parent_collection",\n      "available": true\n    },\n    "lang-1729": {\n      "phase": "parent_collection",\n      "available": true\n    },\n    "jfreechart-1729": {\n      "phase": "parent_collection",\n      "available": true\n    },\n    "jfreechart-2718": {\n      "phase": "parent_collection",\n      "available": true\n    },\n    "lang-2718": {\n      "phase": "parent_collection",\n      "available": false,\n      "status": "timeout_no_response"\n    },\n    "codec_url-2718": {\n      "phase": "resumed_collection",\n      "available": true\n    },\n    "codec_standard-2718": {\n      "phase": "resumed_collection",\n      "available": true\n    },\n    "jsoup-2718": {\n      "phase": "resumed_collection",\n      "available": true\n    }\n  },\n  "retained_actual_responses": 6,\n  "parent_error": "TimeoutError timed out",\n  "resumed_initial_requests": 3,\n  "assertion_enabled_feedback_exactly_matches_delivered": true,\n  "verified_distinct_callback_requests": 8,\n  "final_raw_prompts_routing_and_normalization_verified": true,\n  "matched_control_eligible_slots": 0,\n  "no_raw_response_regeneration_or_researcher_code_correction": true\n}\n')

if GEMMA_REPLAY_KIT is not None:
    packet=Path(GEMMA_REPLAY_KIT).read_bytes()
    assert sha256(packet).hexdigest()==EXPECTED_GEMMA_KIT_SHA256
    with zipfile.ZipFile(io.BytesIO(packet)) as archive:
        names=archive.namelist()
        assert len(names)==len(set(names))
        assert not any(n.startswith('/') or '..' in Path(n).parts for n in names)
        assert sum(n.file_size for n in archive.infolist())<=100_000_000
        manifest={}
        for line in archive.read('SHA256SUMS').decode().splitlines():
            digest,name=line.split('  ',1)
            assert name not in manifest
            manifest[name]=digest
            assert sha256(archive.read(name)).hexdigest()==digest
        assert set(manifest)==set(names)-{'SHA256SUMS'}
        assert json.loads(archive.read('evidence/SUMMARY.json'))==gemma_summary
        assert json.loads(archive.read('evidence/COVERAGE_REPORT.json'))==gemma_coverage
        assert json.loads(archive.read('evidence/PROVENANCE_AUDIT.json'))==gemma_provenance
        initial=json.loads(archive.read('evidence/INITIAL.json'))
        final=json.loads(archive.read('evidence/FINAL.json'))
        feedback=json.loads(archive.read('evidence/FEEDBACK.json'))
        assert final['complete'] and final['feedback_packet']==feedback
        assert final['initial']==initial['initial']
        assert set(final['repairs'])==set(feedback['tasks'])
        assert sha256(archive.read('evidence/INITIAL.json')).hexdigest()==feedback['initial_sha256']
        provenance=json.loads(archive.read('evidence/EVIDENCE_PROVENANCE.json'))
        for label in ('initial_a','initial_b','final_a','final_b','qualify_a','qualify_b','coverage'):
            raw=archive.read('evidence/'+label+'.tar.gz')
            assert sha256(raw).hexdigest()==provenance[label]['sanitized_scientific_archive_sha256']
            with tarfile.open(fileobj=io.BytesIO(raw)) as native:
                assert all(n.isfile() and n.name.startswith('study/') and '..' not in Path(n.name).parts for n in native.getmembers())
                report=json.load(native.extractfile('study/native/REPORT.json'))
                assert report['completed']
                if label=='coverage':
                    assert report==gemma_coverage
                else:
                    assert all(not v['errors'] for v in report['units'].values())
                    if label.startswith('qualify'):
                        assert all(v['qualified'] for v in report['units'].values())
                    for unit in report['units']:
                        control=json.load(native.extractfile('study/native/'+unit+'/ASSERTION_CONTROL.json'))
                        assert control['verdict']=='FAIL'
                        refusal=native.extractfile('study/native/'+unit+'/assertion_disabled_refusal.stderr').read()
                        assert b'Java assertions must be enabled with -ea' in refusal
    print('Hash-bound reviewer-kit evidence verified. This is saved-record verification, not fresh native execution.')
else:
    print('Embedded recorded projection. Use the kit and the separate native commands to reproduce executions.')

assert gemma_summary['completed'] and gemma_summary['java_language_assertions_enabled']
assert (gemma_summary['paired_seed_observations'],gemma_summary['repair_units'],gemma_summary['historical_commit_clusters'])==(10,5,4)
assert gemma_summary['model']['sha256']=='676c35070db6dbe52f93e9c864ee0fba4eddea94b9c875d9cb10daff453fbaee'
assert gemma_summary['activated_initial_grader_feedback_contrasts']==0
assert gemma_provenance['verified']
assert gemma_provenance['assertion_enabled_feedback_exactly_matches_delivered']
assert gemma_provenance['final_raw_prompts_routing_and_normalization_verified']
assert gemma_provenance['retained_actual_responses']==6
assert gemma_provenance['resumed_initial_requests']==3
assert gemma_provenance['matched_control_eligible_slots']==0
assert len(gemma_summary['rows'])==10
assert len({r['sample'] for r in gemma_summary['rows']})==10
assert sum(r['initial_output_status']=='available' for r in gemma_summary['rows'])==gemma_summary['counts']['initial_model_outputs_available']
for arm in ('released','repaired'):
    assert sum(bool(r['arms'][arm]['oracle_qualified']) for r in gemma_summary['rows'])==gemma_summary['counts'][arm+'_qualified']
assert all(r['arms']['released']['oracle_qualified']==r['arms']['repaired']['oracle_qualified'] for r in gemma_summary['rows'])

print('\nFive source-exposed method units, four commits, two seeds per unit. Codec units share one commit.')
print('Initial available responses',gemma_summary['counts']['initial_model_outputs_available'],'of 10')
print('Distinct repair requests',gemma_summary['distinct_repair_calls'],'and complete repair outputs',gemma_summary['actual_repair_outputs'])
print('No initial grader-feedback contrast activated. The matched subset is empty.')
print('\nSample | Initial admission | Initial adequate | Released final | Repaired final')
for row in gemma_summary['rows']:
    print(' | '.join([row['sample'],row['initial_grade_status'],str(row['initial_oracle_qualified']),
        str(row['arms']['released']['oracle_qualified']),str(row['arms']['repaired']['oracle_qualified'])]))
print('Adequate means complete developer PASS and complete declared-fault FAIL. It is not exhaustive correctness.')

assert gemma_coverage['completed'] and gemma_coverage['exploratory_posthoc_single_case']
assert gemma_coverage['exact_scope']=='org/jsoup/nodes/Entities'
assert gemma_coverage['counterexample_admitted'] and gemma_coverage['oracle_contrast']
assert gemma_coverage['all_four_exact_vectors_equal'] and gemma_coverage['all_four_class_and_method_counters_equal']
runs=gemma_coverage['runs']
assert {(r['role'],r['repetition']) for r in runs}=={('gold',0),('gold',1),('gemma',0),('gemma',1)}
assert all(r['class_origins_verified'] and r['native']['verdict']=='PASS' for r in runs)
assert len({(r['class_id'],r['probe_bits']) for r in runs})==1
assert all(r['probe_bits'].count('1')==73 and len(r['probe_bits'])==162 for r in runs)
assert all(r['class_counters']['BRANCH']=={'missed':75,'covered':40} for r in runs)
assert all(r['class_counters']==runs[0]['class_counters'] and r['method_counters']==runs[0]['method_counters'] for r in runs)
assert gemma_coverage['grades']['gold']['oracle']['developer']['contract']['verdict']=='PASS'
assert gemma_coverage['grades']['gold']['oracle']['fault']['contract']['verdict']=='FAIL'
assert gemma_coverage['grades']['gemma']['oracle']['developer']['contract']['verdict']=='PASS'
assert gemma_coverage['grades']['gemma']['oracle']['fault']['contract']['verdict']=='PASS'
print('\nPost-hoc single-class coverage diagnostic, same 73/162 probes and 40/115 branches in all four fresh JVM runs.')
print('Developer reference | intended production PASS | historical fault FAIL')
print('Saved Gemma body    | intended production PASS | historical fault PASS')
print('Equal changed-class coverage does not establish equal traces or assertion adequacy. No leaderboard regrade is claimed.')


Embedded recorded projection. Use the kit and the separate native commands to reproduce executions.

Five source-exposed method units, four commits, two seeds per unit. Codec units share one commit.
Initial available responses 9 of 10
Distinct repair requests 8 and complete repair outputs 6
No initial grader-feedback contrast activated. The matched subset is empty.

Sample | Initial admission | Initial adequate | Released final | Repaired final
jsoup-1729 | REJECTED | False | False | False
codec_standard-1729 | COMPILE_ERROR | False | False | False
codec_url-1729 | REJECTED | False | False | False
lang-1729 | REJECTED | False | False | False
jfreechart-1729 | REJECTED | False | True | True
jfreechart-2718 | GRADED | True | True | True
lang-2718 | REJECTED | False | False | False
codec_url-2718 | REJECTED | False | False | False
codec_standard-2718 | REJECTED | False | False | False
jsoup-2718 | GRADED | False | False | False
Adequate means complete developer PASS and complete declared-